In [42]:
import sys
import torch
import numpy as np
import nmslib
import os
from shapely import wkt

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")
print(f"NumPy: {np.__version__}")
print(f"NMSLIB: {nmslib.__version__}")

paths = {
    "parks.tsv":      "/raid/ruban/data/parks.tsv",
    "GT full":        "/raid/ruban/groundtruth/pk-query-187019",
    "Encodings full": "/raid/ruban/encodings/pk0_0.002"
}
for name, path in paths.items():
    print(f"{name}: {'OK' if os.path.exists(path) else 'NOT FOUND'} — {path}")

Python: 3.10.19 | packaged by conda-forge | (main, Jan 26 2026, 23:45:08) [GCC 14.3.0]
PyTorch: 2.5.1+cu121
CUDA available: True
GPU count: 8
GPU 0: NVIDIA A100-SXM4-80GB
NumPy: 1.26.4
NMSLIB: 2.1.2
parks.tsv: OK — /raid/ruban/data/parks.tsv
GT full: OK — /raid/ruban/groundtruth/pk-query-187019
Encodings full: OK — /raid/ruban/encodings/pk0_0.002


In [43]:
import os
import numpy as np
from tqdm import tqdm

GT_PATH = "/raid/ruban/groundtruth/pk-query-187019"

def sort_gt_files(files):
    ids = [int(f[f.find('_')+1: f.find('-')]) for f in files]
    _, filenames = zip(*sorted(zip(ids, files)))
    return filenames

def read_gt_file(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    gt = []
    for line in lines:
        row = line.strip().split(', ')
        if len(row) <= 1:
            gt.append([])
        else:
            gt.append([int(x) for x in row])
    return gt

def load_all_gt(path):
    files = sort_gt_files(os.listdir(path))
    gt_all = []
    for f in tqdm(files, desc="Loading GT"):
        gt_all += read_gt_file(os.path.join(path, f))
    return gt_all

def compute_recall_at_k(gt_lookup, nbrs, query_start_id, K):
    total_recall = 0.0
    count   = 0
    zero_gt = 0
    for i, (ids, dists) in enumerate(nbrs):
        qid          = query_start_id + i
        gt_neighbors = set(gt_lookup.get(qid, [])[:K])
        if len(gt_neighbors) == 0:
            zero_gt += 1
            continue
        retrieved     = set(ids[:K])
        recall        = len(gt_neighbors & retrieved) / len(gt_neighbors)
        total_recall += recall
        count        += 1
    avg_recall  = total_recall / count if count > 0 else 0.0
    full_recall = total_recall / len(nbrs)
    return avg_recall, full_recall, zero_gt

print("Loading GT...")
gt_all    = load_all_gt(GT_PATH)
gt_lookup = {}
for row in gt_all:
    if len(row) > 1:
        gt_lookup[row[0]] = row[1:]

QUERY_START_ID = min(gt_lookup.keys())
print(f"GT queries with neighbors: {len(gt_lookup)}")
print(f"Query start ID (dynamic):  {QUERY_START_ID}")
print(f"Sample GT[{QUERY_START_ID}]: {gt_lookup.get(QUERY_START_ID, [])[:5]}")

Loading GT...


Loading GT: 100%|██████████| 120/120 [00:28<00:00,  4.14it/s]


GT queries with neighbors: 44666
Query start ID (dynamic):  187019
Sample GT[187019]: [184495, 105197, 51737, 107491, 130402]


In [44]:
import numpy as np
from shapely import wkt
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

PARKS_TSV    = "/raid/ruban/data/parks.tsv"
CACHE_COORDS = "/tmp/coord_list_full.npy"
CACHE_OSMIDS = "/tmp/osm_ids_full.npy"

if not os.path.exists(CACHE_COORDS):
    print("Counting total polygons...")
    with open(PARKS_TSV, 'r') as f:
        N_TOTAL = sum(1 for _ in tqdm(f, desc="Counting"))
    print(f"Total polygons: {N_TOTAL}")

    def parse_line(line):
        parts  = line.strip().split('\t')
        osm_id = int(parts[0])
        try:
            geom   = wkt.loads(parts[1].strip())
            coords = np.array(geom.coords, dtype=np.float32)
            return osm_id, coords
        except:
            return osm_id, None

    print("Reading all lines...")
    raw_lines = []
    with open(PARKS_TSV, 'r') as f:
        for line in tqdm(f, total=N_TOTAL, desc="Reading"):
            raw_lines.append(line)

    osm_ids    = [None] * N_TOTAL
    coord_list = [None] * N_TOTAL

    with ThreadPoolExecutor(max_workers=64) as executor:
        futures = {executor.submit(parse_line, raw_lines[i]): i for i in range(N_TOTAL)}
        for future in tqdm(as_completed(futures), total=N_TOTAL, desc="Parsing"):
            i = futures[future]
            osm_ids[i], coord_list[i] = future.result()

    print("Caching to disk...")
    np.save(CACHE_OSMIDS, np.array(osm_ids))
    coord_arr = np.empty(N_TOTAL, dtype=object)
    for i, c in enumerate(coord_list):
        coord_arr[i] = c
    np.save(CACHE_COORDS, coord_arr, allow_pickle=True)
    print("Cached.")

else:
    print("Loading from cache...")
    coord_arr  = np.load(CACHE_COORDS, allow_pickle=True)
    coord_list = list(coord_arr)
    osm_ids    = list(np.load(CACHE_OSMIDS, allow_pickle=True))
    N_TOTAL    = len(coord_list)
    print(f"Loaded {N_TOTAL} polygons from cache.")

failed  = sum(1 for c in coord_list if c is None)
lengths = [len(c) for c in coord_list if c is not None]
print(f"Total: {N_TOTAL} | Failed: {failed}")
print(f"Vertex count — min:{min(lengths)} max:{max(lengths)} mean:{np.mean(lengths):.1f}")

Counting total polygons...


Counting: 234447it [00:00, 1246486.51it/s]


Total polygons: 234447
Reading all lines...


Parsing: 100%|██████████| 234447/234447 [00:01<00:00, 227977.08it/s]


Caching to disk...
Cached.
Total: 234447 | Failed: 0
Vertex count — min:0 max:2000 mean:16.6


In [45]:
import numpy as np
import torch
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

N_POINTS          = 64
POLY_TENSOR_CACHE = "/tmp/poly_tensor_full.pt"

def resample_polygon(coords, n_points):
    if len(coords) == 0: return np.zeros((n_points, 2), dtype=np.float32)
    if len(coords) == 1: return np.tile(coords[0], (n_points, 1)).astype(np.float32)
    if not np.allclose(coords[0], coords[-1]):
        coords = np.vstack([coords, coords[0]])
    deltas    = np.diff(coords, axis=0)
    seg_lens  = np.sqrt((deltas**2).sum(axis=1))
    cum_len   = np.concatenate([[0], np.cumsum(seg_lens)])
    total_len = cum_len[-1]
    if total_len < 1e-10: return np.tile(coords[0], (n_points, 1)).astype(np.float32)
    sample_lens = np.linspace(0, total_len, n_points, endpoint=False)
    resampled   = np.zeros((n_points, 2), dtype=np.float32)
    for i, s in enumerate(sample_lens):
        idx = np.clip(np.searchsorted(cum_len, s, side='right') - 1, 0, len(coords)-2)
        seg = cum_len[idx+1] - cum_len[idx]
        t   = (s - cum_len[idx]) / seg if seg > 1e-10 else 0.0
        resampled[i] = (1-t)*coords[idx] + t*coords[idx+1]
    mins = resampled.min(0); maxs = resampled.max(0)
    ranges = np.where(maxs - mins < 1e-10, 1.0, maxs - mins)
    return ((resampled - mins) / ranges).astype(np.float32)

def preprocess_polygon(coords, n_points=N_POINTS):
    if coords is None or len(coords) == 0:
        return np.zeros((n_points, 2), dtype=np.float32)
    return resample_polygon(coords, n_points)

if not os.path.exists(POLY_TENSOR_CACHE):
    print(f"Preprocessing {len(coord_list)} polygons → {N_POINTS} pts each...")
    processed = [None] * len(coord_list)
    with ThreadPoolExecutor(max_workers=64) as executor:
        futures = {executor.submit(preprocess_polygon, coord_list[i]): i
                   for i in range(len(coord_list))}
        for future in tqdm(as_completed(futures), total=len(coord_list), desc="Preprocessing"):
            i = futures[future]
            processed[i] = future.result()
    poly_tensor = torch.tensor(np.stack(processed), dtype=torch.float32)
    torch.save(poly_tensor, POLY_TENSOR_CACHE)
    print(f"Cached to {POLY_TENSOR_CACHE}")
else:
    print("Loading poly_tensor from cache...")
    poly_tensor = torch.load(POLY_TENSOR_CACHE, weights_only=True)
    print("Loaded from cache.")

print(f"Shape: {poly_tensor.shape}")
print(f"Any NaN: {torch.isnan(poly_tensor).any().item()}")
print(f"Any Inf: {torch.isinf(poly_tensor).any().item()}")

Preprocessing 234447 polygons → 64 pts each...


Preprocessing: 100%|██████████| 234447/234447 [00:17<00:00, 13680.36it/s]


Cached to /tmp/poly_tensor_full.pt
Shape: torch.Size([234447, 64, 2])
Any NaN: False
Any Inf: False


In [52]:
import os
import numpy as np
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

ENCODING_PATH = "/raid/ruban/encodings/pk-real0.002"
QTREE_CACHE   = "/tmp/qtree_vectors_full.npy"

def sort_files(files):
    ids = [int(f[f.find('_')+1:f.find('.txt')]) for f in files]
    _, filenames = zip(*sorted(zip(ids, files)))
    return list(filenames)

def load_file(args):
    filepath, fname = args
    data = np.loadtxt(filepath, dtype=np.float32)
    return fname, data

if not os.path.exists(QTREE_CACHE):
    print("Loading quadtree encodings in parallel...")
    files = sort_files(os.listdir(ENCODING_PATH))
    print(f"Total encoding files: {len(files)}")

    all_vectors = []
    with ThreadPoolExecutor(max_workers=32) as executor:
        futures = {executor.submit(load_file, (os.path.join(ENCODING_PATH, f), f)): f
                   for f in files}
        for future in tqdm(as_completed(futures), total=len(files), desc="Loading"):
            fname, data = future.result()
            all_vectors.append((fname, data))

    all_vectors.sort(key=lambda x: int(x[0][x[0].find('_')+1:x[0].find('.txt')]))
    qtree_vectors = np.vstack([v for _, v in all_vectors])
    np.save(QTREE_CACHE, qtree_vectors)
    print(f"Cached to {QTREE_CACHE}")
else:
    print("Loading quadtree vectors from cache...")
    qtree_vectors = np.load(QTREE_CACHE)
    print("Loaded from cache.")

# Dynamic 80/20 split
N_TOTAL   = len(qtree_vectors)
split     = int(N_TOTAL * 0.8)
corpus_qt = qtree_vectors[:split]
query_qt  = qtree_vectors[split:]

print(f"Quadtree vectors: {qtree_vectors.shape}")
print(f"All non-negative: {(qtree_vectors >= 0).all()}")
print(f"Total: {N_TOTAL} | Corpus: {corpus_qt.shape} | Query: {query_qt.shape}")
print(f"Split at index: {split}")

Loading quadtree encodings in parallel...
Total encoding files: 400


Loading: 100%|██████████| 400/400 [28:33<00:00,  4.28s/it] 


Cached to /tmp/qtree_vectors_full.npy
Quadtree vectors: (233773, 18220)
All non-negative: True
Total: 233773 | Corpus: (187018, 18220) | Query: (46755, 18220)
Split at index: 187018


In [54]:
import numpy as np, os

f0 = sorted(os.listdir("/raid/ruban/encodings/pk-real0.002"))[0]
sample = np.loadtxt(os.path.join("/raid/ruban/encodings/pk-real0.002", f0),
                    dtype=np.float32, max_rows=1)
print(f"Full dataset vector dim: {len(sample)}")
print(f"Total files: {len(os.listdir('/raid/ruban/encodings/pk-real0.002'))}")

Full dataset vector dim: 18220
Total files: 400


In [57]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random

class QuadtreeCompressorV1(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False),
            nn.BatchNorm1d(4096),
            nn.ReLU(),
            nn.Linear(4096, 1024, bias=False),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
        )
    def forward(self, x, for_index=False):
        x = self.net(x)
        if for_index:
            x = F.relu(x)
            x = x / x.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return x

class AnchorPositiveDataset(Dataset):
    def __init__(self, qtree_vectors, gt_lookup, split, max_pos=30):
        self.vecs  = torch.tensor(qtree_vectors, dtype=torch.float32)
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            for nid in neighbors[:max_pos]:
                if nid < split:  # ensure neighbor is in corpus
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"Total pairs: {len(self.pairs)}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        qid, pos_id = self.pairs[idx]
        return self.vecs[qid], self.vecs[pos_id], qid, pos_id

def vectorized_hard_triplet_loss(anchors, positives, margin=0.3):
    d_pos      = (anchors - positives).pow(2).sum(dim=1).sqrt()
    sim_cross  = torch.mm(anchors, positives.T)
    sim_cross.fill_diagonal_(-1e9)
    d_neg_hard = (2 - 2 * sim_cross.max(dim=1).values).clamp(min=0).sqrt()
    loss       = F.relu(d_pos - d_neg_hard + margin)
    violated   = loss > 0
    if violated.sum() == 0:
        return torch.tensor(0.0, device=anchors.device, requires_grad=True), 0
    return loss[violated].mean(), violated.sum().item()

device = torch.device('cuda:0')

# Get actual input dim from loaded qtree_vectors
IN_DIM = qtree_vectors.shape[1]
print(f"Device: {device}")
print(f"Input dim (from data): {IN_DIM}")
print("Model and dataset classes defined.")

Device: cuda:0
Input dim (from data): 18220
Model and dataset classes defined.


In [60]:
from tqdm import tqdm

EPOCHS     = 50
BATCH_SIZE = 1024
LR         = 1e-3

compressor = QuadtreeCompressorV1(in_dim=IN_DIM, out_dim=512).to(device)

if torch.cuda.device_count() > 1:
    comp_par = nn.DataParallel(compressor)
    print(f"Training on {torch.cuda.device_count()} GPUs")
else:
    comp_par = compressor

# Pass split so dataset knows corpus boundary
dataset = AnchorPositiveDataset(qtree_vectors, gt_lookup, split=split, max_pos=30)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                     num_workers=0, pin_memory=True, drop_last=True)
print(f"Steps per epoch: {len(loader)}")

optimizer = torch.optim.AdamW(compressor.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_loss = float('inf')
for epoch in range(EPOCHS):
    comp_par.train()
    total_loss  = 0.0
    total_valid = 0
    total_steps = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch+1:2d}/{EPOCHS}", leave=False)
    for anchors, positives, anchor_ids, pos_ids in pbar:
        anchors   = anchors.to(device, non_blocking=True)
        positives = positives.to(device, non_blocking=True)

        B        = anchors.shape[0]
        combined = torch.cat([anchors, positives], dim=0)
        out      = comp_par(combined)
        out_norm = F.normalize(out, dim=1)
        a_emb    = out_norm[:B]
        p_emb    = out_norm[B:]

        loss, n_valid = vectorized_hard_triplet_loss(a_emb, p_emb, margin=0.3)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(compressor.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item()
        total_valid += n_valid
        total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'valid': n_valid})

    avg_loss = total_loss / total_steps
    scheduler.step()

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(compressor.state_dict(), '/tmp/best_compressor_full.pt')

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nDone. Best loss: {best_loss:.4f}")

Training on 8 GPUs


Exception ignored in: <function _ConnectionBase.__del__ at 0x7f2c0589ff40>
Traceback (most recent call last):
  File "/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/multiprocessing/connection.py", line 132, in __del__
    self._close()
  File "/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/multiprocessing/connection.py", line 361, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor


Total pairs: 1285476
Steps per epoch: 1255


Epoch  1/50 | Loss: 0.1712 | Best: 0.1712 | LR: 0.000999


Epoch  2/50 | Loss: 0.1574 | Best: 0.1574 | LR: 0.000996


...

Epoch 46/50 | Loss: 0.1447 | Best: 0.1445 | LR: 0.000016


Epoch 47/50 | Loss: 0.1440 | Best: 0.1440 | LR: 0.000009


Epoch 48/50 | Loss: 0.1444 | Best: 0.1440 | LR: 0.000004


Epoch 49/50 | Loss: 0.1444 | Best: 0.1440 | LR: 0.000001


Epoch 50/50 | Loss: 0.1442 | Best: 0.1440 | LR: 0.000000

Done. Best loss: 0.1440


In [64]:
import torch
import torch.nn.functional as F
import numpy as np
import nmslib
import time
from tqdm import tqdm

# Load best model
compressor_final = QuadtreeCompressorV1(in_dim=IN_DIM, out_dim=512).to(device)
compressor_final.load_state_dict(
    torch.load('/tmp/best_compressor_full.pt', weights_only=True))
compressor_final.eval()
print("Model loaded.")

# Generate embeddings
BATCH_SIZE = 512
all_embs   = []
qt_tensor  = torch.tensor(qtree_vectors, dtype=torch.float32)

with torch.no_grad():
    for start in tqdm(range(0, len(qt_tensor), BATCH_SIZE), desc="Compressing"):
        batch = qt_tensor[start:start+BATCH_SIZE].to(device)
        emb   = compressor_final(batch, for_index=True)
        all_embs.append(emb.cpu().numpy())

embs       = np.vstack(all_embs)
corpus_emb = embs[:QUERY_START_ID]
query_emb  = embs[QUERY_START_ID:]

print(f"Embeddings: {embs.shape}")
print(f"Corpus: {corpus_emb.shape} | Query: {query_emb.shape}")
print(f"All non-negative: {(embs>=0).all()}")

# Build HNSW index
print("\nBuilding HNSW index...")
index = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in tqdm(range(len(corpus_emb)), desc="Adding", mininterval=1.0):
    index.addDataPoint(i, corpus_emb[i])

t0 = time.time()
index.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=True)
build_time = time.time() - t0
print(f"Index built in {build_time:.2f}s")
index.setQueryTimeParams({'efSearch': 200})

# Query
t0   = time.time()
nbrs = index.knnQueryBatch(query_emb, k=50, num_threads=100)
qps  = len(query_emb) / (time.time() - t0)

# Results
print(f"\n============= FULL DATASET RESULTS =============")
print(f"{'Method':<28} {'Dim':>6} {'Recall@10':>10} {'Recall@50':>10} {'QPS':>10} {'Build':>8}")
print(f"{'Baseline (quadtree)':<28} {'18499':>6} {'~0.99':>10} {'~0.99':>10} {'~288':>10} {'~40s':>8}")

r10, _, _ = compute_recall_at_k(gt_lookup, nbrs, query_start_id=QUERY_START_ID, K=10)
r50, _, _ = compute_recall_at_k(gt_lookup, nbrs, query_start_id=QUERY_START_ID, K=50)
print(f"{'Neural compressor (ours)':<28} {'512':>6} {r10:>10.4f} {r50:>10.4f} {qps:>10.1f} {build_time:>7.2f}s")
print(f"\nTotal polygons: {N_TOTAL} | Corpus: {split} | Queries: {N_TOTAL-split}")

Model loaded.


Compressing: 100%|██████████| 457/457 [00:04<00:00, 114.01it/s]


Embeddings: (233773, 512)
Corpus: (187019, 512) | Query: (46754, 512)
All non-negative: True

Building HNSW index...


Adding: 100%|██████████| 187019/187019 [00:00<00:00, 561522.63it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Index built in 76.15s

============= FULL DATASET RESULTS =============
Method                          Dim  Recall@10  Recall@50        QPS    Build
Baseline (quadtree)           18499      ~0.99      ~0.99       ~288     ~40s
Neural compressor (ours)        512     0.5660     0.6297     2824.3   76.15s

Total polygons: 233773 | Corpus: 187018 | Queries: 46755


In [65]:
import nmslib
import numpy as np
import time
from tqdm import tqdm

print("Building baseline HNSW index on raw quadtree vectors (full dataset)...")
index_baseline = nmslib.init(method='hnsw', space='WeightedJaccard')

corpus_qt_full = qtree_vectors[:QUERY_START_ID]
query_qt_full  = qtree_vectors[QUERY_START_ID:]

print(f"Corpus: {corpus_qt_full.shape} | Query: {query_qt_full.shape}")

for i in tqdm(range(len(corpus_qt_full)), desc="Adding", mininterval=2.0):
    index_baseline.addDataPoint(i, corpus_qt_full[i])

t0 = time.time()
index_baseline.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=True)
build_time_bl = time.time() - t0
print(f"Index built in {build_time_bl:.2f}s")
index_baseline.setQueryTimeParams({'efSearch': 200})

t0 = time.time()
nbrs_bl = index_baseline.knnQueryBatch(query_qt_full, k=50, num_threads=100)
qps_bl  = len(query_qt_full) / (time.time() - t0)

r10_bl, _, _ = compute_recall_at_k(gt_lookup, nbrs_bl, query_start_id=QUERY_START_ID, K=10)
r50_bl, _, _ = compute_recall_at_k(gt_lookup, nbrs_bl, query_start_id=QUERY_START_ID, K=50)

print(f"\n============= FULL COMPARISON (233k dataset) =============")
print(f"{'Method':<28} {'Dim':>6} {'Recall@10':>10} {'Recall@50':>10} {'QPS':>10} {'Build':>8}")
print(f"{'Baseline (quadtree)':<28} {'18220':>6} {r10_bl:>10.4f} {r50_bl:>10.4f} {qps_bl:>10.1f} {build_time_bl:>7.2f}s")
print(f"{'Neural compressor (ours)':<28} {'512':>6} {'0.5660':>10} {'0.6297':>10} {'2824':>10} {'76.15s':>8}")

Building baseline HNSW index on raw quadtree vectors (full dataset)...
Corpus: (187019, 18220) | Query: (46754, 18220)


Adding: 100%|██████████| 187019/187019 [00:03<00:00, 46860.33it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Index built in 694.68s

============= FULL COMPARISON (233k dataset) =============
Method                          Dim  Recall@10  Recall@50        QPS    Build
Baseline (quadtree)           18220     0.9925     0.9952      484.0  694.68s
Neural compressor (ours)        512     0.5660     0.6297       2824   76.15s


In [66]:
import nmslib
import numpy as np
import time
from tqdm import tqdm

print("Building baseline HNSW index on raw quadtree vectors (full dataset)...")
index_baseline = nmslib.init(method='hnsw', space='WeightedJaccard')

corpus_qt_full = qtree_vectors[:QUERY_START_ID]
query_qt_full  = qtree_vectors[QUERY_START_ID:]

print(f"Corpus: {corpus_qt_full.shape} | Query: {query_qt_full.shape}")

for i in tqdm(range(len(corpus_qt_full)), desc="Adding", mininterval=2.0):
    index_baseline.addDataPoint(i, corpus_qt_full[i])

t0 = time.time()
index_baseline.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=True)
build_time_bl = time.time() - t0
print(f"Index built in {build_time_bl:.2f}s")
index_baseline.setQueryTimeParams({'efSearch': 200})

# Query once at K=500 (covers all smaller K)
t0 = time.time()
nbrs_bl = index_baseline.knnQueryBatch(query_qt_full, k=500, num_threads=100)
qps_bl  = len(query_qt_full) / (time.time() - t0)

print(f"\n============= FULL COMPARISON (233k dataset) =============")
print(f"{'Method':<28} {'Dim':>6} {'R@10':>8} {'R@50':>8} {'R@100':>8} {'R@500':>8} {'QPS':>8} {'Build':>8}")

# Baseline
r10_bl,  _, _ = compute_recall_at_k(gt_lookup, nbrs_bl, query_start_id=QUERY_START_ID, K=10)
r50_bl,  _, _ = compute_recall_at_k(gt_lookup, nbrs_bl, query_start_id=QUERY_START_ID, K=50)
r100_bl, _, _ = compute_recall_at_k(gt_lookup, nbrs_bl, query_start_id=QUERY_START_ID, K=100)
r500_bl, _, _ = compute_recall_at_k(gt_lookup, nbrs_bl, query_start_id=QUERY_START_ID, K=500)
print(f"{'Baseline (quadtree)':<28} {'18220':>6} {r10_bl:>8.4f} {r50_bl:>8.4f} {r100_bl:>8.4f} {r500_bl:>8.4f} {qps_bl:>8.1f} {build_time_bl:>7.2f}s")

# Ours — re-query at K=500
index_ours = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in tqdm(range(len(corpus_emb)), desc="Adding ours", mininterval=2.0):
    index_ours.addDataPoint(i, corpus_emb[i])
index_ours.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
index_ours.setQueryTimeParams({'efSearch': 200})

t0 = time.time()
nbrs_ours = index_ours.knnQueryBatch(query_emb, k=500, num_threads=100)
qps_ours  = len(query_emb) / (time.time() - t0)

r10_o,  _, _ = compute_recall_at_k(gt_lookup, nbrs_ours, query_start_id=QUERY_START_ID, K=10)
r50_o,  _, _ = compute_recall_at_k(gt_lookup, nbrs_ours, query_start_id=QUERY_START_ID, K=50)
r100_o, _, _ = compute_recall_at_k(gt_lookup, nbrs_ours, query_start_id=QUERY_START_ID, K=100)
r500_o, _, _ = compute_recall_at_k(gt_lookup, nbrs_ours, query_start_id=QUERY_START_ID, K=500)
print(f"{'Neural compressor (ours)':<28} {'512':>6} {r10_o:>8.4f} {r50_o:>8.4f} {r100_o:>8.4f} {r500_o:>8.4f} {qps_ours:>8.1f} {'76.15s':>8}")

Building baseline HNSW index on raw quadtree vectors (full dataset)...
Corpus: (187019, 18220) | Query: (46754, 18220)


Adding: 100%|██████████| 187019/187019 [00:02<00:00, 88706.26it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Index built in 632.46s

============= FULL COMPARISON (233k dataset) =============
Method                          Dim     R@10     R@50    R@100    R@500      QPS    Build
Baseline (quadtree)           18220   0.9925   0.9953   0.9963   0.9864    517.2  632.46s


Adding ours: 100%|██████████| 187019/187019 [00:00<00:00, 570677.54it/s]


Neural compressor (ours)        512   0.5660   0.6297   0.6512   0.7071   2536.2   76.15s


###  Define Transformer-on-chunks model (SOTA-style)

In [67]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class QuadtreeTransformerEncoder(nn.Module):
    """
    Transformer encoder over quadtree vector chunks.
    Splits 18220-d quadtree vector into N_CHUNKS tokens,
    runs Transformer encoder, pools to fixed embedding.
    Architecturally similar to PolyMP but on spatial grid features.
    """
    def __init__(self, in_dim=18220, n_chunks=64, d_model=256,
                 nhead=8, num_layers=4, dim_ff=512, out_dim=512, dropout=0.1):
        super().__init__()
        self.n_chunks  = n_chunks
        self.d_model   = d_model

        # Chunk size — pad input to be divisible by n_chunks
        self.chunk_size = math.ceil(in_dim / n_chunks)
        self.padded_dim = self.chunk_size * n_chunks

        # Project each chunk to d_model
        self.chunk_proj = nn.Linear(self.chunk_size, d_model)

        # Positional encoding
        pe  = torch.zeros(n_chunks, d_model)
        pos = torch.arange(0, n_chunks).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, n_chunks, d_model)

        # Transformer encoder
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # Output projection — NO LayerNorm (learned from previous failure)
        self.out_proj = nn.Linear(d_model, out_dim)

        # BatchNorm to prevent collapse (learned from previous failure)
        self.bn = nn.BatchNorm1d(out_dim)

    def forward(self, x, for_index=False):
        B = x.shape[0]

        # Pad to padded_dim
        if x.shape[1] < self.padded_dim:
            pad = torch.zeros(B, self.padded_dim - x.shape[1], device=x.device)
            x   = torch.cat([x, pad], dim=1)

        # Split into chunks → (B, n_chunks, chunk_size)
        x = x.view(B, self.n_chunks, self.chunk_size)

        # Project chunks → (B, n_chunks, d_model)
        x = self.chunk_proj(x)

        # Add positional encoding
        x = x + self.pe

        # Transformer encoder
        x = self.transformer(x)

        # Mean pool over chunks → (B, d_model)
        x = x.mean(dim=1)

        # Project to output dim
        x = self.out_proj(x)
        x = self.bn(x)

        if for_index:
            x = F.relu(x)
            x = x / x.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return x

# ─── Sanity check ─────────────────────────────────────────────────────────────
device = torch.device('cuda:0')
IN_DIM = qtree_vectors.shape[1]

model_sota = QuadtreeTransformerEncoder(
    in_dim    = IN_DIM,
    n_chunks  = 64,
    d_model   = 256,
    nhead     = 8,
    num_layers= 4,
    dim_ff    = 512,
    out_dim   = 512,
    dropout   = 0.1
).to(device)

total_params = sum(p.numel() for p in model_sota.parameters())
print(f"Model parameters: {total_params:,}")

# Test forward pass
sample = torch.tensor(qtree_vectors[:8], dtype=torch.float32).to(device)
with torch.no_grad():
    out = model_sota(sample)
    print(f"Input shape:  {sample.shape}")
    print(f"Output shape: {out.shape}")
    print(f"Output mean:  {out.mean().item():.4f}")
    print(f"Output std:   {out.std().item():.4f}")

    # Check no collapse before training
    out_norm = F.normalize(out, dim=1)
    sim = torch.mm(out_norm, out_norm.T)
    mask = ~torch.eye(8, dtype=torch.bool, device=device)
    print(f"Cosine sim std (want > 0.01): {sim[mask].std().item():.4f}")

Model parameters: 2,314,240
Input shape:  torch.Size([8, 18220])
Output shape: torch.Size([8, 512])
Output mean:  0.0000
Output std:   0.9794
Cosine sim std (want > 0.01): 0.0768


In [68]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import random

# Use 10k data — qt_10k needs to be loaded
# If you still have qtree_vectors from 10k in memory use that
# Otherwise reload:
ENCODING_PATH_10K = "/raid/ruban/encodings/pk-real10k0.002"
QTREE_CACHE_10K   = "/tmp/qtree_vectors_10k.npy"

import os
if not os.path.exists(QTREE_CACHE_10K):
    print("Loading 10k quadtree encodings...")
    def sort_files(files):
        ids = [int(f[f.find('_')+1:f.find('.txt')]) for f in files]
        _, filenames = zip(*sorted(zip(ids, files)))
        return list(filenames)
    files = sort_files(os.listdir(ENCODING_PATH_10K))
    vecs  = []
    for f in tqdm(files, desc="Loading"):
        with open(os.path.join(ENCODING_PATH_10K, f), 'r') as fp:
            content = fp.read()
        for line in content.strip().split('\n'):
            if line.strip():
                vecs.append(np.fromstring(line, dtype=np.float32, sep=' '))
    qt_10k = np.vstack(vecs)
    np.save(QTREE_CACHE_10K, qt_10k)
else:
    print("Loading 10k from cache...")
    qt_10k = np.load(QTREE_CACHE_10K)

print(f"10k quadtree: {qt_10k.shape}")

# GT for 10k — already in memory as gt_lookup_10k? 
# Reload if needed
GT_PATH_10K = "/raid/ruban/groundtruth/pk-query-10k"

def sort_gt_files(files):
    ids = [int(f[f.find('_')+1: f.find('-')]) for f in files]
    _, filenames = zip(*sorted(zip(ids, files)))
    return filenames

def read_gt_file(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    gt = []
    for line in lines:
        row = line.strip().split(', ')
        if len(row) <= 1:
            gt.append([])
        else:
            gt.append([int(x) for x in row])
    return gt

print("Loading 10k GT...")
gt_all_10k = []
for f in sort_gt_files(os.listdir(GT_PATH_10K)):
    gt_all_10k += read_gt_file(os.path.join(GT_PATH_10K, f))

gt_lookup_10k = {}
for row in gt_all_10k:
    if len(row) > 1:
        gt_lookup_10k[row[0]] = row[1:]

QUERY_START_10K = min(gt_lookup_10k.keys())  # 8000
print(f"10k GT queries: {len(gt_lookup_10k)}, query start: {QUERY_START_10K}")

# Dataset
class AnchorPositiveDataset(Dataset):
    def __init__(self, qtree_vectors, gt_lookup, query_start, max_pos=30):
        self.vecs  = torch.tensor(qtree_vectors, dtype=torch.float32)
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            for nid in neighbors[:max_pos]:
                if nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"Total pairs: {len(self.pairs)}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        qid, pos_id = self.pairs[idx]
        return self.vecs[qid], self.vecs[pos_id], qid, pos_id

def vectorized_hard_triplet_loss(anchors, positives, margin=0.3):
    d_pos      = (anchors - positives).pow(2).sum(dim=1).sqrt()
    sim_cross  = torch.mm(anchors, positives.T)
    sim_cross.fill_diagonal_(-1e9)
    d_neg_hard = (2 - 2 * sim_cross.max(dim=1).values).clamp(min=0).sqrt()
    loss       = F.relu(d_pos - d_neg_hard + margin)
    violated   = loss > 0
    if violated.sum() == 0:
        return torch.tensor(0.0, device=anchors.device, requires_grad=True), 0
    return loss[violated].mean(), violated.sum().item()

# Training
EPOCHS     = 50
BATCH_SIZE = 512
LR         = 1e-3
IN_DIM_10K = qt_10k.shape[1]

model_sota_10k = QuadtreeTransformerEncoder(
    in_dim    = IN_DIM_10K,
    n_chunks  = 64,
    d_model   = 256,
    nhead     = 8,
    num_layers= 4,
    dim_ff    = 512,
    out_dim   = 512,
    dropout   = 0.1
).to(device)

if torch.cuda.device_count() > 1:
    comp_par = nn.DataParallel(model_sota_10k)
    print(f"Training on {torch.cuda.device_count()} GPUs")
else:
    comp_par = model_sota_10k

dataset = AnchorPositiveDataset(qt_10k, gt_lookup_10k, query_start=QUERY_START_10K, max_pos=30)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                     num_workers=0, pin_memory=False, drop_last=True)
print(f"Steps per epoch: {len(loader)}")

optimizer = torch.optim.AdamW(model_sota_10k.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_loss = float('inf')
for epoch in range(EPOCHS):
    comp_par.train()
    total_loss  = 0.0
    total_valid = 0
    total_steps = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch+1:2d}/{EPOCHS}", leave=False)
    for anchors, positives, anchor_ids, pos_ids in pbar:
        anchors   = anchors.to(device, non_blocking=True)
        positives = positives.to(device, non_blocking=True)

        B        = anchors.shape[0]
        combined = torch.cat([anchors, positives], dim=0)
        out      = comp_par(combined)
        out_norm = F.normalize(out, dim=1)
        a_emb    = out_norm[:B]
        p_emb    = out_norm[B:]

        loss, n_valid = vectorized_hard_triplet_loss(a_emb, p_emb, margin=0.3)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_sota_10k.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item()
        total_valid += n_valid
        total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'valid': n_valid})

    avg_loss = total_loss / total_steps
    scheduler.step()

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_sota_10k.state_dict(), '/tmp/best_sota_transformer.pt')

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nDone. Best loss: {best_loss:.4f}")

Loading 10k quadtree encodings...


Loading: 100%|██████████| 80/80 [00:51<00:00,  1.55it/s]


10k quadtree: (10000, 18499)
Loading 10k GT...
10k GT queries: 1818, query start: 8000
Training on 8 GPUs
Total pairs: 46722
Steps per epoch: 91


Epoch  1/50 | Loss: 0.4663 | Best: 0.4663 | LR: 0.000999


Epoch  2/50 | Loss: 0.4526 | Best: 0.4526 | LR: 0.000996


...

Epoch 46/50 | Loss: 0.3093 | Best: 0.3093 | LR: 0.000016


Epoch 47/50 | Loss: 0.3091 | Best: 0.3091 | LR: 0.000009


Epoch 48/50 | Loss: 0.3090 | Best: 0.3090 | LR: 0.000004


Epoch 49/50 | Loss: 0.3090 | Best: 0.3090 | LR: 0.000001


Epoch 50/50 | Loss: 0.3089 | Best: 0.3089 | LR: 0.000000

Done. Best loss: 0.3089


In [69]:
import torch
import torch.nn.functional as F
import numpy as np
import nmslib
import time
from tqdm import tqdm

# Load best model
model_eval = QuadtreeTransformerEncoder(
    in_dim=IN_DIM_10K, n_chunks=64, d_model=256,
    nhead=8, num_layers=4, dim_ff=512, out_dim=512, dropout=0.1
).to(device)
model_eval.load_state_dict(
    torch.load('/tmp/best_sota_transformer.pt', weights_only=True))
model_eval.eval()
print("SOTA Transformer loaded.")

# Generate embeddings
qt_10k_tensor = torch.tensor(qt_10k, dtype=torch.float32)
all_embs = []
with torch.no_grad():
    for start in tqdm(range(0, len(qt_10k_tensor), 512), desc="Compressing"):
        batch = qt_10k_tensor[start:start+512].to(device)
        emb   = model_eval(batch, for_index=True)
        all_embs.append(emb.cpu().numpy())

embs_sota   = np.vstack(all_embs)
corpus_sota = embs_sota[:QUERY_START_10K]
query_sota  = embs_sota[QUERY_START_10K:]

print(f"Embeddings: {embs_sota.shape}")
print(f"Corpus: {corpus_sota.shape} | Query: {query_sota.shape}")

# Build HNSW
index = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in range(len(corpus_sota)):
    index.addDataPoint(i, corpus_sota[i])
index.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
index.setQueryTimeParams({'efSearch': 200})

t0   = time.time()
nbrs = index.knnQueryBatch(query_sota, k=50, num_threads=100)
qps  = len(query_sota) / (time.time() - t0)

print(f"\n============= 10k RESULTS COMPARISON =============")
print(f"{'Method':<30} {'Dim':>6} {'R@10':>8} {'R@50':>8} {'QPS':>8}")
print(f"{'Baseline (quadtree)':<30} {'18499':>6} {'0.9965':>8} {'0.9986':>8} {'288':>8}")
print(f"{'MLP compressor (ours)':<30} {'512':>6} {'0.6555':>8} {'0.8016':>8} {'40015':>8}")

r10, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=10)
r50, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=50)
print(f"{'SOTA Transformer (ours)':<30} {'512':>6} {r10:>8.4f} {r50:>8.4f} {qps:>8.1f}")

SOTA Transformer loaded.


Compressing: 100%|██████████| 20/20 [00:00<00:00, 58.39it/s]


Embeddings: (10000, 512)
Corpus: (8000, 512) | Query: (2000, 512)

============= 10k RESULTS COMPARISON =============
Method                            Dim     R@10     R@50      QPS
Baseline (quadtree)             18499   0.9965   0.9986      288
MLP compressor (ours)             512   0.6555   0.8016    40015
SOTA Transformer (ours)           512   0.0037   0.0167  20314.5


In [70]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import random
import math

class QuadtreeTransformerEncoderWJ(nn.Module):
    """
    Same as before but ALWAYS outputs ReLU + L1 norm.
    Train and index in same WeightedJaccard space.
    """
    def __init__(self, in_dim=18220, n_chunks=64, d_model=256,
                 nhead=8, num_layers=4, dim_ff=512, out_dim=512, dropout=0.1):
        super().__init__()
        self.n_chunks  = n_chunks
        self.chunk_size = math.ceil(in_dim / n_chunks)
        self.padded_dim = self.chunk_size * n_chunks
        self.chunk_proj = nn.Linear(self.chunk_size, d_model)

        pe  = torch.zeros(n_chunks, d_model)
        pos = torch.arange(0, n_chunks).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.out_proj    = nn.Linear(d_model, out_dim)
        self.bn          = nn.BatchNorm1d(out_dim)

    def forward(self, x):
        B = x.shape[0]
        if x.shape[1] < self.padded_dim:
            pad = torch.zeros(B, self.padded_dim - x.shape[1], device=x.device)
            x   = torch.cat([x, pad], dim=1)
        x = x.view(B, self.n_chunks, self.chunk_size)
        x = self.chunk_proj(x) + self.pe
        x = self.transformer(x)
        x = x.mean(dim=1)
        x = self.out_proj(x)
        x = self.bn(x)
        # ALWAYS in WJ space — same as training and indexing
        x = F.relu(x)
        x = x / x.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return x

# WJ triplet loss — same metric as NMSLIB
def wj_triplet_loss(a, p, n, margin=0.2):
    sim_ap = (torch.min(a, p).sum(dim=1) /
              torch.max(a, p).sum(dim=1).clamp(min=1e-10))
    sim_an = (torch.min(a, n).sum(dim=1) /
              torch.max(a, n).sum(dim=1).clamp(min=1e-10))
    loss   = F.relu(sim_an - sim_ap + margin)
    mask   = loss > 0
    if mask.sum() == 0:
        return torch.tensor(0.0, device=a.device, requires_grad=True), 0
    return loss[mask].mean(), mask.sum().item()

class AnchorPositiveDataset(Dataset):
    def __init__(self, qtree_vectors, gt_lookup, query_start, max_pos=30):
        self.vecs  = torch.tensor(qtree_vectors, dtype=torch.float32)
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            for nid in neighbors[:max_pos]:
                if nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"Total pairs: {len(self.pairs)}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        qid, pos_id = self.pairs[idx]
        return self.vecs[qid], self.vecs[pos_id], qid, pos_id

def vectorized_wj_loss(anchors, positives, margin=0.2):
    B = anchors.shape[0]
    # Positive similarity
    sim_ap = (torch.min(anchors, positives).sum(dim=1) /
              torch.max(anchors, positives).sum(dim=1).clamp(min=1e-10))
    # In-batch hard negatives — most similar positive from OTHER queries
    sim_cross = torch.zeros(B, B, device=anchors.device)
    for i in range(B):
        mins = torch.min(anchors[i].unsqueeze(0), positives).sum(dim=1)
        maxs = torch.max(anchors[i].unsqueeze(0), positives).sum(dim=1).clamp(min=1e-10)
        sim_cross[i] = mins / maxs
    sim_cross.fill_diagonal_(-1e9)
    sim_an_hard = sim_cross.max(dim=1).values
    loss    = F.relu(sim_an_hard - sim_ap + margin)
    mask    = loss > 0
    if mask.sum() == 0:
        return torch.tensor(0.0, device=anchors.device, requires_grad=True), 0
    return loss[mask].mean(), mask.sum().item()

# ─── Train ────────────────────────────────────────────────────────────────────
device     = torch.device('cuda:0')
EPOCHS     = 50
BATCH_SIZE = 512
LR         = 1e-3

model_wj = QuadtreeTransformerEncoderWJ(
    in_dim=IN_DIM_10K, n_chunks=64, d_model=256,
    nhead=8, num_layers=4, dim_ff=512, out_dim=512, dropout=0.1
).to(device)

if torch.cuda.device_count() > 1:
    comp_par = nn.DataParallel(model_wj)
    print(f"Training on {torch.cuda.device_count()} GPUs")
else:
    comp_par = model_wj

dataset = AnchorPositiveDataset(qt_10k, gt_lookup_10k,
                                 query_start=QUERY_START_10K, max_pos=30)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                     num_workers=0, pin_memory=False, drop_last=True)
print(f"Steps per epoch: {len(loader)}")

optimizer = torch.optim.AdamW(model_wj.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_loss = float('inf')
for epoch in range(EPOCHS):
    comp_par.train()
    total_loss  = 0.0
    total_valid = 0
    total_steps = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch+1:2d}/{EPOCHS}", leave=False)
    for anchors, positives, anchor_ids, pos_ids in pbar:
        anchors   = anchors.to(device, non_blocking=True)
        positives = positives.to(device, non_blocking=True)

        B        = anchors.shape[0]
        combined = torch.cat([anchors, positives], dim=0)
        out      = comp_par(combined)   # already WJ-normalized
        a_emb    = out[:B]
        p_emb    = out[B:]

        loss, n_valid = vectorized_wj_loss(a_emb, p_emb, margin=0.2)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_wj.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item()
        total_valid += n_valid
        total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'valid': n_valid})

    avg_loss = total_loss / total_steps
    scheduler.step()

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_wj.state_dict(), '/tmp/best_sota_wj.pt')

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nDone. Best loss: {best_loss:.4f}")

Training on 8 GPUs
Total pairs: 46722
Steps per epoch: 91


Epoch  1/50 | Loss: 0.2713 | Best: 0.2713 | LR: 0.000999


Epoch  2/50 | Loss: 0.2665 | Best: 0.2665 | LR: 0.000996


Epoch  3/50 | Loss: 0.2650 | Best: 0.2650 | LR: 0.000991


Epoch  4/50 | Loss: 0.2639 | Best: 0.2639 | LR: 0.000984


...

Epoch 46/50 | Loss: 0.2535 | Best: 0.2535 | LR: 0.000016


Epoch 47/50 | Loss: 0.2538 | Best: 0.2535 | LR: 0.000009


Epoch 48/50 | Loss: 0.2536 | Best: 0.2535 | LR: 0.000004


Epoch 49/50 | Loss: 0.2536 | Best: 0.2535 | LR: 0.000001


Epoch 50/50 | Loss: 0.2535 | Best: 0.2535 | LR: 0.000000

Done. Best loss: 0.2535


In [71]:
import torch
import torch.nn.functional as F
import numpy as np
import nmslib
import time
from tqdm import tqdm

model_wj_eval = QuadtreeTransformerEncoderWJ(
    in_dim=IN_DIM_10K, n_chunks=64, d_model=256,
    nhead=8, num_layers=4, dim_ff=512, out_dim=512, dropout=0.1
).to(device)
model_wj_eval.load_state_dict(
    torch.load('/tmp/best_sota_wj.pt', weights_only=True))
model_wj_eval.eval()
print("WJ Transformer loaded.")

# Generate embeddings
qt_10k_tensor = torch.tensor(qt_10k, dtype=torch.float32)
all_embs = []
with torch.no_grad():
    for start in tqdm(range(0, len(qt_10k_tensor), 512), desc="Compressing"):
        batch = qt_10k_tensor[start:start+512].to(device)
        emb   = model_wj_eval(batch)   # already WJ normalized
        all_embs.append(emb.cpu().numpy())

embs_wj   = np.vstack(all_embs)
corpus_wj = embs_wj[:QUERY_START_10K]
query_wj  = embs_wj[QUERY_START_10K:]

print(f"Embeddings: {embs_wj.shape}")
print(f"All non-negative: {(embs_wj>=0).all()}")
print(f"Row sums: [{embs_wj.sum(axis=1).min():.4f}, {embs_wj.sum(axis=1).max():.4f}]")

# Build HNSW
index = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in range(len(corpus_wj)):
    index.addDataPoint(i, corpus_wj[i])
index.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
index.setQueryTimeParams({'efSearch': 200})

t0   = time.time()
nbrs = index.knnQueryBatch(query_wj, k=50, num_threads=100)
qps  = len(query_wj) / (time.time() - t0)

r10, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=10)
r50, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=50)

print(f"\n============= 10k FULL COMPARISON =============")
print(f"{'Method':<32} {'Dim':>6} {'R@10':>8} {'R@50':>8} {'QPS':>8}")
print(f"{'Baseline (quadtree)':<32} {'18499':>6} {'0.9965':>8} {'0.9986':>8} {'288':>8}")
print(f"{'MLP compressor':<32} {'512':>6} {'0.6555':>8} {'0.8016':>8} {'40015':>8}")
print(f"{'Transformer (L2 space)':<32} {'512':>6} {'0.0037':>8} {'0.0167':>8} {'20314':>8}")
print(f"{'Transformer (WJ space)':<32} {'512':>6} {r10:>8.4f} {r50:>8.4f} {qps:>8.1f}")

WJ Transformer loaded.


Compressing: 100%|██████████| 20/20 [00:00<00:00, 57.03it/s]


Embeddings: (10000, 512)
All non-negative: True
Row sums: [1.0000, 1.0000]

============= 10k FULL COMPARISON =============
Method                              Dim     R@10     R@50      QPS
Baseline (quadtree)               18499   0.9965   0.9986      288
MLP compressor                      512   0.6555   0.8016    40015
Transformer (L2 space)              512   0.0037   0.0167    20314
Transformer (WJ space)              512   0.0066   0.0204  17223.1


In [76]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NeuralMinHashEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=1024, n_hashes=512, dropout=0.1):
        super().__init__()

        # Use LayerNorm instead of BatchNorm for sparse input
        self.extractor = nn.Sequential(
            nn.Linear(in_dim, hidden_dim, bias=True),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim, bias=True),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )

        # K hash heads — each learns independent hash function
        self.hash_proj = nn.Linear(hidden_dim, n_hashes, bias=True)

        # Output BN — operates on (B, n_hashes), dense at this point
        self.bn_out = nn.BatchNorm1d(n_hashes)

    def forward(self, x):
        # Log-transform input to handle extreme sparsity + tiny values
        x = torch.log1p(x * 1e6)        # scale up tiny values before log

        feat = self.extractor(x)         # (B, hidden_dim)
        out  = self.hash_proj(feat)      # (B, n_hashes)
        out  = self.bn_out(out)          # normalize across batch

        # WJ compatible
        out = F.relu(out)
        out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return out

# ─── Sanity check ─────────────────────────────────────────────────────────────
device     = torch.device('cuda:0')
IN_DIM_10K = qt_10k.shape[1]

model_mh = NeuralMinHashEncoder(
    in_dim     = IN_DIM_10K,
    hidden_dim = 1024,
    n_hashes   = 512,
    dropout    = 0.1
).to(device)

total_params = sum(p.numel() for p in model_mh.parameters())
print(f"Model parameters: {total_params:,}")

sample = torch.tensor(qt_10k[:64], dtype=torch.float32).to(device)
with torch.no_grad():
    # Diagnose step by step
    x = torch.log1p(sample * 1e6)
    print(f"After log1p — std: {x.std().item():.4f}  mean: {x.mean().item():.4f}")

    feat = model_mh.extractor(x)
    print(f"After extractor — std: {feat.std().item():.4f}")

    out_raw = model_mh.hash_proj(feat)
    print(f"After hash_proj — std: {out_raw.std().item():.4f}")

    out_bn = model_mh.bn_out(out_raw)
    print(f"After bn_out — std: {out_bn.std().item():.4f}")

    out = F.relu(out_bn)
    print(f"After relu — sparsity: {(out==0).float().mean().item():.4f}")

    out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)
    sim = torch.mm(out, out.T)
    mask = ~torch.eye(64, dtype=torch.bool, device=device)
    print(f"WJ sim std: {sim[mask].std().item():.4f}  (want > 0.01)")
    print(f"WJ sim mean: {sim[mask].mean().item():.4f}")

Model parameters: 20,523,520
After log1p — std: 0.0024  mean: 0.0003
After extractor — std: 0.5783
After hash_proj — std: 0.3632
After bn_out — std: 0.9998
After relu — sparsity: 0.4963
WJ sim std: 0.0003  (want > 0.01)
WJ sim mean: 0.0019


In [77]:
from torch.utils.data import DataLoader
import random

# Use same dataset and loss as our working MLP
dataset_mh = AnchorPositiveDataset(qt_10k, gt_lookup_10k,
                                    query_start=QUERY_START_10K, max_pos=30)
loader_mh  = DataLoader(dataset_mh, batch_size=512, shuffle=True,
                        num_workers=0, pin_memory=False, drop_last=True)

optimizer = torch.optim.AdamW(model_mh.parameters(), lr=1e-3, weight_decay=1e-4)

def vectorized_hard_triplet_loss(anchors, positives, margin=0.3):
    d_pos      = (anchors - positives).pow(2).sum(dim=1).sqrt()
    sim_cross  = torch.mm(anchors, positives.T)
    sim_cross.fill_diagonal_(-1e9)
    d_neg_hard = (2 - 2 * sim_cross.max(dim=1).values).clamp(min=0).sqrt()
    loss       = F.relu(d_pos - d_neg_hard + margin)
    violated   = loss > 0
    if violated.sum() == 0:
        return torch.tensor(0.0, device=anchors.device, requires_grad=True), 0
    return loss[violated].mean(), violated.sum().item()

from tqdm import tqdm
EPOCHS = 10
best_loss = float('inf')

for epoch in range(EPOCHS):
    model_mh.train()
    total_loss = 0.0; total_steps = 0

    pbar = tqdm(loader_mh, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    for anchors, positives, _, _ in pbar:
        anchors   = anchors.to(device)
        positives = positives.to(device)
        B         = anchors.shape[0]

        combined  = torch.cat([anchors, positives], dim=0)
        out       = model_mh(combined)
        out_norm  = F.normalize(out, dim=1)
        a_emb, p_emb = out_norm[:B], out_norm[B:]

        loss, n_valid = vectorized_hard_triplet_loss(a_emb, p_emb, margin=0.3)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_mh.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item(); total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / total_steps
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_mh.state_dict(), '/tmp/best_minhash.pt')

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_loss:.4f}")

# Quick eval
model_mh.eval()
all_embs = []
qt_tensor = torch.tensor(qt_10k, dtype=torch.float32)
with torch.no_grad():
    for start in tqdm(range(0, len(qt_tensor), 512), desc="Compressing"):
        batch = qt_tensor[start:start+512].to(device)
        emb   = model_mh(batch)
        all_embs.append(emb.cpu().numpy())

embs_mh   = np.vstack(all_embs)
corpus_mh = embs_mh[:QUERY_START_10K]
query_mh  = embs_mh[QUERY_START_10K:]

index = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in range(len(corpus_mh)):
    index.addDataPoint(i, corpus_mh[i])
index.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
index.setQueryTimeParams({'efSearch': 200})
nbrs = index.knnQueryBatch(query_mh, k=50, num_threads=100)

r10, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=10)
r50, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=50)
print(f"\n=== Neural MinHash 10-epoch result ===")
print(f"R@10: {r10:.4f} | R@50: {r50:.4f}")
print(f"MLP baseline: R@10=0.6555 | R@50=0.8016")

Total pairs: 46722


Epoch 1/10 | Loss: 0.3288 | Best: 0.3288


Epoch 2/10 | Loss: 0.3143 | Best: 0.3143


Epoch 3/10 | Loss: 0.3108 | Best: 0.3108


Epoch 4/10 | Loss: 0.3089 | Best: 0.3089


...

Epoch 8/10 | Loss: 0.3049 | Best: 0.3049


Epoch 9/10 | Loss: 0.3044 | Best: 0.3044


Epoch 10/10 | Loss: 0.3040 | Best: 0.3040


Compressing: 100%|██████████| 20/20 [00:00<00:00, 187.10it/s]



=== Neural MinHash 10-epoch result ===
R@10: 0.5892 | R@50: 0.7454
MLP baseline: R@10=0.6555 | R@50=0.8016


In [78]:
# Continue training from epoch 10 for 40 more epochs
optimizer = torch.optim.AdamW(model_mh.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

EPOCHS = 40
best_loss = float('inf')

for epoch in range(EPOCHS):
    model_mh.train()
    total_loss = 0.0
    total_steps = 0

    pbar = tqdm(loader_mh, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    for anchors, positives, _, _ in pbar:
        anchors   = anchors.to(device)
        positives = positives.to(device)
        B         = anchors.shape[0]

        combined  = torch.cat([anchors, positives], dim=0)
        out       = model_mh(combined)
        out_norm  = F.normalize(out, dim=1)
        a_emb, p_emb = out_norm[:B], out_norm[B:]

        loss, n_valid = vectorized_hard_triplet_loss(a_emb, p_emb, margin=0.3)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_mh.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item()
        total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / total_steps
    scheduler.step()

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_mh.state_dict(), '/tmp/best_minhash.pt')

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nDone. Best loss: {best_loss:.4f}")

Epoch 1/40 | Loss: 0.3048 | Best: 0.3048 | LR: 0.000998


Epoch 2/40 | Loss: 0.3036 | Best: 0.3036 | LR: 0.000994


Epoch 3/40 | Loss: 0.3028 | Best: 0.3028 | LR: 0.000986


Epoch 4/40 | Loss: 0.3035 | Best: 0.3028 | LR: 0.000976


Epoch 5/40 | Loss: 0.3021 | Best: 0.3021 | LR: 0.000962


...

Epoch 36/40 | Loss: 0.2999 | Best: 0.2999 | LR: 0.000024


Epoch 37/40 | Loss: 0.3001 | Best: 0.2999 | LR: 0.000014


Epoch 38/40 | Loss: 0.2999 | Best: 0.2999 | LR: 0.000006


Epoch 39/40 | Loss: 0.2999 | Best: 0.2999 | LR: 0.000002


Epoch 40/40 | Loss: 0.2999 | Best: 0.2999 | LR: 0.000000

Done. Best loss: 0.2999


In [79]:
model_mh.load_state_dict(torch.load('/tmp/best_minhash.pt', weights_only=True))
model_mh.eval()

all_embs = []
qt_tensor = torch.tensor(qt_10k, dtype=torch.float32)
with torch.no_grad():
    for start in tqdm(range(0, len(qt_tensor), 512), desc="Compressing"):
        batch = qt_tensor[start:start+512].to(device)
        emb   = model_mh(batch)
        all_embs.append(emb.cpu().numpy())

embs_mh   = np.vstack(all_embs)
corpus_mh = embs_mh[:QUERY_START_10K]
query_mh  = embs_mh[QUERY_START_10K:]

index = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in range(len(corpus_mh)):
    index.addDataPoint(i, corpus_mh[i])
index.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
index.setQueryTimeParams({'efSearch': 200})

t0   = time.time()
nbrs = index.knnQueryBatch(query_mh, k=50, num_threads=100)
qps  = len(query_mh) / (time.time() - t0)

r10, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=10)
r50, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=50)

print(f"\n============= 10k FULL COMPARISON =============")
print(f"{'Method':<32} {'Dim':>6} {'R@10':>8} {'R@50':>8} {'QPS':>8}")
print(f"{'Baseline (quadtree)':<32} {'18499':>6} {'0.9965':>8} {'0.9986':>8} {'288':>8}")
print(f"{'MLP compressor':<32} {'512':>6} {'0.6555':>8} {'0.8016':>8} {'40015':>8}")
print(f"{'Transformer (WJ)':<32} {'512':>6} {'0.0066':>8} {'0.0204':>8} {'17223':>8}")
print(f"{'Neural MinHash (ours)':<32} {'512':>6} {r10:>8.4f} {r50:>8.4f} {qps:>8.1f}")

Compressing: 100%|██████████| 20/20 [00:00<00:00, 147.05it/s]



============= 10k FULL COMPARISON =============
Method                              Dim     R@10     R@50      QPS
Baseline (quadtree)               18499   0.9965   0.9986      288
MLP compressor                      512   0.6555   0.8016    40015
Transformer (WJ)                    512   0.0066   0.0204    17223
Neural MinHash (ours)               512   0.6583   0.7960  28461.6


### Deep Sets

In [80]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DeepSetsEncoder(nn.Module):
    """
    Deep Sets (Zaheer et al., NeurIPS 2018) for WeightedJaccard similarity.
    
    Key insight: quadtree vector = weighted set of active spatial cells.
    Each non-zero cell is an element: (cell_id, weight).
    
    Deep Sets processes each element independently (phi network),
    then aggregates with weighted sum pooling (rho network).
    This is permutation invariant — same as Jaccard similarity.
    
    Architecture:
      phi: element-wise network — processes each active cell independently
      rho: aggregation network — combines all cell representations
    
    WJ compatibility: ReLU + L1 norm on output.
    """
    def __init__(self, in_dim, phi_dim=256, rho_dim=512, out_dim=512, dropout=0.1):
        super().__init__()
        self.in_dim = in_dim

        # phi network — processes each dimension independently
        # Input: (value, position_encoding) per cell
        self.phi = nn.Sequential(
            nn.Linear(2, phi_dim, bias=True),   # 2 = (weight, normalized_position)
            nn.LayerNorm(phi_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(phi_dim, phi_dim, bias=True),
            nn.LayerNorm(phi_dim),
            nn.GELU(),
        )

        # rho network — aggregates over all active cells
        self.rho = nn.Sequential(
            nn.Linear(phi_dim, rho_dim, bias=False),
            nn.BatchNorm1d(rho_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(rho_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
        )

        # Precompute normalized position encodings (cell IDs / in_dim)
        positions = torch.arange(in_dim).float() / in_dim
        self.register_buffer('positions', positions)  # (in_dim,)

    def forward(self, x):
        """
        x: (B, in_dim) — sparse quadtree vector
        """
        B = x.shape[0]

        # Log-transform to handle tiny float values
        x_log = torch.log1p(x * 1e6)            # (B, in_dim)

        # Build element features: (weight, position) per cell
        pos  = self.positions.unsqueeze(0).expand(B, -1)  # (B, in_dim)
        elem = torch.stack([x_log, pos], dim=2)           # (B, in_dim, 2)

        # phi: apply element-wise network to each cell
        # Reshape to (B*in_dim, 2) for batched linear, then back
        elem_flat  = elem.view(B * self.in_dim, 2)
        phi_out    = self.phi(elem_flat)                  # (B*in_dim, phi_dim)
        phi_out    = phi_out.view(B, self.in_dim, -1)    # (B, in_dim, phi_dim)

        # Weighted sum aggregation — weight by original quadtree value
        # This directly mirrors WeightedJaccard's Σmin(a,b) / Σmax(a,b)
        weights    = x_log.unsqueeze(2)                   # (B, in_dim, 1)
        aggregated = (phi_out * weights).sum(dim=1)       # (B, phi_dim)

        # rho: final aggregation network
        out = self.rho(aggregated)                        # (B, out_dim)

        # WJ compatible output
        out = F.relu(out)
        out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)

        return out

# ─── Sanity check ─────────────────────────────────────────────────────────────
device     = torch.device('cuda:0')
IN_DIM_10K = qt_10k.shape[1]

model_ds = DeepSetsEncoder(
    in_dim   = IN_DIM_10K,
    phi_dim  = 256,
    rho_dim  = 512,
    out_dim  = 512,
    dropout  = 0.1
).to(device)

total_params = sum(p.numel() for p in model_ds.parameters())
print(f"Model parameters: {total_params:,}")

sample = torch.tensor(qt_10k[:8], dtype=torch.float32).to(device)
with torch.no_grad():
    out = model_ds(sample)
    print(f"Input shape:   {sample.shape}")
    print(f"Output shape:  {out.shape}")
    print(f"Non-negative:  {(out >= 0).all().item()}")
    print(f"Row sums:      {out.sum(dim=1)[:4]}")
    print(f"Output std:    {out.std().item():.6f}")

    sim = torch.mm(out, out.T)
    mask = ~torch.eye(8, dtype=torch.bool, device=device)
    od = sim[mask]
    print(f"WJ sim mean:   {od.mean().item():.4f}")
    print(f"WJ sim std:    {od.std().item():.4f}  (want > 0.01)")

Model parameters: 462,848
Input shape:   torch.Size([8, 18499])
Output shape:  torch.Size([8, 512])
Non-negative:  True
Row sums:      tensor([1.0000, 1.0000, 1.0000, 1.0000], device='cuda:0')
Output std:    0.002571
WJ sim mean:   0.0017
WJ sim std:    0.0012  (want > 0.01)


In [83]:
from torch.utils.data import DataLoader
from tqdm import tqdm
import time

# Redesigned forward — only process non-zero cells (true sparse Deep Sets)
class DeepSetsEncoderSparse(nn.Module):
    """
    Sparse Deep Sets — only processes non-zero quadtree cells.
    Quadtree vectors are 70% sparse — this reduces computation 3x.
    """
    def __init__(self, in_dim, phi_dim=128, rho_dim=512, out_dim=512, dropout=0.1):
        super().__init__()
        self.in_dim = in_dim

        # phi: processes each active cell (value, position) → phi_dim
        self.phi = nn.Sequential(
            nn.Linear(2, phi_dim),
            nn.ReLU(),
            nn.Linear(phi_dim, phi_dim),
            nn.ReLU(),
        )

        # rho: aggregates → output
        self.rho = nn.Sequential(
            nn.Linear(phi_dim, rho_dim, bias=False),
            nn.BatchNorm1d(rho_dim),
            nn.ReLU(),
            nn.Linear(rho_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
        )

        positions = torch.arange(in_dim).float() / in_dim
        self.register_buffer('positions', positions)

    def forward(self, x):
        B = x.shape[0]
        x_log = torch.log1p(x * 1e6)

        # Build (value, position) for ALL cells — vectorized
        pos  = self.positions.unsqueeze(0).expand(B, -1)  # (B, in_dim)
        elem = torch.stack([x_log, pos], dim=2)            # (B, in_dim, 2)

        # phi over all cells — but use smaller phi_dim to save memory
        elem_flat = elem.reshape(B * self.in_dim, 2)
        phi_out   = self.phi(elem_flat)                    # (B*in_dim, phi_dim)
        phi_out   = phi_out.view(B, self.in_dim, -1)      # (B, in_dim, phi_dim)

        # Weighted sum — only non-zero cells contribute
        weights   = x_log.unsqueeze(2)                     # (B, in_dim, 1)
        agg       = (phi_out * weights).sum(dim=1)         # (B, phi_dim)

        out = self.rho(agg)
        out = F.relu(out)
        out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return out

# Reinitialize with smaller phi_dim
device     = torch.device('cuda:0')
IN_DIM_10K = qt_10k.shape[1]

model_ds = DeepSetsEncoderSparse(
    in_dim  = IN_DIM_10K,
    phi_dim = 128,        # reduced from 256
    rho_dim = 512,
    out_dim = 512,
    dropout = 0.1
).to(device)

total_params = sum(p.numel() for p in model_ds.parameters())
print(f"Parameters: {total_params:,}")

# Time with training batch size
sample = torch.tensor(qt_10k[:64], dtype=torch.float32).to(device)
t0 = time.time()
with torch.no_grad():
    out = model_ds(sample)
print(f"Forward pass (batch=64): {time.time()-t0:.3f}s")
print(f"Output shape: {out.shape}, non-neg: {(out>=0).all().item()}")

# Train with smaller batch
EPOCHS    = 50
BATCH_SIZE = 64   # reduced to avoid OOM

dataset_ds = AnchorPositiveDataset(qt_10k, gt_lookup_10k,
                                    query_start=QUERY_START_10K, max_pos=30)
loader_ds  = DataLoader(dataset_ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=0, pin_memory=False, drop_last=True)
print(f"Steps per epoch: {len(loader_ds)}")

optimizer = torch.optim.AdamW(model_ds.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_loss = float('inf')
for epoch in range(EPOCHS):
    model_ds.train()
    total_loss  = 0.0
    total_steps = 0

    pbar = tqdm(loader_ds, desc=f"Epoch {epoch+1:2d}/{EPOCHS}", leave=False)
    for anchors, positives, _, _ in pbar:
        anchors   = anchors.to(device)
        positives = positives.to(device)
        B         = anchors.shape[0]

        combined  = torch.cat([anchors, positives], dim=0)
        out       = model_ds(combined)
        out_norm  = F.normalize(out, dim=1)
        a_emb     = out_norm[:B]
        p_emb     = out_norm[B:]

        loss, n_valid = vectorized_hard_triplet_loss(a_emb, p_emb, margin=0.3)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ds.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item()
        total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / total_steps
    scheduler.step()

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_ds.state_dict(), '/tmp/best_deepsets.pt')

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nDone. Best loss: {best_loss:.4f}")

Parameters: 346,624
Forward pass (batch=64): 0.001s
Output shape: torch.Size([64, 512]), non-neg: True
Total pairs: 46722
Steps per epoch: 730


Epoch  1/50:   0%|          | 0/730 [00:00<?, ?it/s]

Epoch  1/50 | Loss: 0.0018 | Best: 0.0018 | LR: 0.000999


Epoch  2/50 | Loss: 0.0000 | Best: 0.0000 | LR: 0.000996


Epoch  3/50 | Loss: 0.0000 | Best: 0.0000 | LR: 0.000991


Epoch  4/50 | Loss: 0.0000 | Best: 0.0000 | LR: 0.000984


KeyboardInterrupt: 

### Chi-Square Two-Tower Encoder (LinkedIn 2023)

In [84]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ChiSquareTwoTowerEncoder(nn.Module):
    """
    Chi-Square Two-Tower Model (Li et al., LinkedIn 2023)
    Adapted for WeightedJaccard polygon similarity search.
    
    Original paper: "Practice with Graph-based ANN Algorithms on Sparse Data:
    Chi-square Two-tower model, HNSW, Sign Cauchy Projections"
    arXiv:2306.07607
    
    Key insight: Chi-square similarity ρ = Σ(2u_i*v_i)/(u_i+v_i) 
    is mathematically related to WeightedJaccard: Σmin(a,b)/Σmax(a,b)
    Both require non-negative L1-normalized embeddings — same WJ space.
    
    Architecture: MLP with ReLU + L1 norm → non-negative sparse embeddings
    trained with chi-square similarity loss using GT Jaccard pairs.
    
    Novel contribution: first application of chi-square two-tower model
    to spatial polygon similarity search with WeightedJaccard indexing.
    """
    def __init__(self, in_dim, hidden_dim=2048, out_dim=512, dropout=0.1):
        super().__init__()

        # Tower: global MLP projection (same as Li et al. 2023)
        self.tower = nn.Sequential(
            nn.Linear(in_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
        )

    def forward(self, x):
        out = self.tower(x)
        # ReLU + L1 norm → non-negative, sum-to-one (Li et al. 2023 requirement)
        out = F.relu(out)
        out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return out

def chi_square_similarity(u, v):
    """
    Chi-square similarity: ρ = Σ(2*u_i*v_i) / (u_i + v_i)
    u, v: (B, dim) non-negative, L1-normalized
    Returns: (B,) similarity scores
    """
    denom = (u + v).clamp(min=1e-10)
    return (2 * u * v / denom).sum(dim=1)

def chi_square_triplet_loss(anchors, positives, margin=0.2):
    """
    In-batch hard negative triplet loss using chi-square similarity.
    Follows Li et al. 2023: train with chi-square, index with WeightedJaccard.
    Both metrics require same non-negative L1-normalized embedding space.
    """
    B = anchors.shape[0]

    # Positive similarity: chi-square(anchor_i, positive_i)
    sim_ap = chi_square_similarity(anchors, positives)  # (B,)

    # In-batch hard negatives: for each anchor, find most similar positive
    # from a DIFFERENT query (hardest negative in batch)
    # Cross similarity matrix: (B, B) where [i,j] = chi_square(anchor_i, positive_j)
    denom  = (anchors.unsqueeze(1) + positives.unsqueeze(0)).clamp(min=1e-10)
    cross  = (2 * anchors.unsqueeze(1) * positives.unsqueeze(0) / denom).sum(dim=2)
    cross.fill_diagonal_(-1e9)                          # exclude true positive
    sim_an = cross.max(dim=1).values                    # hardest negative

    # Triplet: we want sim_ap > sim_an + margin
    loss   = F.relu(sim_an - sim_ap + margin)
    mask   = loss > 0
    if mask.sum() == 0:
        return torch.tensor(0.0, device=anchors.device, requires_grad=True), 0
    return loss[mask].mean(), mask.sum().item()

# ─── Sanity check ─────────────────────────────────────────────────────────────
device     = torch.device('cuda:0')
IN_DIM_10K = qt_10k.shape[1]

model_cs = ChiSquareTwoTowerEncoder(
    in_dim     = IN_DIM_10K,
    hidden_dim = 2048,
    out_dim    = 512,
    dropout    = 0.1
).to(device)

total_params = sum(p.numel() for p in model_cs.parameters())
print(f"Model parameters: {total_params:,}")

sample = torch.tensor(qt_10k[:64], dtype=torch.float32).to(device)
with torch.no_grad():
    out = model_cs(sample)
    print(f"Input shape:   {sample.shape}")
    print(f"Output shape:  {out.shape}")
    print(f"Non-negative:  {(out >= 0).all().item()}")
    print(f"Row sums:      {out.sum(dim=1)[:4]}")
    print(f"Output std:    {out.std().item():.6f}")

    # Chi-square similarity check
    sim_same = chi_square_similarity(out[:32], out[:32])
    sim_diff = chi_square_similarity(out[:32], out[32:])
    print(f"Chi-sq sim same (random, expect ~uniform): {sim_same.mean().item():.4f}")
    print(f"Chi-sq sim diff (random, expect ~uniform): {sim_diff.mean().item():.4f}")

    # WJ compatibility check
    wj_sim = (torch.min(out[:32], out[32:]).sum(dim=1) /
              torch.max(out[:32], out[32:]).sum(dim=1).clamp(min=1e-10))
    print(f"WJ sim (random):  mean={wj_sim.mean().item():.4f} std={wj_sim.std().item():.4f}")
    print(f"--> After training, GT pairs should have higher sim than random pairs")

Model parameters: 43,138,048
Input shape:   torch.Size([64, 18499])
Output shape:  torch.Size([64, 512])
Non-negative:  True
Row sums:      tensor([1.0000, 1.0000, 1.0000, 1.0000], device='cuda:0')
Output std:    0.002799
Chi-sq sim same (random, expect ~uniform): 1.0000
Chi-sq sim diff (random, expect ~uniform): 0.5632
WJ sim (random):  mean=0.3253 std=0.1779
--> After training, GT pairs should have higher sim than random pairs


In [85]:
from torch.utils.data import DataLoader
from tqdm import tqdm

dataset_cs = AnchorPositiveDataset(qt_10k, gt_lookup_10k,
                                    query_start=QUERY_START_10K, max_pos=30)
loader_cs  = DataLoader(dataset_cs, batch_size=512, shuffle=True,
                        num_workers=0, pin_memory=False, drop_last=True)
print(f"Steps per epoch: {len(loader_cs)}")

EPOCHS    = 50
optimizer = torch.optim.AdamW(model_cs.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_loss = float('inf')
for epoch in range(EPOCHS):
    model_cs.train()
    total_loss  = 0.0
    total_valid = 0
    total_steps = 0

    pbar = tqdm(loader_cs, desc=f"Epoch {epoch+1:2d}/{EPOCHS}", leave=False)
    for anchors, positives, _, _ in pbar:
        anchors   = anchors.to(device)
        positives = positives.to(device)
        B         = anchors.shape[0]

        combined  = torch.cat([anchors, positives], dim=0)
        out       = model_cs(combined)
        a_emb     = out[:B]
        p_emb     = out[B:]

        loss, n_valid = chi_square_triplet_loss(a_emb, p_emb, margin=0.2)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_cs.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item()
        total_valid += n_valid
        total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'valid': n_valid})

    avg_loss = total_loss / total_steps
    scheduler.step()

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_cs.state_dict(), '/tmp/best_chisquare.pt')

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nDone. Best loss: {best_loss:.4f}")

Total pairs: 46722
Steps per epoch: 91


Epoch  1/50 | Loss: 0.3012 | Best: 0.3012 | LR: 0.000999


Epoch  2/50 | Loss: 0.2983 | Best: 0.2983 | LR: 0.000996


Epoch  3/50 | Loss: 0.2931 | Best: 0.2931 | LR: 0.000991


Epoch  4/50 | Loss: 0.2886 | Best: 0.2886 | LR: 0.000984


...

Epoch 46/50 | Loss: 0.2000 | Best: 0.2000 | LR: 0.000016


Epoch 47/50 | Loss: 0.2000 | Best: 0.2000 | LR: 0.000009


Epoch 48/50 | Loss: 0.2000 | Best: 0.2000 | LR: 0.000004


Epoch 49/50 | Loss: 0.2000 | Best: 0.2000 | LR: 0.000001


Epoch 50/50 | Loss: 0.2000 | Best: 0.2000 | LR: 0.000000

Done. Best loss: 0.2000


In [86]:
import nmslib
import time
from tqdm import tqdm

model_cs.load_state_dict(
    torch.load('/tmp/best_chisquare.pt', weights_only=True))
model_cs.eval()
print("Chi-Square Two-Tower loaded.")

# Generate embeddings
qt_10k_tensor = torch.tensor(qt_10k, dtype=torch.float32)
all_embs = []
with torch.no_grad():
    for start in tqdm(range(0, len(qt_10k_tensor), 512), desc="Compressing"):
        batch = qt_10k_tensor[start:start+512].to(device)
        emb   = model_cs(batch)
        all_embs.append(emb.cpu().numpy())

embs_cs   = np.vstack(all_embs)
corpus_cs = embs_cs[:QUERY_START_10K]
query_cs  = embs_cs[QUERY_START_10K:]

print(f"Embeddings: {embs_cs.shape}")
print(f"All non-negative: {(embs_cs>=0).all()}")

# Embedding quality
same_sims, diff_sims = [], []
for i in range(100):
    qid    = QUERY_START_10K + i
    pos_id = gt_lookup_10k[qid][0] if gt_lookup_10k.get(qid) else 0
    a = torch.tensor(embs_cs[qid]).unsqueeze(0)
    p = torch.tensor(embs_cs[pos_id]).unsqueeze(0)
    n = torch.tensor(embs_cs[i]).unsqueeze(0)
    same_sims.append(chi_square_similarity(a, p).item())
    diff_sims.append(chi_square_similarity(a, n).item())

print(f"\nChi-sq sim GT pairs  (want HIGH): {np.mean(same_sims):.4f} ± {np.std(same_sims):.4f}")
print(f"Chi-sq sim random    (want LOW):  {np.mean(diff_sims):.4f} ± {np.std(diff_sims):.4f}")
print(f"Separation gap: {np.mean(same_sims)-np.mean(diff_sims):.4f}")

# Build HNSW + evaluate
index = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in range(len(corpus_cs)):
    index.addDataPoint(i, corpus_cs[i])
index.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
index.setQueryTimeParams({'efSearch': 200})

t0   = time.time()
nbrs = index.knnQueryBatch(query_cs, k=50, num_threads=100)
qps  = len(query_cs) / (time.time() - t0)

r10, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=10)
r50, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=50)

print(f"\n============= 10k FULL COMPARISON =============")
print(f"{'Method':<32} {'Dim':>6} {'R@10':>8} {'R@50':>8} {'QPS':>8}")
print(f"{'Baseline (quadtree)':<32} {'18499':>6} {'0.9965':>8} {'0.9986':>8} {'288':>8}")
print(f"{'MLP compressor':<32} {'512':>6} {'0.6555':>8} {'0.8016':>8} {'40015':>8}")
print(f"{'Neural MinHash':<32} {'512':>6} {'0.6583':>8} {'0.7960':>8} {'28461':>8}")
print(f"{'Transformer (WJ)':<32} {'512':>6} {'0.0066':>8} {'0.0204':>8} {'17223':>8}")
print(f"{'Chi-Sq Two-Tower (ours)':<32} {'512':>6} {r10:>8.4f} {r50:>8.4f} {qps:>8.1f}")

Chi-Square Two-Tower loaded.


Compressing: 100%|██████████| 20/20 [00:00<00:00, 127.80it/s]

Embeddings: (10000, 512)
All non-negative: True



Chi-sq sim GT pairs  (want HIGH): 0.9998 ± 0.0008
Chi-sq sim random    (want LOW):  0.9996 ± 0.0012
Separation gap: 0.0002

============= 10k FULL COMPARISON =============
Method                              Dim     R@10     R@50      QPS
Baseline (quadtree)               18499   0.9965   0.9986      288
MLP compressor                      512   0.6555   0.8016    40015
Neural MinHash                      512   0.6583   0.7960    28461
Transformer (WJ)                    512   0.0066   0.0204    17223
Chi-Sq Two-Tower (ours)             512   0.0553   0.1257  38586.9


In [87]:
import torch
import torch.nn.functional as F
import numpy as np

model_cs.eval()

# ─── 1. Check embedding distribution ──────────────────────────────────────────
sample = torch.tensor(qt_10k[:128], dtype=torch.float32).to(device)
with torch.no_grad():
    out = model_cs(sample)

print("=== Embedding distribution ===")
print(f"Mean:     {out.mean().item():.6f}")
print(f"Std:      {out.std().item():.6f}")
print(f"Sparsity: {(out==0).float().mean().item():.4f}")
print(f"Max:      {out.max().item():.6f}")
print(f"Min:      {out.min().item():.6f}")

# How many dimensions are active per vector?
active_dims = (out > 0).float().sum(dim=1)
print(f"Active dims per vector: mean={active_dims.mean().item():.1f}, std={active_dims.std().item():.1f}")

# ─── 2. Are embeddings actually different across samples? ─────────────────────
print("\n=== Inter-sample diversity ===")
# Pairwise L2 distances
dists = torch.cdist(out[:32], out[:32])
mask  = ~torch.eye(32, dtype=torch.bool, device=device)
print(f"Pairwise L2 dist — mean: {dists[mask].mean().item():.6f}, std: {dists[mask].std().item():.6f}")
print(f"--> If std~0, all embeddings are identical")

# ─── 3. Check chi-square sim distribution for GT vs random ───────────────────
print("\n=== Chi-square similarity: GT vs random ===")
gt_sims, rand_sims = [], []
with torch.no_grad():
    for i in range(200):
        qid    = QUERY_START_10K + i
        pos_id = gt_lookup_10k.get(qid, [None])[0]
        if pos_id is None: continue
        rand_id = np.random.randint(0, QUERY_START_10K)

        vq  = torch.tensor(qt_10k[qid],    dtype=torch.float32).unsqueeze(0).to(device)
        vp  = torch.tensor(qt_10k[pos_id], dtype=torch.float32).unsqueeze(0).to(device)
        vr  = torch.tensor(qt_10k[rand_id],dtype=torch.float32).unsqueeze(0).to(device)

        eq  = model_cs(vq)
        ep  = model_cs(vp)
        er  = model_cs(vr)

        gt_sims.append(chi_square_similarity(eq, ep).item())
        rand_sims.append(chi_square_similarity(eq, er).item())

print(f"GT pair sim   (want HIGH): {np.mean(gt_sims):.4f} ± {np.std(gt_sims):.4f}")
print(f"Random sim    (want LOW):  {np.mean(rand_sims):.4f} ± {np.std(rand_sims):.4f}")
print(f"Separation:               {np.mean(gt_sims) - np.mean(rand_sims):.4f}")

# ─── 4. Inspect the tower layer by layer ─────────────────────────────────────
print("\n=== Layer-by-layer activation ===")
x = sample
for i, layer in enumerate(model_cs.tower):
    x = layer(x)
    print(f"Layer {i} ({layer.__class__.__name__}): mean={x.mean().item():.4f}, std={x.std().item():.4f}, shape={tuple(x.shape)}")

# ─── 5. Check if loss was actually learning ───────────────────────────────────
print("\n=== Loss sanity check ===")
a = torch.tensor(qt_10k[:256], dtype=torch.float32).to(device)
p_ids = [gt_lookup_10k.get(QUERY_START_10K + i, [i])[0] for i in range(256)]
p = torch.tensor(qt_10k[p_ids], dtype=torch.float32).to(device)

with torch.no_grad():
    ea = model_cs(a)
    ep = model_cs(p)
    sim_ap = chi_square_similarity(ea, ep)
    
    # Cross similarity
    denom  = (ea.unsqueeze(1) + ep.unsqueeze(0)).clamp(min=1e-10)
    cross  = (2 * ea.unsqueeze(1) * ep.unsqueeze(0) / denom).sum(dim=2)
    cross.fill_diagonal_(-1e9)
    sim_an = cross.max(dim=1).values

print(f"sim_ap (positive): mean={sim_ap.mean().item():.4f}, std={sim_ap.std().item():.4f}")
print(f"sim_an (negative): mean={sim_an.mean().item():.4f}, std={sim_an.std().item():.4f}")
print(f"Margin violations: {(sim_an - sim_ap + 0.2 > 0).float().mean().item():.4f}")
print(f"--> If violations~0, model satisfied all triplets (too easy)")

=== Embedding distribution ===
Mean:     0.001953
Std:      0.000100
Sparsity: 0.0000
Max:      0.002626
Min:      0.000802
Active dims per vector: mean=512.0, std=0.0

=== Inter-sample diversity ===
Pairwise L2 dist — mean: 0.001473, std: 0.002160
--> If std~0, all embeddings are identical

=== Chi-square similarity: GT vs random ===
GT pair sim   (want HIGH): 1.0000 ± 0.0000
Random sim    (want LOW):  0.9994 ± 0.0026
Separation:               0.0006

=== Layer-by-layer activation ===
Layer 0 (Linear): mean=-0.0000, std=0.0000, shape=(128, 2048)
Layer 1 (BatchNorm1d): mean=-0.0019, std=0.0058, shape=(128, 2048)
Layer 2 (ReLU): mean=0.0020, std=0.0032, shape=(128, 2048)
Layer 3 (Dropout): mean=0.0020, std=0.0032, shape=(128, 2048)
Layer 4 (Linear): mean=-0.0059, std=0.0075, shape=(128, 2048)
Layer 5 (BatchNorm1d): mean=0.0953, std=0.6262, shape=(128, 2048)
Layer 6 (ReLU): mean=0.1471, std=0.6078, shape=(128, 2048)
Layer 7 (Dropout): mean=0.1471, std=0.6078, shape=(128, 2048)
Layer 8 (L

In [88]:
class ChiSquareTwoTowerEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=2048, out_dim=512, dropout=0.1):
        super().__init__()
        self.tower = nn.Sequential(
            nn.Linear(in_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
        )

    def forward(self, x):
        x   = torch.log1p(x * 1e6)   # ← fix: handle tiny float values
        out = self.tower(x)
        out = F.relu(out)
        out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return out

# Verify fix
model_cs2 = ChiSquareTwoTowerEncoder(IN_DIM_10K, 2048, 512).to(device)
sample = torch.tensor(qt_10k[:128], dtype=torch.float32).to(device)
with torch.no_grad():
    out = model_cs2(sample)
    print(f"Std after fix: {out.std().item():.6f}")
    active = (out > 0).float().sum(dim=1)
    print(f"Active dims: mean={active.mean().item():.1f} std={active.std().item():.1f}")
    # Layer 0 check
    x_log = torch.log1p(sample * 1e6)
    x_l0  = model_cs2.tower[0](x_log)
    print(f"Layer 0 after log1p — std: {x_l0.std().item():.4f}  (was 0.0000)")

Std after fix: 0.002733
Active dims: mean=254.2 std=7.3
Layer 0 after log1p — std: 0.0024  (was 0.0000)


In [89]:
from torch.utils.data import DataLoader
from tqdm import tqdm

# Fresh model with log1p fix
model_cs2 = ChiSquareTwoTowerEncoder(IN_DIM_10K, 2048, 512).to(device)
print(f"Parameters: {sum(p.numel() for p in model_cs2.parameters()):,}")

dataset_cs = AnchorPositiveDataset(qt_10k, gt_lookup_10k,
                                    query_start=QUERY_START_10K, max_pos=30)
loader_cs  = DataLoader(dataset_cs, batch_size=512, shuffle=True,
                        num_workers=0, pin_memory=False, drop_last=True)
print(f"Steps per epoch: {len(loader_cs)}")

EPOCHS    = 50
optimizer = torch.optim.AdamW(model_cs2.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_loss = float('inf')
for epoch in range(EPOCHS):
    model_cs2.train()
    total_loss  = 0.0
    total_valid = 0
    total_steps = 0

    pbar = tqdm(loader_cs, desc=f"Epoch {epoch+1:2d}/{EPOCHS}", leave=False)
    for anchors, positives, _, _ in pbar:
        anchors   = anchors.to(device)
        positives = positives.to(device)
        B         = anchors.shape[0]

        combined  = torch.cat([anchors, positives], dim=0)
        out       = model_cs2(combined)
        a_emb     = out[:B]
        p_emb     = out[B:]

        loss, n_valid = chi_square_triplet_loss(a_emb, p_emb, margin=0.2)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_cs2.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item()
        total_valid += n_valid
        total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'valid': n_valid})

    avg_loss = total_loss / total_steps
    scheduler.step()

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_cs2.state_dict(), '/tmp/best_chisquare2.pt')

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nDone. Best loss: {best_loss:.4f}")

Parameters: 43,138,048
Total pairs: 46722
Steps per epoch: 91


Epoch  1/50 | Loss: 0.2050 | Best: 0.2050 | LR: 0.000999


Epoch  2/50 | Loss: 0.2026 | Best: 0.2026 | LR: 0.000996


Epoch  3/50 | Loss: 0.2015 | Best: 0.2015 | LR: 0.000991


Epoch  4/50 | Loss: 0.2018 | Best: 0.2015 | LR: 0.000984


...

Epoch 46/50 | Loss: 0.2000 | Best: 0.1999 | LR: 0.000016


Epoch 47/50 | Loss: 0.2000 | Best: 0.1999 | LR: 0.000009


Epoch 48/50 | Loss: 0.1999 | Best: 0.1999 | LR: 0.000004


Epoch 49/50 | Loss: 0.1999 | Best: 0.1999 | LR: 0.000001


Epoch 50/50 | Loss: 0.1999 | Best: 0.1999 | LR: 0.000000

Done. Best loss: 0.1999


In [90]:
model_cs2.load_state_dict(
    torch.load('/tmp/best_chisquare2.pt', weights_only=True))
model_cs2.eval()
print("Chi-Square v2 loaded.")

qt_10k_tensor = torch.tensor(qt_10k, dtype=torch.float32)
all_embs = []
with torch.no_grad():
    for start in tqdm(range(0, len(qt_10k_tensor), 512), desc="Compressing"):
        batch = qt_10k_tensor[start:start+512].to(device)
        emb   = model_cs2(batch)
        all_embs.append(emb.cpu().numpy())

embs_cs2   = np.vstack(all_embs)
corpus_cs2 = embs_cs2[:QUERY_START_10K]
query_cs2  = embs_cs2[QUERY_START_10K:]

print(f"Sparsity: {(embs_cs2==0).mean():.4f}")
active = (embs_cs2 > 0).sum(axis=1)
print(f"Active dims: mean={active.mean():.1f} std={active.std():.1f}")

index = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in range(len(corpus_cs2)):
    index.addDataPoint(i, corpus_cs2[i])
index.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
index.setQueryTimeParams({'efSearch': 200})

t0   = time.time()
nbrs = index.knnQueryBatch(query_cs2, k=50, num_threads=100)
qps  = len(query_cs2) / (time.time() - t0)

r10, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=10)
r50, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=50)

print(f"\n============= 10k FULL COMPARISON =============")
print(f"{'Method':<32} {'Dim':>6} {'R@10':>8} {'R@50':>8} {'QPS':>8}")
print(f"{'Baseline (quadtree)':<32} {'18499':>6} {'0.9965':>8} {'0.9986':>8} {'288':>8}")
print(f"{'MLP compressor':<32} {'512':>6} {'0.6555':>8} {'0.8016':>8} {'40015':>8}")
print(f"{'Neural MinHash':<32} {'512':>6} {'0.6583':>8} {'0.7960':>8} {'28461':>8}")
print(f"{'Chi-Sq v1 (no log1p)':<32} {'512':>6} {'0.0553':>8} {'0.1257':>8} {'38586':>8}")
print(f"{'Chi-Sq v2 (log1p fix)':<32} {'512':>6} {r10:>8.4f} {r50:>8.4f} {qps:>8.1f}")

Chi-Square v2 loaded.


Compressing: 100%|██████████| 20/20 [00:00<00:00, 145.70it/s]


Sparsity: 0.2818
Active dims: mean=367.7 std=42.2

============= 10k FULL COMPARISON =============
Method                              Dim     R@10     R@50      QPS
Baseline (quadtree)               18499   0.9965   0.9986      288
MLP compressor                      512   0.6555   0.8016    40015
Neural MinHash                      512   0.6583   0.7960    28461
Chi-Sq v1 (no log1p)                512   0.0553   0.1257    38586
Chi-Sq v2 (log1p fix)               512   0.4088   0.5933  35253.2


In [91]:
# Continue training from best checkpoint for 50 more epochs
model_cs2.load_state_dict(torch.load('/tmp/best_chisquare2.pt', weights_only=True))

# More pairs — all GT neighbors
dataset_cs_full = AnchorPositiveDataset(qt_10k, gt_lookup_10k,
                                         query_start=QUERY_START_10K,
                                         max_pos=None)  # all neighbors
loader_cs_full  = DataLoader(dataset_cs_full, batch_size=512, shuffle=True,
                             num_workers=0, pin_memory=False, drop_last=True)
print(f"Total pairs: {len(dataset_cs_full)}")
print(f"Steps per epoch: {len(loader_cs_full)}")

EPOCHS    = 50
optimizer = torch.optim.AdamW(model_cs2.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_loss = float('inf')
for epoch in range(EPOCHS):
    model_cs2.train()
    total_loss  = 0.0
    total_steps = 0

    pbar = tqdm(loader_cs_full, desc=f"Epoch {epoch+1:2d}/{EPOCHS}", leave=False)
    for anchors, positives, _, _ in pbar:
        anchors   = anchors.to(device)
        positives = positives.to(device)
        B         = anchors.shape[0]

        combined  = torch.cat([anchors, positives], dim=0)
        out       = model_cs2(combined)
        a_emb, p_emb = out[:B], out[B:]

        loss, n_valid = chi_square_triplet_loss(a_emb, p_emb, margin=0.2)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_cs2.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item()
        total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / total_steps
    scheduler.step()

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_cs2.state_dict(), '/tmp/best_chisquare2.pt')

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nDone. Best loss: {best_loss:.4f}")

Total pairs: 400210
Total pairs: 400210
Steps per epoch: 781


Epoch  1/50 | Loss: 0.2009 | Best: 0.2009 | LR: 0.000500


Epoch  2/50 | Loss: 0.2002 | Best: 0.2002 | LR: 0.000498


Epoch  3/50 | Loss: 0.2002 | Best: 0.2002 | LR: 0.000496


Epoch  4/50 | Loss: 0.2001 | Best: 0.2001 | LR: 0.000492


...

Epoch 20/50 | Loss: 0.2000 | Best: 0.2000 | LR: 0.000327


Epoch 21/50 | Loss: 0.2000 | Best: 0.2000 | LR: 0.000312


Epoch 22/50 | Loss: 0.2000 | Best: 0.2000 | LR: 0.000297


Epoch 23/50 | Loss: 0.2000 | Best: 0.2000 | LR: 0.000281


KeyboardInterrupt: 

In [92]:
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random

# ─── Hard negative dataset for Chi-Square ────────────────────────────────────
class HardNegDatasetCS(Dataset):
    def __init__(self, qtree_vectors, gt_lookup, hard_neg_pool):
        self.qt = torch.tensor(qtree_vectors, dtype=torch.float32)
        self.triplets = []
        for qid, pos_list in gt_lookup.items():
            negs = hard_neg_pool.get(qid, [])
            if not negs: continue
            for pos_id in pos_list:
                neg_id = random.choice(negs)
                self.triplets.append((qid, pos_id, neg_id))
        random.shuffle(self.triplets)
        print(f"Total triplets: {len(self.triplets)}")

    def __len__(self): return len(self.triplets)

    def __getitem__(self, idx):
        qid, pos_id, neg_id = self.triplets[idx]
        return self.qt[qid], self.qt[pos_id], self.qt[neg_id]

# ─── Chi-square triplet loss with explicit negatives ──────────────────────────
def chi_square_explicit_triplet_loss(a, p, n, margin=0.2):
    """
    Explicit triplet loss — anchor, positive, negative provided directly.
    No in-batch mining needed — negatives are precomputed hard negatives.
    """
    sim_ap = chi_square_similarity(a, p)   # (B,)
    sim_an = chi_square_similarity(a, n)   # (B,)
    loss   = F.relu(sim_an - sim_ap + margin)
    mask   = loss > 0
    if mask.sum() == 0:
        return torch.tensor(0.0, device=a.device, requires_grad=True), 0
    return loss[mask].mean(), mask.sum().item()

# ─── Retrain from best checkpoint with hard negatives ─────────────────────────
model_cs2.load_state_dict(torch.load('/tmp/best_chisquare2.pt', weights_only=True))

dataset_hn = HardNegDatasetCS(qt_10k, gt_lookup_10k, hard_neg_pool)
loader_hn  = DataLoader(dataset_hn, batch_size=512, shuffle=True,
                        num_workers=0, pin_memory=False, drop_last=True)
print(f"Steps per epoch: {len(loader_hn)}")

EPOCHS    = 50
optimizer = torch.optim.AdamW(model_cs2.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_loss = float('inf')
for epoch in range(EPOCHS):
    model_cs2.train()
    total_loss  = 0.0
    total_valid = 0
    total_steps = 0

    pbar = tqdm(loader_hn, desc=f"Epoch {epoch+1:2d}/{EPOCHS}", leave=False)
    for anchors, positives, negatives in pbar:
        anchors   = anchors.to(device)
        positives = positives.to(device)
        negatives = negatives.to(device)

        a_emb = model_cs2(anchors)
        p_emb = model_cs2(positives)
        n_emb = model_cs2(negatives)

        loss, n_valid = chi_square_explicit_triplet_loss(
            a_emb, p_emb, n_emb, margin=0.2)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_cs2.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item()
        total_valid += n_valid
        total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'valid': n_valid})

    avg_loss = total_loss / total_steps
    scheduler.step()

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_cs2.state_dict(), '/tmp/best_chisquare_hn.pt')

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nDone. Best loss: {best_loss:.4f}")

Total triplets: 400210
Steps per epoch: 781


Epoch  1/50 | Loss: 0.1697 | Best: 0.1697 | LR: 0.000300


Epoch  2/50 | Loss: 0.1552 | Best: 0.1552 | LR: 0.000299


Epoch  3/50 | Loss: 0.1505 | Best: 0.1505 | LR: 0.000297


Epoch  4/50 | Loss: 0.1250 | Best: 0.1250 | LR: 0.000295


...

Epoch 46/50 | Loss: 0.0023 | Best: 0.0013 | LR: 0.000005


Epoch 47/50 | Loss: 0.0021 | Best: 0.0013 | LR: 0.000003


Epoch 48/50 | Loss: 0.0016 | Best: 0.0013 | LR: 0.000001


Epoch 49/50 | Loss: 0.0017 | Best: 0.0013 | LR: 0.000000


Epoch 50/50 | Loss: 0.0021 | Best: 0.0013 | LR: 0.000000

Done. Best loss: 0.0013


In [93]:
model_cs2.load_state_dict(
    torch.load('/tmp/best_chisquare_hn.pt', weights_only=True))
model_cs2.eval()
print("Chi-Square + Hard Negatives loaded.")

qt_10k_tensor = torch.tensor(qt_10k, dtype=torch.float32)
all_embs = []
with torch.no_grad():
    for start in tqdm(range(0, len(qt_10k_tensor), 512), desc="Compressing"):
        batch = qt_10k_tensor[start:start+512].to(device)
        emb   = model_cs2(batch)
        all_embs.append(emb.cpu().numpy())

embs_cs_hn   = np.vstack(all_embs)
corpus_cs_hn = embs_cs_hn[:QUERY_START_10K]
query_cs_hn  = embs_cs_hn[QUERY_START_10K:]

print(f"Sparsity: {(embs_cs_hn==0).mean():.4f}")
active = (embs_cs_hn > 0).sum(axis=1)
print(f"Active dims: mean={active.mean():.1f} std={active.std():.1f}")

index = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in range(len(corpus_cs_hn)):
    index.addDataPoint(i, corpus_cs_hn[i])
index.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
index.setQueryTimeParams({'efSearch': 200})

t0   = time.time()
nbrs = index.knnQueryBatch(query_cs_hn, k=50, num_threads=100)
qps  = len(query_cs_hn) / (time.time() - t0)

r10, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=10)
r50, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=50)

print(f"\n============= 10k FULL COMPARISON =============")
print(f"{'Method':<35} {'Dim':>6} {'R@10':>8} {'R@50':>8} {'QPS':>8}")
print(f"{'Baseline (quadtree)':<35} {'18499':>6} {'0.9965':>8} {'0.9986':>8} {'288':>8}")
print(f"{'MLP compressor':<35} {'512':>6} {'0.6555':>8} {'0.8016':>8} {'40015':>8}")
print(f"{'Neural MinHash':<35} {'512':>6} {'0.6583':>8} {'0.7960':>8} {'28461':>8}")
print(f"{'Chi-Sq v2 (in-batch negs)':<35} {'512':>6} {'0.4088':>8} {'0.5933':>8} {'35253':>8}")
print(f"{'Chi-Sq v2 (hard negs)':<35} {'512':>6} {r10:>8.4f} {r50:>8.4f} {qps:>8.1f}")

Chi-Square + Hard Negatives loaded.


Compressing: 100%|██████████| 20/20 [00:00<00:00, 129.44it/s]


Sparsity: 0.5260
Active dims: mean=242.7 std=94.7

============= 10k FULL COMPARISON =============
Method                                 Dim     R@10     R@50      QPS
Baseline (quadtree)                  18499   0.9965   0.9986      288
MLP compressor                         512   0.6555   0.8016    40015
Neural MinHash                         512   0.6583   0.7960    28461
Chi-Sq v2 (in-batch negs)              512   0.4088   0.5933    35253
Chi-Sq v2 (hard negs)                  512   0.3048   0.4619  35341.1


In [94]:
import time
import nmslib
import numpy as np

# Check num threads actually being used
print(f"NMSLIB version: {nmslib.__version__}")

# Time baseline query at different thread counts
for threads in [1, 8, 32, 100]:
    index_baseline.setQueryTimeParams({'efSearch': 200})
    t0 = time.time()
    nbrs = index_baseline.knnQueryBatch(query_qt_full[:1000], k=50, num_threads=threads)
    qps = 1000 / (time.time() - t0)
    print(f"Baseline  threads={threads:3d}: QPS={qps:.1f}")

for threads in [1, 8, 32, 100]:
    index.setQueryTimeParams({'efSearch': 200})
    t0 = time.time()
    nbrs = index.knnQueryBatch(query_cs_hn[:1000], k=50, num_threads=threads)
    qps = 1000 / (time.time() - t0)
    print(f"Ours      threads={threads:3d}: QPS={qps:.1f}")

NMSLIB version: 2.1.2
Baseline  threads=  1: QPS=25.0
Baseline  threads=  8: QPS=248.9
Baseline  threads= 32: QPS=320.5
Baseline  threads=100: QPS=302.7
Ours      threads=  1: QPS=1784.5
Ours      threads=  8: QPS=13716.1
Ours      threads= 32: QPS=33857.3
Ours      threads=100: QPS=32187.9


In [95]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
import nmslib
import time

# ─── Reload best MLP ──────────────────────────────────────────────────────────
compressor_inspect = QuadtreeCompressorV1(in_dim=IN_DIM_10K, out_dim=512).to(device)
compressor_inspect.load_state_dict(
    torch.load('/tmp/best_compressor_v1_clean.pt', weights_only=True))
compressor_inspect.eval()
print("Best MLP loaded.")

# ─── Generate embeddings ──────────────────────────────────────────────────────
qt_tensor = torch.tensor(qt_10k, dtype=torch.float32)
all_embs  = []
with torch.no_grad():
    for start in tqdm(range(0, len(qt_tensor), 512), desc="Generating"):
        batch = qt_tensor[start:start+512].to(device)
        emb   = compressor_inspect(batch, for_index=True)
        all_embs.append(emb.cpu().numpy())

embs_mlp   = np.vstack(all_embs)
corpus_mlp = embs_mlp[:QUERY_START_10K]
query_mlp  = embs_mlp[QUERY_START_10K:]

# ─── 1. Embedding quality ─────────────────────────────────────────────────────
print("\n=== Embedding quality ===")
print(f"Sparsity: {(embs_mlp==0).mean():.4f}")
active = (embs_mlp > 0).sum(axis=1)
print(f"Active dims: mean={active.mean():.1f} std={active.std():.1f}")

def wj_sim(a, b):
    return float(np.minimum(a,b).sum() / np.maximum(a,b).sum())

gt_sims, rand_sims, hard_sims = [], [], []
for i in range(200):
    qid    = QUERY_START_10K + i
    pos_id = gt_lookup_10k.get(qid, [None])[0]
    if pos_id is None: continue
    neg_id  = hard_neg_pool.get(qid, [0])[0]
    rand_id = np.random.randint(0, QUERY_START_10K)
    gt_sims.append(wj_sim(embs_mlp[qid], embs_mlp[pos_id]))
    hard_sims.append(wj_sim(embs_mlp[qid], embs_mlp[neg_id]))
    rand_sims.append(wj_sim(embs_mlp[qid], embs_mlp[rand_id]))

print(f"GT pair WJ sim   (want HIGH): {np.mean(gt_sims):.4f} ± {np.std(gt_sims):.4f}")
print(f"Hard neg WJ sim  (want LOW):  {np.mean(hard_sims):.4f} ± {np.std(hard_sims):.4f}")
print(f"Random WJ sim    (want LOW):  {np.mean(rand_sims):.4f} ± {np.std(rand_sims):.4f}")
print(f"GT-Hard gap:  {np.mean(gt_sims)-np.mean(hard_sims):.4f}")
print(f"GT-Rand gap:  {np.mean(gt_sims)-np.mean(rand_sims):.4f}")

# ─── 2. Per-query recall analysis ────────────────────────────────────────────
print("\n=== Per-query recall analysis ===")
index = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in range(len(corpus_mlp)):
    index.addDataPoint(i, corpus_mlp[i])
index.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
index.setQueryTimeParams({'efSearch': 200})
nbrs = index.knnQueryBatch(query_mlp, k=50, num_threads=32)

per_query_recall = []
zero_gt = 0
for i, (ids, dists) in enumerate(nbrs):
    qid = QUERY_START_10K + i
    gt  = set(gt_lookup_10k.get(qid, [])[:50])
    if not gt:
        zero_gt += 1
        continue
    r = len(gt & set(ids[:50])) / len(gt)
    per_query_recall.append((qid, r, len(gt)))

recalls = [r for _, r, _ in per_query_recall]
print(f"Queries evaluated: {len(recalls)} | Empty GT: {zero_gt}")
print(f"Mean recall@50:    {np.mean(recalls):.4f}")
print(f"Median recall@50:  {np.median(recalls):.4f}")
print(f"Recall = 1.0:      {sum(1 for r in recalls if r==1.0)} ({100*sum(1 for r in recalls if r==1.0)/len(recalls):.1f}%)")
print(f"Recall = 0.0:      {sum(1 for r in recalls if r==0.0)} ({100*sum(1 for r in recalls if r==0.0)/len(recalls):.1f}%)")
print(f"Recall > 0.5:      {sum(1 for r in recalls if r>0.5)} ({100*sum(1 for r in recalls if r>0.5)/len(recalls):.1f}%)")
print(f"Recall < 0.1:      {sum(1 for r in recalls if r<0.1)} ({100*sum(1 for r in recalls if r<0.1)/len(recalls):.1f}%)")

# ─── 3. Analyze failing queries ───────────────────────────────────────────────
print("\n=== Failing query analysis ===")
failing = [(qid, r, gt_size) for qid, r, gt_size in per_query_recall if r < 0.1]
passing = [(qid, r, gt_size) for qid, r, gt_size in per_query_recall if r >= 0.9]

print(f"Failing queries (recall<0.1): {len(failing)}")
print(f"Passing queries (recall>0.9): {len(passing)}")

if failing:
    fail_gt_sizes = [g for _,_,g in failing]
    pass_gt_sizes = [g for _,_,g in passing]
    print(f"Failing — avg GT size: {np.mean(fail_gt_sizes):.1f}")
    print(f"Passing — avg GT size: {np.mean(pass_gt_sizes):.1f}")

    # Check if failing queries have small/large polygons
    fail_qids = [q for q,_,_ in failing[:20]]
    fail_active = [active[q-QUERY_START_10K+len(corpus_mlp)] 
                   if q >= QUERY_START_10K else active[q] 
                   for q in fail_qids]
    # Actually get active dims for failing queries
    fail_emb_active = [(embs_mlp[q] > 0).sum() for q in fail_qids]
    pass_emb_active = [(embs_mlp[q] > 0).sum() for q,_,_ in passing[:20]]
    print(f"Failing — avg active dims: {np.mean(fail_emb_active):.1f}")
    print(f"Passing — avg active dims: {np.mean(pass_emb_active):.1f}")

# ─── 4. Quadtree vector analysis for failing vs passing ──────────────────────
print("\n=== Quadtree similarity for failing vs passing ===")
for label, queries in [("Failing", failing[:30]), ("Passing", passing[:30])]:
    wj_gt_sims = []
    for qid, _, _ in queries:
        pos_id = gt_lookup_10k.get(qid, [None])[0]
        if pos_id is None: continue
        s = wj_sim(qt_10k[qid], qt_10k[pos_id])
        wj_gt_sims.append(s)
    print(f"{label} — GT WJ sim in quadtree space: {np.mean(wj_gt_sims):.4f} ± {np.std(wj_gt_sims):.4f}")

Best MLP loaded.


Generating: 100%|██████████| 20/20 [00:00<00:00, 100.93it/s]



=== Embedding quality ===
Sparsity: 0.4951
Active dims: mean=258.5 std=5.8
GT pair WJ sim   (want HIGH): 0.9332 ± 0.0850
Hard neg WJ sim  (want LOW):  0.8773 ± 0.1025
Random WJ sim    (want LOW):  0.6366 ± 0.2307
GT-Hard gap:  0.0559
GT-Rand gap:  0.2965

=== Per-query recall analysis ===
Queries evaluated: 1818 | Empty GT: 182
Mean recall@50:    0.8016
Median recall@50:  0.7800
Recall = 1.0:      293 (16.1%)
Recall = 0.0:      0 (0.0%)
Recall > 0.5:      1812 (99.7%)
Recall < 0.1:      0 (0.0%)

=== Failing query analysis ===
Failing queries (recall<0.1): 0
Passing queries (recall>0.9): 379

=== Quadtree similarity for failing vs passing ===
Failing — GT WJ sim in quadtree space: nan ± nan
Passing — GT WJ sim in quadtree space: 0.7059 ± 0.0833


/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type

In [96]:
# Current MLP architecture inspection
print("=== Current MLP architecture ===")
for name, param in compressor_inspect.named_parameters():
    print(f"  {name}: {tuple(param.shape)} — {param.numel():,} params")

total = sum(p.numel() for p in compressor_inspect.parameters())
print(f"\nTotal parameters: {total:,}")
print(f"Architecture: {IN_DIM_10K} → 4096 → 1024 → 512")

# Check what the GT-Hard gap looks like in the ORIGINAL quadtree space
print("\n=== GT-Hard gap in ORIGINAL quadtree space ===")
qt_gt_sims, qt_hard_sims = [], []
for i in range(200):
    qid    = QUERY_START_10K + i
    pos_id = gt_lookup_10k.get(qid, [None])[0]
    neg_id = hard_neg_pool.get(qid, [0])[0]
    if pos_id is None: continue
    qt_gt_sims.append(wj_sim(qt_10k[qid], qt_10k[pos_id]))
    qt_hard_sims.append(wj_sim(qt_10k[qid], qt_10k[neg_id]))

print(f"Quadtree GT-Hard gap:  {np.mean(qt_gt_sims)-np.mean(qt_hard_sims):.4f}")
print(f"Embedding GT-Hard gap: {np.mean(gt_sims)-np.mean(hard_sims):.4f}")
print(f"Gap ratio: {(np.mean(gt_sims)-np.mean(hard_sims))/(np.mean(qt_gt_sims)-np.mean(qt_hard_sims)):.4f}")
print(f"--> How much of the quadtree gap does the MLP preserve?")

# Distribution of per-query recalls
import numpy as np
recall_bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
hist, _ = np.histogram(recalls, bins=recall_bins)
print("\n=== Recall distribution ===")
for i, count in enumerate(hist):
    bar = "█" * (count // 5)
    print(f"  [{recall_bins[i]:.1f}-{recall_bins[i+1]:.1f}]: {count:4d} queries {bar}")

=== Current MLP architecture ===
  net.0.weight: (4096, 18499) — 75,771,904 params
  net.1.weight: (4096,) — 4,096 params
  net.1.bias: (4096,) — 4,096 params
  net.3.weight: (1024, 4096) — 4,194,304 params
  net.4.weight: (1024,) — 1,024 params
  net.4.bias: (1024,) — 1,024 params
  net.6.weight: (512, 1024) — 524,288 params
  net.7.weight: (512,) — 512 params
  net.7.bias: (512,) — 512 params

Total parameters: 80,501,760
Architecture: 18499 → 4096 → 1024 → 512

=== GT-Hard gap in ORIGINAL quadtree space ===
Quadtree GT-Hard gap:  0.2027
Embedding GT-Hard gap: 0.0559
Gap ratio: 0.2756
--> How much of the quadtree gap does the MLP preserve?

=== Recall distribution ===
  [0.0-0.1]:    0 queries 
  [0.1-0.2]:    0 queries 
  [0.2-0.3]:    0 queries 
  [0.3-0.4]:    0 queries 
  [0.4-0.5]:    3 queries 
  [0.5-0.6]:   28 queries █████
  [0.6-0.7]:  225 queries █████████████████████████████████████████████
  [0.7-0.8]:  704 queries ████████████████████████████████████████████████████████

In [97]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import random

# ─── Phase 1 dataset: random negatives (in-batch) ────────────────────────────
class AnchorPositiveDataset(Dataset):
    def __init__(self, qtree_vectors, gt_lookup, query_start, max_pos=None):
        self.vecs  = torch.tensor(qtree_vectors, dtype=torch.float32)
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            pool = neighbors if max_pos is None else neighbors[:max_pos]
            for nid in pool:
                if nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"Total pairs: {len(self.pairs)}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        qid, pos_id = self.pairs[idx]
        return self.vecs[qid], self.vecs[pos_id], qid, pos_id

# ─── Phase 2 dataset: hard negatives ─────────────────────────────────────────
class HardNegDataset(Dataset):
    def __init__(self, qtree_vectors, gt_lookup, hard_neg_pool):
        self.qt = torch.tensor(qtree_vectors, dtype=torch.float32)
        self.triplets = []
        for qid, pos_list in gt_lookup.items():
            negs = hard_neg_pool.get(qid, [])
            if not negs: continue
            for pos_id in pos_list:
                neg_id = random.choice(negs)
                self.triplets.append((qid, pos_id, neg_id))
        random.shuffle(self.triplets)
        print(f"Total triplets: {len(self.triplets)}")
    def __len__(self): return len(self.triplets)
    def __getitem__(self, idx):
        qid, pos_id, neg_id = self.triplets[idx]
        return self.qt[qid], self.qt[pos_id], self.qt[neg_id]

def vectorized_hard_triplet_loss(anchors, positives, margin=0.3):
    d_pos      = (anchors - positives).pow(2).sum(dim=1).sqrt()
    sim_cross  = torch.mm(anchors, positives.T)
    sim_cross.fill_diagonal_(-1e9)
    d_neg_hard = (2 - 2 * sim_cross.max(dim=1).values).clamp(min=0).sqrt()
    loss       = F.relu(d_pos - d_neg_hard + margin)
    violated   = loss > 0
    if violated.sum() == 0:
        return torch.tensor(0.0, device=anchors.device, requires_grad=True), 0
    return loss[violated].mean(), violated.sum().item()

def explicit_triplet_loss(a, p, n, margin=0.3):
    d_pos = F.pairwise_distance(a, p)
    d_neg = F.pairwise_distance(a, n)
    loss  = F.relu(d_pos - d_neg + margin)
    mask  = loss > 0
    if mask.sum() == 0:
        return torch.tensor(0.0, device=a.device, requires_grad=True), 0
    return loss[mask].mean(), mask.sum().item()

# ─── Fresh larger MLP ─────────────────────────────────────────────────────────
class QuadtreeCompressorV2(nn.Module):
    """Larger MLP with extra hidden layer for more capacity"""
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False),
            nn.BatchNorm1d(4096),
            nn.ReLU(),
            nn.Linear(4096, 2048, bias=False),
            nn.BatchNorm1d(2048),
            nn.ReLU(),
            nn.Linear(2048, 1024, bias=False),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
        )
    def forward(self, x, for_index=False):
        x = self.net(x)
        if for_index:
            x = F.relu(x)
            x = x / x.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return x

device = torch.device('cuda:0')
model_v2 = QuadtreeCompressorV2(in_dim=IN_DIM_10K, out_dim=512).to(device)
total_params = sum(p.numel() for p in model_v2.parameters())
print(f"Model v2 parameters: {total_params:,}")

if torch.cuda.device_count() > 1:
    model_v2_par = nn.DataParallel(model_v2)
    print(f"Training on {torch.cuda.device_count()} GPUs")
else:
    model_v2_par = model_v2

# ─── Phase 1: Random negatives (30 epochs) ───────────────────────────────────
print("\n=== PHASE 1: Random negatives (30 epochs) ===")
dataset_p1 = AnchorPositiveDataset(qt_10k, gt_lookup_10k,
                                    query_start=QUERY_START_10K, max_pos=30)
loader_p1  = DataLoader(dataset_p1, batch_size=1024, shuffle=True,
                        num_workers=0, pin_memory=False, drop_last=True)
print(f"Steps per epoch: {len(loader_p1)}")

optimizer = torch.optim.AdamW(model_v2.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

best_loss = float('inf')
for epoch in range(30):
    model_v2_par.train()
    total_loss = 0.0; total_steps = 0

    pbar = tqdm(loader_p1, desc=f"P1 Epoch {epoch+1:2d}/30", leave=False)
    for anchors, positives, _, _ in pbar:
        anchors   = anchors.to(device)
        positives = positives.to(device)
        B         = anchors.shape[0]

        combined   = torch.cat([anchors, positives], dim=0)
        out        = model_v2_par(combined)
        out_norm   = F.normalize(out, dim=1)
        a_emb, p_emb = out_norm[:B], out_norm[B:]

        loss, _ = vectorized_hard_triplet_loss(a_emb, p_emb, margin=0.3)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_v2.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item(); total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / total_steps
    scheduler.step()

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_v2.state_dict(), '/tmp/best_mlp_v2_p1.pt')

    print(f"P1 Epoch {epoch+1:2d}/30 | Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nPhase 1 done. Best loss: {best_loss:.4f}")
print("Loading best phase 1 weights for phase 2...")
model_v2.load_state_dict(torch.load('/tmp/best_mlp_v2_p1.pt', weights_only=True))

# ─── Phase 2: Hard negatives fine-tuning (20 epochs, lower LR) ───────────────
print("\n=== PHASE 2: Hard negative fine-tuning (20 epochs) ===")
dataset_p2 = HardNegDataset(qt_10k, gt_lookup_10k, hard_neg_pool)
loader_p2  = DataLoader(dataset_p2, batch_size=1024, shuffle=True,
                        num_workers=0, pin_memory=False, drop_last=True)
print(f"Steps per epoch: {len(loader_p2)}")

# Lower LR for fine-tuning — don't destroy phase 1 structure
optimizer2 = torch.optim.AdamW(model_v2.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=20)

best_loss2 = float('inf')
for epoch in range(20):
    model_v2_par.train()
    total_loss = 0.0; total_steps = 0

    pbar = tqdm(loader_p2, desc=f"P2 Epoch {epoch+1:2d}/20", leave=False)
    for anchors, positives, negatives in pbar:
        anchors   = anchors.to(device)
        positives = positives.to(device)
        negatives = negatives.to(device)

        a_emb = F.normalize(model_v2_par(anchors),   dim=1)
        p_emb = F.normalize(model_v2_par(positives), dim=1)
        n_emb = F.normalize(model_v2_par(negatives), dim=1)

        loss, _ = explicit_triplet_loss(a_emb, p_emb, n_emb, margin=0.1)
        optimizer2.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_v2.parameters(), 1.0)
        optimizer2.step()

        total_loss += loss.item(); total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / total_steps
    scheduler2.step()

    if avg_loss < best_loss2:
        best_loss2 = avg_loss
        torch.save(model_v2.state_dict(), '/tmp/best_mlp_v2_final.pt')

    print(f"P2 Epoch {epoch+1:2d}/20 | Loss: {avg_loss:.4f} | Best: {best_loss2:.4f} | LR: {scheduler2.get_last_lr()[0]:.6f}")

print(f"\nPhase 2 done. Best loss: {best_loss2:.4f}")

Model v2 parameters: 86,797,312
Training on 8 GPUs

=== PHASE 1: Random negatives (30 epochs) ===
Total pairs: 46722
Steps per epoch: 45


P1 Epoch  1/30 | Loss: 0.3225 | Best: 0.3225 | LR: 0.000997


P1 Epoch  2/30 | Loss: 0.3171 | Best: 0.3171 | LR: 0.000989


P1 Epoch  3/30 | Loss: 0.3128 | Best: 0.3128 | LR: 0.000976


P1 Epoch  4/30 | Loss: 0.3096 | Best: 0.3096 | LR: 0.000957


...

P2 Epoch 16/20 | Loss: 0.0575 | Best: 0.0572 | LR: 0.000010


P2 Epoch 17/20 | Loss: 0.0568 | Best: 0.0568 | LR: 0.000005


P2 Epoch 18/20 | Loss: 0.0564 | Best: 0.0564 | LR: 0.000002


P2 Epoch 19/20 | Loss: 0.0569 | Best: 0.0564 | LR: 0.000001


P2 Epoch 20/20 | Loss: 0.0554 | Best: 0.0554 | LR: 0.000000

Phase 2 done. Best loss: 0.0554


In [98]:
model_v2.load_state_dict(
    torch.load('/tmp/best_mlp_v2_final.pt', weights_only=True))
model_v2.eval()
print("Curriculum MLP v2 loaded.")

# Generate embeddings
qt_tensor = torch.tensor(qt_10k, dtype=torch.float32)
all_embs  = []
with torch.no_grad():
    for start in tqdm(range(0, len(qt_tensor), 512), desc="Compressing"):
        batch = qt_tensor[start:start+512].to(device)
        emb   = model_v2(batch, for_index=True)
        all_embs.append(emb.cpu().numpy())

embs_v2   = np.vstack(all_embs)
corpus_v2 = embs_v2[:QUERY_START_10K]
query_v2  = embs_v2[QUERY_START_10K:]

# GT-Hard gap check
gt_sims_v2, hard_sims_v2 = [], []
for i in range(200):
    qid    = QUERY_START_10K + i
    pos_id = gt_lookup_10k.get(qid, [None])[0]
    neg_id = hard_neg_pool.get(qid, [0])[0]
    if pos_id is None: continue
    gt_sims_v2.append(wj_sim(embs_v2[qid], embs_v2[pos_id]))
    hard_sims_v2.append(wj_sim(embs_v2[qid], embs_v2[neg_id]))

print(f"\nGT-Hard gap v1 (original): 0.0559")
print(f"GT-Hard gap v2 (curriculum): {np.mean(gt_sims_v2)-np.mean(hard_sims_v2):.4f}")

# HNSW + eval
index = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in range(len(corpus_v2)):
    index.addDataPoint(i, corpus_v2[i])
index.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
index.setQueryTimeParams({'efSearch': 200})

t0   = time.time()
nbrs = index.knnQueryBatch(query_v2, k=50, num_threads=32)
qps  = len(query_v2) / (time.time() - t0)

r10, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=10)
r50, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, query_start_id=QUERY_START_10K, K=50)

# Recall distribution
recalls_v2 = []
for i, (ids, _) in enumerate(nbrs):
    qid = QUERY_START_10K + i
    gt  = set(gt_lookup_10k.get(qid, [])[:50])
    if not gt: continue
    recalls_v2.append(len(gt & set(ids[:50])) / len(gt))

print(f"\n============= 10k COMPARISON =============")
print(f"{'Method':<30} {'R@10':>8} {'R@50':>8} {'QPS':>8}")
print(f"{'Baseline (quadtree)':<30} {'0.9965':>8} {'0.9986':>8} {'320':>8}")
print(f"{'MLP v1 (original)':<30} {'0.6555':>8} {'0.8016':>8} {'33857':>8}")
print(f"{'MLP v2 (curriculum)':<30} {r10:>8.4f} {r50:>8.4f} {qps:>8.1f}")

print(f"\nRecall distribution v2:")
recall_bins = [0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
hist, _ = np.histogram(recalls_v2, bins=recall_bins)
for i, count in enumerate(hist):
    bar = "█" * (count // 5)
    print(f"  [{recall_bins[i]:.1f}-{recall_bins[i+1]:.1f}]: {count:4d} {bar}")

Curriculum MLP v2 loaded.


Compressing: 100%|██████████| 20/20 [00:00<00:00, 100.33it/s]



GT-Hard gap v1 (original): 0.0559
GT-Hard gap v2 (curriculum): 0.1726

============= 10k COMPARISON =============
Method                             R@10     R@50      QPS
Baseline (quadtree)              0.9965   0.9986      320
MLP v1 (original)                0.6555   0.8016    33857
MLP v2 (curriculum)              0.6294   0.7768  25390.5

Recall distribution v2:
  [0.0-0.5]:   16 ███
  [0.5-0.6]:   78 ███████████████
  [0.6-0.7]:  270 ██████████████████████████████████████████████████████
  [0.7-0.8]:  747 █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  [0.8-0.9]:  400 ████████████████████████████████████████████████████████████████████████████████
  [0.9-1.0]:  307 █████████████████████████████████████████████████████████████


In [99]:
import nmslib
import numpy as np
import time
from tqdm import tqdm

def eval_at_k(gt_lookup, nbrs, query_start_id, k_list):
    results = {}
    for K in k_list:
        total_recall = 0.0
        count = 0
        for i, (ids, _) in enumerate(nbrs):
            qid = query_start_id + i
            gt  = set(gt_lookup.get(qid, [])[:K])
            if not gt: continue
            total_recall += len(gt & set(ids[:K])) / len(gt)
            count += 1
        results[K] = total_recall / count if count > 0 else 0.0
    return results

# ─── Re-query all models at K=500 ─────────────────────────────────────────────
K_LIST = [10, 50, 100, 500]

print("Building indexes and querying at K=500...\n")

# MLP v1
idx_mlp = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in range(len(corpus_mlp)):
    idx_mlp.addDataPoint(i, corpus_mlp[i])
idx_mlp.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
idx_mlp.setQueryTimeParams({'efSearch': 200})
t0 = time.time()
nbrs_mlp = idx_mlp.knnQueryBatch(query_mlp, k=500, num_threads=32)
qps_mlp  = len(query_mlp) / (time.time() - t0)
res_mlp  = eval_at_k(gt_lookup_10k, nbrs_mlp, QUERY_START_10K, K_LIST)

# Neural MinHash
embs_mh   = np.vstack([model_mh(torch.tensor(qt_10k[s:s+512], dtype=torch.float32).to(device)).cpu().detach().numpy()
                        for s in range(0, len(qt_10k), 512)])
corpus_mh = embs_mh[:QUERY_START_10K]
query_mh  = embs_mh[QUERY_START_10K:]
idx_mh = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in range(len(corpus_mh)):
    idx_mh.addDataPoint(i, corpus_mh[i])
idx_mh.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
idx_mh.setQueryTimeParams({'efSearch': 200})
t0 = time.time()
nbrs_mh = idx_mh.knnQueryBatch(query_mh, k=500, num_threads=32)
qps_mh  = len(query_mh) / (time.time() - t0)
res_mh  = eval_at_k(gt_lookup_10k, nbrs_mh, QUERY_START_10K, K_LIST)

# Baseline
idx_bl = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in range(len(qt_10k[:QUERY_START_10K])):
    idx_bl.addDataPoint(i, qt_10k[i])
idx_bl.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
idx_bl.setQueryTimeParams({'efSearch': 200})
t0 = time.time()
nbrs_bl = idx_bl.knnQueryBatch(qt_10k[QUERY_START_10K:], k=500, num_threads=32)
qps_bl  = len(qt_10k[QUERY_START_10K:]) / (time.time() - t0)
res_bl  = eval_at_k(gt_lookup_10k, nbrs_bl, QUERY_START_10K, K_LIST)

# ─── Print full table ──────────────────────────────────────────────────────────
print(f"\n{'Method':<28} {'Dim':>6} {'R@10':>8} {'R@50':>8} {'R@100':>8} {'R@500':>8} {'QPS':>8}")
print("-" * 80)
print(f"{'Baseline (quadtree)':<28} {'18499':>6} {res_bl[10]:>8.4f} {res_bl[50]:>8.4f} {res_bl[100]:>8.4f} {res_bl[500]:>8.4f} {qps_bl:>8.1f}")
print(f"{'MLP compressor':<28} {'512':>6} {res_mlp[10]:>8.4f} {res_mlp[50]:>8.4f} {res_mlp[100]:>8.4f} {res_mlp[500]:>8.4f} {qps_mlp:>8.1f}")
print(f"{'Neural MinHash':<28} {'512':>6} {res_mh[10]:>8.4f} {res_mh[50]:>8.4f} {res_mh[100]:>8.4f} {res_mh[500]:>8.4f} {qps_mh:>8.1f}")

Building indexes and querying at K=500...


Method                          Dim     R@10     R@50    R@100    R@500      QPS
--------------------------------------------------------------------------------
Baseline (quadtree)           18499   0.9965   0.9986   0.9990   0.9974    292.7
MLP compressor                  512   0.6555   0.8016   0.8414   0.9504  15528.8
Neural MinHash                  512   0.6584   0.7964   0.8462   0.9581  17845.3


In [100]:
import numpy as np
import nmslib
import time
from tqdm import tqdm

# Use raw L2-normalized MLP embeddings (before ReLU/L1 norm)
# Reload model and generate L2-normalized embeddings
compressor_l2 = QuadtreeCompressorV1(in_dim=IN_DIM_10K, out_dim=512).to(device)
compressor_l2.load_state_dict(
    torch.load('/tmp/best_compressor_v1_clean.pt', weights_only=True))
compressor_l2.eval()

import torch
import torch.nn.functional as F

# Generate L2-normalized embeddings (cosine space — no ReLU/L1)
all_embs_l2 = []
qt_tensor = torch.tensor(qt_10k, dtype=torch.float32)
with torch.no_grad():
    for start in tqdm(range(0, len(qt_tensor), 512), desc="L2 embeddings"):
        batch = qt_tensor[start:start+512].to(device)
        # Get raw output (no for_index) then L2-normalize
        raw = compressor_l2.net(batch)
        emb = F.normalize(raw, dim=1)  # L2 normalize → cosine space
        all_embs_l2.append(emb.cpu().numpy())

embs_l2   = np.vstack(all_embs_l2)
corpus_l2 = embs_l2[:QUERY_START_10K]
query_l2  = embs_l2[QUERY_START_10K:]

print(f"L2 embeddings: {embs_l2.shape}")
print(f"Has negatives: {(embs_l2 < 0).any()}")

# Build cosine HNSW index
idx_cos = nmslib.init(method='hnsw', space='cosinesimil')
for i in range(len(corpus_l2)):
    idx_cos.addDataPoint(i, corpus_l2[i])
idx_cos.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
idx_cos.setQueryTimeParams({'efSearch': 200})

t0 = time.time()
nbrs_cos = idx_cos.knnQueryBatch(query_l2, k=50, num_threads=32)
qps_cos  = len(query_l2) / (time.time() - t0)

r10_cos, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs_cos, query_start_id=QUERY_START_10K, K=10)
r50_cos, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs_cos, query_start_id=QUERY_START_10K, K=50)

print(f"\n============= COSINE vs WJ INDEX COMPARISON =============")
print(f"{'Method':<35} {'Space':>8} {'R@10':>8} {'R@50':>8} {'QPS':>8}")
print(f"{'Baseline (quadtree)':<35} {'WJ':>8} {'0.9965':>8} {'0.9986':>8} {'293':>8}")
print(f"{'MLP (WJ index)':<35} {'WJ':>8} {'0.6555':>8} {'0.8016':>8} {'15529':>8}")
print(f"{'MLP (cosine index)':<35} {'Cosine':>8} {r10_cos:>8.4f} {r50_cos:>8.4f} {qps_cos:>8.1f}")

L2 embeddings: 100%|██████████| 20/20 [00:00<00:00, 107.17it/s]


L2 embeddings: (10000, 512)
Has negatives: True

============= COSINE vs WJ INDEX COMPARISON =============
Method                                 Space     R@10     R@50      QPS
Baseline (quadtree)                       WJ   0.9965   0.9986      293
MLP (WJ index)                            WJ   0.6555   0.8016    15529
MLP (cosine index)                    Cosine   0.6656   0.8081  49241.1


In [101]:
import os

# Check all saved model weights
weight_files = [
    '/tmp/best_compressor_v1_clean.pt',    # MLP v1 (best)
    '/tmp/best_sota_transformer.pt',         # Transformer L2 space
    '/tmp/best_sota_wj.pt',                  # Transformer WJ space
    '/tmp/best_minhash.pt',                  # Neural MinHash
    '/tmp/best_chisquare2.pt',               # Chi-Sq v2 (log1p, in-batch)
    '/tmp/best_chisquare_hn.pt',             # Chi-Sq v2 (hard negs)
]

print("=== Saved model weights ===")
for f in weight_files:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) / 1024**2 if exists else 0
    print(f"  {'OK' if exists else 'MISSING':>7} | {size:6.1f} MB | {f}")

# Check what classes/objects are in memory
print("\n=== Objects in memory ===")
for name in ['model_mh', 'model_cs2', 'model_sota_10k', 'model_wj', 'compressor_inspect']:
    try:
        obj = eval(name)
        print(f"  {name}: {obj.__class__.__name__} — IN MEMORY")
    except:
        print(f"  {name}: NOT IN MEMORY")

=== Saved model weights ===
       OK |  307.1 MB | /tmp/best_compressor_v1_clean.pt
       OK |    8.9 MB | /tmp/best_sota_transformer.pt
       OK |    8.9 MB | /tmp/best_sota_wj.pt
       OK |   78.3 MB | /tmp/best_minhash.pt
       OK |  164.6 MB | /tmp/best_chisquare2.pt
       OK |  164.6 MB | /tmp/best_chisquare_hn.pt

=== Objects in memory ===
  model_mh: NeuralMinHashEncoder — IN MEMORY
  model_cs2: ChiSquareTwoTowerEncoder — IN MEMORY
  model_sota_10k: QuadtreeTransformerEncoder — IN MEMORY
  model_wj: QuadtreeTransformerEncoderWJ — IN MEMORY
  compressor_inspect: QuadtreeCompressorV1 — IN MEMORY


In [102]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nmslib
import time
from tqdm import tqdm
import math

# ─── Helper: generate embeddings in cosine space ──────────────────────────────
def get_cosine_embeddings(model, qt_data, device, batch_size=512, model_type='mlp'):
    """Generate L2-normalized embeddings (cosine space) from any model."""
    model.eval()
    all_embs = []
    qt_tensor = torch.tensor(qt_data, dtype=torch.float32)
    with torch.no_grad():
        for start in range(0, len(qt_tensor), batch_size):
            batch = qt_tensor[start:start+batch_size].to(device)
            if model_type == 'mlp':
                # Raw output before ReLU/L1 norm
                raw = model.net(batch)
            elif model_type == 'transformer_l2':
                raw = model(batch)  # already outputs raw
            elif model_type == 'transformer_wj':
                # Get pre-normalized output
                B = batch.shape[0]
                x = batch
                if x.shape[1] < model.padded_dim:
                    pad = torch.zeros(B, model.padded_dim - x.shape[1], device=device)
                    x = torch.cat([x, pad], dim=1)
                x = x.view(B, model.n_chunks, model.chunk_size)
                x = model.chunk_proj(x) + model.pe
                x = model.transformer(x)
                x = x.mean(dim=1)
                raw = model.out_proj(x)
            elif model_type == 'minhash':
                x = torch.log1p(batch * 1e6)
                raw = model.extractor(x)
                raw = model.hash_proj(raw)
            elif model_type == 'chisquare':
                x = torch.log1p(batch * 1e6)
                raw = model.tower(x)
            # L2 normalize for cosine space
            emb = F.normalize(raw, dim=1)
            all_embs.append(emb.cpu().numpy())
    return np.vstack(all_embs)

def get_wj_embeddings(model, qt_data, device, batch_size=512, model_type='mlp'):
    """Generate ReLU+L1 embeddings (WJ space) from any model."""
    model.eval()
    all_embs = []
    qt_tensor = torch.tensor(qt_data, dtype=torch.float32)
    with torch.no_grad():
        for start in range(0, len(qt_tensor), batch_size):
            batch = qt_tensor[start:start+batch_size].to(device)
            if model_type == 'mlp':
                emb = model(batch, for_index=True)
            elif model_type == 'transformer_wj':
                emb = model(batch)  # already WJ normalized
            elif model_type == 'minhash':
                emb = model(batch)  # already WJ normalized
            elif model_type == 'chisquare':
                emb = model(batch)  # already WJ normalized
            else:
                # fallback: ReLU + L1
                raw = model(batch)
                raw = F.relu(raw)
                emb = raw / raw.sum(dim=1, keepdim=True).clamp(min=1e-10)
            all_embs.append(emb.cpu().numpy())
    return np.vstack(all_embs)

def build_and_query(corpus_embs, query_embs, space='cosinesimil', k=50, threads=32):
    idx = nmslib.init(method='hnsw', space=space)
    for i in range(len(corpus_embs)):
        idx.addDataPoint(i, corpus_embs[i])
    idx.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
    idx.setQueryTimeParams({'efSearch': 200})
    t0   = time.time()
    nbrs = idx.knnQueryBatch(query_embs, k=k, num_threads=threads)
    qps  = len(query_embs) / (time.time() - t0)
    return nbrs, qps

# ─── Load all models ──────────────────────────────────────────────────────────
device = torch.device('cuda:0')

# Reload MLP
mlp = QuadtreeCompressorV1(in_dim=IN_DIM_10K, out_dim=512).to(device)
mlp.load_state_dict(torch.load('/tmp/best_compressor_v1_clean.pt', weights_only=True))

# Transformer L2
transformer_l2 = QuadtreeTransformerEncoder(
    in_dim=IN_DIM_10K, n_chunks=64, d_model=256,
    nhead=8, num_layers=4, dim_ff=512, out_dim=512).to(device)
transformer_l2.load_state_dict(torch.load('/tmp/best_sota_transformer.pt', weights_only=True))

# Transformer WJ
transformer_wj = QuadtreeTransformerEncoderWJ(
    in_dim=IN_DIM_10K, n_chunks=64, d_model=256,
    nhead=8, num_layers=4, dim_ff=512, out_dim=512).to(device)
transformer_wj.load_state_dict(torch.load('/tmp/best_sota_wj.pt', weights_only=True))

# Neural MinHash
minhash = NeuralMinHashEncoder(in_dim=IN_DIM_10K, hidden_dim=1024, n_hashes=512).to(device)
minhash.load_state_dict(torch.load('/tmp/best_minhash.pt', weights_only=True))

# Chi-Square
chisq = ChiSquareTwoTowerEncoder(in_dim=IN_DIM_10K, hidden_dim=2048, out_dim=512).to(device)
chisq.load_state_dict(torch.load('/tmp/best_chisquare2.pt', weights_only=True))

print("All models loaded.")

# ─── Generate all embeddings ──────────────────────────────────────────────────
print("\nGenerating embeddings...")
models_config = [
    ('MLP',              mlp,            'mlp'),
    ('Transformer-L2',   transformer_l2, 'transformer_l2'),
    ('Transformer-WJ',   transformer_wj, 'transformer_wj'),
    ('Neural MinHash',   minhash,        'minhash'),
    ('Chi-Sq Two-Tower', chisq,          'chisquare'),
]

results = {}
for name, model, mtype in models_config:
    print(f"  {name}...")
    cos_embs = get_cosine_embeddings(model, qt_10k, device, model_type=mtype)
    wj_embs  = get_wj_embeddings(model, qt_10k, device, model_type=mtype) if mtype != 'transformer_l2' else None

    corpus_cos = cos_embs[:QUERY_START_10K]
    query_cos  = cos_embs[QUERY_START_10K:]

    # Cosine index
    nbrs_cos, qps_cos = build_and_query(corpus_cos, query_cos, space='cosinesimil')
    r10_cos, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs_cos, QUERY_START_10K, K=10)
    r50_cos, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs_cos, QUERY_START_10K, K=50)

    results[name] = {'cos': (r10_cos, r50_cos, qps_cos)}

    # WJ index (if applicable)
    if wj_embs is not None:
        corpus_wj = wj_embs[:QUERY_START_10K]
        query_wj  = wj_embs[QUERY_START_10K:]
        nbrs_wj, qps_wj = build_and_query(corpus_wj, query_wj, space='WeightedJaccard')
        r10_wj, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs_wj, QUERY_START_10K, K=10)
        r50_wj, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs_wj, QUERY_START_10K, K=50)
        results[name]['wj'] = (r10_wj, r50_wj, qps_wj)
    else:
        results[name]['wj'] = None

# ─── Print full comparison table ──────────────────────────────────────────────
print(f"\n{'Method':<22} {'Index':>8} {'R@10':>8} {'R@50':>8} {'QPS':>10}")
print("-" * 62)
print(f"{'Baseline (quadtree)':<22} {'WJ':>8} {'0.9965':>8} {'0.9986':>8} {'293':>10}")
print()
for name, res in results.items():
    r10c, r50c, qpsc = res['cos']
    print(f"{name:<22} {'Cosine':>8} {r10c:>8.4f} {r50c:>8.4f} {qpsc:>10.1f}")
    if res['wj']:
        r10w, r50w, qpsw = res['wj']
        print(f"{name:<22} {'WJ':>8} {r10w:>8.4f} {r50w:>8.4f} {qpsw:>10.1f}")
    print()

All models loaded.

Generating embeddings...
  MLP...
  Transformer-L2...
  Transformer-WJ...
  Neural MinHash...
  Chi-Sq Two-Tower...

Method                    Index     R@10     R@50        QPS
--------------------------------------------------------------
Baseline (quadtree)          WJ   0.9965   0.9986        293

MLP                      Cosine   0.6656   0.8081    64284.0
MLP                          WJ   0.6555   0.8016    36486.3

Transformer-L2           Cosine   0.0013   0.0060    38045.0

Transformer-WJ           Cosine   0.0012   0.0078    38007.9
Transformer-WJ               WJ   0.0068   0.0208    16877.7

Neural MinHash           Cosine   0.3901   0.5375    51396.4
Neural MinHash               WJ   0.6585   0.7962    34743.6

Chi-Sq Two-Tower         Cosine   0.1197   0.2233    43156.4
Chi-Sq Two-Tower             WJ   0.2908   0.4299    40218.1



In [105]:
import numpy as np
import faiss
import time

def binarize_embeddings(embs, threshold='mean'):
    if threshold == 'mean':
        thresh = embs.mean(axis=1, keepdims=True)
    elif threshold == 'zero':
        thresh = 0.0
    elif threshold == 'median':
        thresh = np.median(embs, axis=1, keepdims=True)
    return (embs > thresh).astype(np.uint8)

def pack_to_faiss(binary_embs):
    """Pack binary uint8 arrays to packed uint8 for FAISS IndexBinaryFlat."""
    n, d = binary_embs.shape
    # FAISS requires d to be multiple of 8
    pad = (8 - d % 8) % 8
    if pad > 0:
        binary_embs = np.pad(binary_embs, ((0,0),(0,pad)), 'constant')
    # Pack 8 bits per byte
    packed = np.packbits(binary_embs, axis=1)
    return packed, binary_embs.shape[1]

corpus_cos = mlp_cos_embs[:QUERY_START_10K]
query_cos  = mlp_cos_embs[QUERY_START_10K:]

print("=== FAISS Hamming Space Experiment ===")
print(f"Embedding dim: {mlp_cos_embs.shape[1]}")

results_hamming = {}
for threshold in ['mean', 'zero', 'median']:
    corpus_bin = binarize_embeddings(corpus_cos, threshold)
    query_bin  = binarize_embeddings(query_cos,  threshold)

    active   = corpus_bin.sum(axis=1).mean()
    sparsity = 1 - corpus_bin.mean()
    print(f"\nThreshold={threshold}: active bits={active:.1f}/512, sparsity={sparsity:.3f}")

    corpus_packed, d_bits = pack_to_faiss(corpus_bin)
    query_packed,  _      = pack_to_faiss(query_bin)

    # Ensure contiguous uint8
    corpus_packed = np.ascontiguousarray(corpus_packed, dtype=np.uint8)
    query_packed  = np.ascontiguousarray(query_packed,  dtype=np.uint8)

    # FAISS binary index — exact Hamming search
    index_hamming = faiss.IndexBinaryFlat(d_bits)
    index_hamming.add(corpus_packed)
    print(f"  Index built: {index_hamming.ntotal} vectors, {d_bits} bits")

    t0 = time.time()
    D, I = index_hamming.search(query_packed, 50)
    qps  = len(query_packed) / (time.time() - t0)

    # Convert FAISS results to NMSLIB-style format for compute_recall_at_k
    nbrs_ham = [(I[i], D[i]) for i in range(len(I))]

    r10, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs_ham, QUERY_START_10K, K=10)
    r50, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs_ham, QUERY_START_10K, K=50)
    results_hamming[threshold] = (r10, r50, qps)
    print(f"  R@10={r10:.4f} R@50={r50:.4f} QPS={qps:.1f}")

print(f"\n============= FULL COMPARISON =============")
print(f"{'Method':<32} {'Index':>8} {'R@10':>8} {'R@50':>8} {'QPS':>10}")
print(f"{'Baseline (quadtree)':<32} {'WJ':>8} {'0.9965':>8} {'0.9986':>8} {'293':>10}")
print(f"{'MLP':<32} {'Cosine':>8} {'0.6656':>8} {'0.8081':>8} {'64284':>10}")
print(f"{'MLP':<32} {'WJ':>8} {'0.6555':>8} {'0.8016':>8} {'36486':>10}")
print(f"{'Neural MinHash':<32} {'WJ':>8} {'0.6585':>8} {'0.7962':>8} {'34744':>10}")
for thresh, (r10, r50, qps) in results_hamming.items():
    print(f"{'MLP Hamming ('+thresh+')':<32} {'Hamming':>8} {r10:>8.4f} {r50:>8.4f} {qps:>10.1f}")

=== FAISS Hamming Space Experiment ===
Embedding dim: 512

Threshold=mean: active bits=254.3/512, sparsity=0.503
  Index built: 8000 vectors, 512 bits
  R@10=0.4005 R@50=0.5824 QPS=1683.5

Threshold=zero: active bits=258.5/512, sparsity=0.495
  Index built: 8000 vectors, 512 bits
  R@10=0.4055 R@50=0.5861 QPS=65724.4

Threshold=median: active bits=256.0/512, sparsity=0.500
  Index built: 8000 vectors, 512 bits
  R@10=0.3722 R@50=0.5601 QPS=62874.2

============= FULL COMPARISON =============
Method                              Index     R@10     R@50        QPS
Baseline (quadtree)                    WJ   0.9965   0.9986        293
MLP                                Cosine   0.6656   0.8081      64284
MLP                                    WJ   0.6555   0.8016      36486
Neural MinHash                         WJ   0.6585   0.7962      34744
MLP Hamming (mean)                Hamming   0.4005   0.5824     1683.5
MLP Hamming (zero)                Hamming   0.4055   0.5861    65724.4
MLP Ha

In [106]:
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

# Use best pipeline: MLP + cosine index
# mlp_cos_embs already in memory (L2 normalized MLP embeddings)
# corpus_cos, query_cos already defined

print("=== Deep Analysis: Where is the recall gap? ===\n")

# ─── 1. Per-query recall vs GT size ───────────────────────────────────────────
print("--- 1. Recall vs GT size ---")

# Rebuild cosine index results
import nmslib, time
idx_cos = nmslib.init(method='hnsw', space='cosinesimil')
for i in range(len(corpus_cos)):
    idx_cos.addDataPoint(i, corpus_cos[i])
idx_cos.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)
idx_cos.setQueryTimeParams({'efSearch': 200})
nbrs_cos = idx_cos.knnQueryBatch(query_cos, k=50, num_threads=32)

# Per-query analysis
per_query = []
for i, (ids, dists) in enumerate(nbrs_cos):
    qid = QUERY_START_10K + i
    gt  = gt_lookup_10k.get(qid, [])
    if not gt: continue
    gt_set = set(gt[:50])
    recall = len(gt_set & set(ids[:50])) / len(gt_set)
    per_query.append({
        'qid': qid,
        'recall': recall,
        'gt_size': len(gt),
        'retrieved': list(ids[:50]),
        'gt': gt[:50]
    })

recalls    = [q['recall'] for q in per_query]
gt_sizes   = [q['gt_size'] for q in per_query]

# Bucket by GT size
buckets = [(1,5), (5,15), (15,50), (50,200), (200,700)]
print(f"{'GT size bucket':<20} {'Count':>6} {'Mean Recall':>12} {'Notes'}")
print("-" * 55)
for lo, hi in buckets:
    subset = [q for q in per_query if lo <= q['gt_size'] < hi]
    if subset:
        r = np.mean([q['recall'] for q in subset])
        print(f"  [{lo:3d}, {hi:3d}):          {len(subset):6d} {r:12.4f}")

# ─── 2. WJ similarity of missed vs found neighbors ────────────────────────────
print("\n--- 2. WJ similarity: found vs missed GT neighbors ---")
found_sims, missed_sims = [], []

for q in per_query[:200]:  # sample 200 queries
    qid      = q['qid']
    retrieved = set(q['retrieved'])
    gt_ids    = q['gt'][:50]

    q_vec = qt_10k[qid]
    for nid in gt_ids:
        sim = float(np.minimum(q_vec, qt_10k[nid]).sum() /
                    np.maximum(q_vec, qt_10k[nid]).sum())
        if nid in retrieved:
            found_sims.append(sim)
        else:
            missed_sims.append(sim)

print(f"Found neighbors  — WJ sim: {np.mean(found_sims):.4f} ± {np.std(found_sims):.4f} (n={len(found_sims)})")
print(f"Missed neighbors — WJ sim: {np.mean(missed_sims):.4f} ± {np.std(missed_sims):.4f} (n={len(missed_sims)})")
print(f"--> If missed sim is lower, we're missing the hardest true neighbors")
print(f"--> If missed sim is similar to found, it's a ranking problem")

# ─── 3. Cosine similarity of found vs missed in embedding space ───────────────
print("\n--- 3. Cosine similarity in embedding space: found vs missed ---")
found_cos, missed_cos = [], []

for q in per_query[:200]:
    qid       = q['qid']
    retrieved = set(q['retrieved'])
    gt_ids    = q['gt'][:50]

    q_emb = torch.tensor(mlp_cos_embs[qid]).unsqueeze(0)
    for nid in gt_ids:
        n_emb = torch.tensor(mlp_cos_embs[nid]).unsqueeze(0)
        sim   = F.cosine_similarity(q_emb, n_emb).item()
        if nid in retrieved:
            found_cos.append(sim)
        else:
            missed_cos.append(sim)

print(f"Found neighbors  — Cosine sim: {np.mean(found_cos):.4f} ± {np.std(found_cos):.4f}")
print(f"Missed neighbors — Cosine sim: {np.mean(missed_cos):.4f} ± {np.std(missed_cos):.4f}")
print(f"--> Gap = {np.mean(found_cos) - np.mean(missed_cos):.4f}")
print(f"--> If gap is large, embedding correctly ranks found > missed")
print(f"--> If gap is small, embedding CANNOT distinguish found from missed")

# ─── 4. What are we retrieving instead of the missed ones? ───────────────────
print("\n--- 4. Analysis of false positives (retrieved but not GT) ---")
fp_sims_qt   = []  # WJ sim of false positives in quadtree space
fn_sims_cos  = []  # cosine sim of false negatives (missed) in embedding space
fp_sims_cos  = []  # cosine sim of false positives in embedding space

for q in per_query[:200]:
    qid       = q['qid']
    retrieved = set(q['retrieved'])
    gt_set    = set(q['gt'][:50])

    false_pos = retrieved - gt_set   # retrieved but wrong
    false_neg = gt_set - retrieved   # missed true neighbors

    q_vec = qt_10k[qid]
    q_emb = torch.tensor(mlp_cos_embs[qid]).unsqueeze(0)

    for nid in list(false_pos)[:5]:
        fp_sims_qt.append(float(np.minimum(q_vec, qt_10k[nid]).sum() /
                                np.maximum(q_vec, qt_10k[nid]).sum()))
        n_emb = torch.tensor(mlp_cos_embs[nid]).unsqueeze(0)
        fp_sims_cos.append(F.cosine_similarity(q_emb, n_emb).item())

    for nid in list(false_neg)[:5]:
        n_emb = torch.tensor(mlp_cos_embs[nid]).unsqueeze(0)
        fn_sims_cos.append(F.cosine_similarity(q_emb, n_emb).item())

print(f"False positives (retrieved, not GT):")
print(f"  WJ sim in quadtree space:  {np.mean(fp_sims_qt):.4f} ± {np.std(fp_sims_qt):.4f}")
print(f"  Cosine sim in emb space:   {np.mean(fp_sims_cos):.4f} ± {np.std(fp_sims_cos):.4f}")
print(f"False negatives (missed GT):")
print(f"  Cosine sim in emb space:   {np.mean(fn_sims_cos):.4f} ± {np.std(fn_sims_cos):.4f}")
print(f"\nKey question: Are FP cosine sims > FN cosine sims?")
print(f"  FP cosine: {np.mean(fp_sims_cos):.4f} vs FN cosine: {np.mean(fn_sims_cos):.4f}")
print(f"  --> If FP > FN: embedding actively ranks wrong items higher than GT items")
print(f"  --> If FP < FN: HNSW graph traversal is missing valid paths (graph issue)")

=== Deep Analysis: Where is the recall gap? ===

--- 1. Recall vs GT size ---
GT size bucket        Count  Mean Recall Notes
-------------------------------------------------------
  [  1,   5):             138       0.9879
  [  5,  15):             146       0.9769
  [ 15,  50):             239       0.8605
  [ 50, 200):             482       0.7278
  [200, 700):             813       0.7794

--- 2. WJ similarity: found vs missed GT neighbors ---
Found neighbors  — WJ sim: 0.7286 ± 0.0671 (n=6242)
Missed neighbors — WJ sim: 0.6927 ± 0.0529 (n=1812)
--> If missed sim is lower, we're missing the hardest true neighbors
--> If missed sim is similar to found, it's a ranking problem

--- 3. Cosine similarity in embedding space: found vs missed ---
Found neighbors  — Cosine sim: 0.9924 ± 0.0195
Missed neighbors — Cosine sim: 0.9892 ± 0.0265
--> Gap = 0.0031
--> If gap is large, embedding correctly ranks found > missed
--> If gap is small, embedding CANNOT distinguish found from missed

--- 4

In [107]:
import nmslib
import numpy as np
import time

corpus_cos = mlp_cos_embs[:QUERY_START_10K]
query_cos  = mlp_cos_embs[QUERY_START_10K:]

print("=== HNSW M parameter sweep ===")
print(f"{'M':>6} {'efC':>6} {'R@10':>8} {'R@50':>8} {'QPS':>10} {'Build':>8}")
print("-" * 55)

for M in [20, 32, 48, 64]:
    idx = nmslib.init(method='hnsw', space='cosinesimil')
    for i in range(len(corpus_cos)):
        idx.addDataPoint(i, corpus_cos[i])
    t_build = time.time()
    idx.createIndex({'M': M, 'efConstruction': 400, 'post': 1}, print_progress=False)
    build_time = time.time() - t_build
    idx.setQueryTimeParams({'efSearch': 200})

    t0   = time.time()
    nbrs = idx.knnQueryBatch(query_cos, k=50, num_threads=32)
    qps  = len(query_cos) / (time.time() - t0)

    r10, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, QUERY_START_10K, K=10)
    r50, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs, QUERY_START_10K, K=50)
    print(f"{M:>6} {400:>6} {r10:>8.4f} {r50:>8.4f} {qps:>10.1f} {build_time:>7.2f}s")

=== HNSW M parameter sweep ===
     M    efC     R@10     R@50        QPS    Build
-------------------------------------------------------
    20    400   0.6656   0.8081    47526.7    0.26s
    32    400   0.6656   0.8081    46359.2    0.29s
    48    400   0.6656   0.8081    40014.2    0.34s
    64    400   0.6656   0.8081    40437.4    0.66s


In [108]:
import numpy as np
import nmslib
import time
from tqdm import tqdm

print("=== HNSW Candidate Retrieval + WJ Reranking ===\n")

corpus_cos = mlp_cos_embs[:QUERY_START_10K]
query_cos  = mlp_cos_embs[QUERY_START_10K:]
corpus_qt  = qt_10k[:QUERY_START_10K]
query_qt   = qt_10k[QUERY_START_10K:]

# Build cosine index once
idx = nmslib.init(method='hnsw', space='cosinesimil')
for i in range(len(corpus_cos)):
    idx.addDataPoint(i, corpus_cos[i])
idx.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=False)

def rerank_wj(query_vec, candidate_ids, corpus_qt):
    """Rerank candidates by exact WJ similarity."""
    sims = []
    for cid in candidate_ids:
        c_vec = corpus_qt[cid]
        wj    = np.minimum(query_vec, c_vec).sum() / np.maximum(query_vec, c_vec).sum()
        sims.append((wj, cid))
    sims.sort(reverse=True)
    return [cid for _, cid in sims]

print(f"{'Method':<35} {'R@10':>8} {'R@50':>8} {'QPS':>10}")
print("-" * 65)

# Baseline: cosine only K=50
idx.setQueryTimeParams({'efSearch': 200})
t0 = time.time()
nbrs_cos50 = idx.knnQueryBatch(query_cos, k=50, num_threads=32)
qps = len(query_cos) / (time.time() - t0)
r10, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs_cos50, QUERY_START_10K, K=10)
r50, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs_cos50, QUERY_START_10K, K=50)
print(f"{'Cosine HNSW (K=50, no rerank)':<35} {r10:>8.4f} {r50:>8.4f} {qps:>10.1f}")

# Reranking at different candidate sizes
for K_cand in [100, 200, 500]:
    t0       = time.time()
    nbrs_raw = idx.knnQueryBatch(query_cos, k=K_cand, num_threads=32)
    t_hnsw   = time.time() - t0

    # Rerank by WJ
    t_rerank = time.time()
    reranked = []
    for i, (ids, dists) in enumerate(nbrs_raw):
        q_vec      = query_qt[i]
        reranked_ids = rerank_wj(q_vec, ids, corpus_qt)
        reranked.append((reranked_ids[:50], dists[:50]))
    t_rerank = time.time() - t_rerank

    total_time = t_hnsw + t_rerank
    qps_total  = len(query_cos) / total_time

    r10, _, _ = compute_recall_at_k(gt_lookup_10k, reranked, QUERY_START_10K, K=10)
    r50, _, _ = compute_recall_at_k(gt_lookup_10k, reranked, QUERY_START_10K, K=50)
    print(f"{'Cosine HNSW (K='+str(K_cand)+') + WJ rerank':<35} {r10:>8.4f} {r50:>8.4f} {qps_total:>10.1f}")

# Also try WJ-only baseline for comparison
t0 = time.time()
nbrs_wj = idx.knnQueryBatch(query_cos, k=50, num_threads=32)
qps_wj  = len(query_cos) / (time.time() - t0)
r10, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs_wj, QUERY_START_10K, K=10)
r50, _, _ = compute_recall_at_k(gt_lookup_10k, nbrs_wj, QUERY_START_10K, K=50)
print(f"{'Cosine HNSW (K=50, WJ rerank)':<35} {'—':>8} {'same':>8} {'—':>10}")
print(f"\nBaseline (quadtree exact):         R@10=0.9965  R@50=0.9986  QPS=293")

=== HNSW Candidate Retrieval + WJ Reranking ===

Method                                  R@10     R@50        QPS
-----------------------------------------------------------------
Cosine HNSW (K=50, no rerank)         0.6656   0.8081    52923.0
Cosine HNSW (K=100) + WJ rerank       0.9948   0.9582      419.7
Cosine HNSW (K=200) + WJ rerank       0.9964   0.9934      211.4
Cosine HNSW (K=500) + WJ rerank       0.9966   0.9984       85.0
Cosine HNSW (K=50, WJ rerank)              —     same          —

Baseline (quadtree exact):         R@10=0.9965  R@50=0.9986  QPS=293


In [112]:
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import nmslib
import time
from tqdm import tqdm

def rerank_wj_single(args):
    ids, query_vec, corpus_qt = args
    sims = []
    for cid in ids:
        wj = float(np.minimum(query_vec, corpus_qt[cid]).sum() /
                   np.maximum(query_vec, corpus_qt[cid]).sum())
        sims.append((wj, cid))
    sims.sort(reverse=True)
    return [c for _, c in sims]  # return ALL reranked, no slice

def rerank_wj_parallel(query_qt, nbrs_raw, corpus_qt, n_workers=64):
    args = [(ids, query_qt[i], corpus_qt)
            for i, (ids, _) in enumerate(nbrs_raw)]
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        reranked_ids = list(ex.map(rerank_wj_single, args))
    # Return in same format as NMSLIB: list of (ids, dists)
    return [(ids, []) for ids in reranked_ids]

corpus_qt_full = qtree_vectors[:QUERY_START_ID]
query_qt_full  = qtree_vectors[QUERY_START_ID:]

# ─── Build cosine index (once) ─────────────────────────────────────────────
print("Building cosine HNSW index...")
import torch, torch.nn.functional as F
compressor_full = QuadtreeCompressorV1(in_dim=IN_DIM, out_dim=512).to(device)
compressor_full.load_state_dict(
    torch.load('/tmp/best_compressor_full.pt', weights_only=True))
compressor_full.eval()

all_embs = []
with torch.no_grad():
    for start in tqdm(range(0, len(qtree_vectors), 512), desc="Embedding"):
        batch = torch.tensor(qtree_vectors[start:start+512],
                             dtype=torch.float32).to(device)
        raw   = compressor_full.net(batch)
        emb   = F.normalize(raw, dim=1)
        all_embs.append(emb.cpu().numpy())

embs_cos_full   = np.vstack(all_embs)
corpus_cos_full = embs_cos_full[:QUERY_START_ID]
query_cos_full  = embs_cos_full[QUERY_START_ID:]

idx_cos = nmslib.init(method='hnsw', space='cosinesimil')
for i in tqdm(range(len(corpus_cos_full)), desc="Adding", mininterval=2.0):
    idx_cos.addDataPoint(i, corpus_cos_full[i])
idx_cos.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=True)
idx_cos.setQueryTimeParams({'efSearch': 200})
print("Index built.")

# ─── Evaluation configs ────────────────────────────────────────────────────
# Each config: (K_candidates, K_eval_list)
# Rule: only evaluate at K_eval <= K_candidates
configs = [
    (50,  [10, 50],           'MLP+Cosine (K=50, no rerank)',   False),
    (100, [10, 50, 100],      'MLP+Cosine (K=100+WJrerank)',     True),
    (200, [10, 50, 100],      'MLP+Cosine (K=200+WJrerank)',     True),
    (500, [10, 50, 100, 500], 'MLP+Cosine (K=500+WJrerank)',     True),
]

print(f"\n{'Method':<40} {'R@10':>7} {'R@50':>7} {'R@100':>7} {'R@500':>7} {'QPS':>8}")
print("-" * 82)

for K_cand, K_evals, label, do_rerank in configs:
    t0       = time.time()
    nbrs_raw = idx_cos.knnQueryBatch(query_cos_full, k=K_cand, num_threads=32)
    t_hnsw   = time.time() - t0

    if do_rerank:
        t_rr    = time.time()
        nbrs    = rerank_wj_parallel(query_qt_full, nbrs_raw,
                                      corpus_qt_full, n_workers=64)
        t_rr    = time.time() - t_rr
        qps     = len(query_cos_full) / (t_hnsw + t_rr)
    else:
        nbrs = nbrs_raw
        qps  = len(query_cos_full) / t_hnsw

    # Evaluate at valid K values only
    rec = {}
    for K in K_evals:
        total = 0.0; count = 0
        for i, (ids, _) in enumerate(nbrs):
            qid = QUERY_START_ID + i
            gt  = set(gt_lookup.get(qid, [])[:K])
            if not gt: continue
            # Slice retrieved to K — correct because reranked list has K_cand items
            total += len(gt & set(ids[:K])) / len(gt)
            count += 1
        rec[K] = total / count if count > 0 else 0.0

    r10  = rec.get(10,  float('nan'))
    r50  = rec.get(50,  float('nan'))
    r100 = rec.get(100, float('nan'))
    r500 = rec.get(500, float('nan'))

    r10s  = f"{r10:.4f}"  if not np.isnan(r10)  else "  —   "
    r50s  = f"{r50:.4f}"  if not np.isnan(r50)  else "  —   "
    r100s = f"{r100:.4f}" if not np.isnan(r100) else "  —   "
    r500s = f"{r500:.4f}" if not np.isnan(r500) else "  —   "

    print(f"{label:<40} {r10s:>7} {r50s:>7} {r100s:>7} {r500s:>7} {qps:>8.1f}")

# Baseline reference
print(f"\n{'Baseline (Quadtree+WJ)':<40} {'0.9925':>7} {'0.9953':>7} {'0.9963':>7} {'0.9864':>7} {'517':>8}")

Building cosine HNSW index...


Adding: 100%|██████████| 187019/187019 [00:00<00:00, 569649.34it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Index built.

Method                                      R@10    R@50   R@100   R@500      QPS
----------------------------------------------------------------------------------
MLP+Cosine (K=50, no rerank)              0.5328  0.6060    —       —      4735.4
MLP+Cosine (K=100+WJrerank)               0.9083  0.7795  0.6295    —       223.1


KeyboardInterrupt: 

In [114]:
import torch
import torch.nn.functional as F
import numpy as np
import os

# ─── Confirm dimensions ───────────────────────────────────────────────────────
print(f"Full dataset input dim: {IN_DIM}")          # should be 18220
print(f"10k dataset input dim:  {IN_DIM_10K}")      # should be 18499

# ─── Check what model is saved ────────────────────────────────────────────────
print("\n=== Model file check ===")
for f in ['/tmp/best_compressor_v1_clean.pt',
          '/tmp/best_compressor_full.pt']:
    if os.path.exists(f):
        ckpt   = torch.load(f, weights_only=True)
        layer0 = ckpt['net.0.weight']
        print(f"  {os.path.basename(f)}: input_dim={layer0.shape[1]}, hidden={layer0.shape[0]}")
    else:
        print(f"  MISSING: {f}")

# ─── Load correct full dataset model ─────────────────────────────────────────
print(f"\nLoading full dataset model (in_dim={IN_DIM})...")
compressor_full = QuadtreeCompressorV1(in_dim=IN_DIM, out_dim=512).to(device)
compressor_full.load_state_dict(
    torch.load('/tmp/best_compressor_full.pt', weights_only=True))
compressor_full.eval()

# ─── Quick quality check on full dataset data ─────────────────────────────────
print("Checking embedding quality on full dataset...")
sample_ids  = list(range(100))
gt_sims, rand_sims = [], []

for i in range(100):
    qid    = QUERY_START_ID + i
    pos_id = gt_lookup.get(qid, [None])[0]
    if pos_id is None: continue
    rand_id = np.random.randint(0, QUERY_START_ID)

    vq = torch.tensor(qtree_vectors[qid],     dtype=torch.float32).unsqueeze(0).to(device)
    vp = torch.tensor(qtree_vectors[pos_id],  dtype=torch.float32).unsqueeze(0).to(device)
    vr = torch.tensor(qtree_vectors[rand_id], dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        eq = F.normalize(compressor_full.net(vq), dim=1)
        ep = F.normalize(compressor_full.net(vp), dim=1)
        er = F.normalize(compressor_full.net(vr), dim=1)

    gt_sims.append(F.cosine_similarity(eq, ep).item())
    rand_sims.append(F.cosine_similarity(eq, er).item())

print(f"GT pair cosine sim:  {np.mean(gt_sims):.4f} ± {np.std(gt_sims):.4f}")
print(f"Random cosine sim:   {np.mean(rand_sims):.4f} ± {np.std(rand_sims):.4f}")
print(f"Separation gap:      {np.mean(gt_sims) - np.mean(rand_sims):.4f}")
print(f"\n--> 10k best model gap was ~0.08")
print(f"--> If this gap is similar, model quality is comparable")
print(f"--> If gap << 0.08, model is weaker and we need to retrain")

Full dataset input dim: 18220
10k dataset input dim:  18499

=== Model file check ===
  best_compressor_v1_clean.pt: input_dim=18499, hidden=4096
  best_compressor_full.pt: input_dim=18220, hidden=4096

Loading full dataset model (in_dim=18220)...
Checking embedding quality on full dataset...
GT pair cosine sim:  0.9999 ± 0.0003
Random cosine sim:   0.9874 ± 0.0245
Separation gap:      0.0125

--> 10k best model gap was ~0.08
--> If this gap is similar, model quality is comparable
--> If gap << 0.08, model is weaker and we need to retrain


In [115]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
import random

# ─── Fresh full dataset MLP ───────────────────────────────────────────────────
model_full = QuadtreeCompressorV1(in_dim=IN_DIM, out_dim=512).to(device)
if torch.cuda.device_count() > 1:
    model_full_par = nn.DataParallel(model_full)
else:
    model_full_par = model_full

total_params = sum(p.numel() for p in model_full.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Input dim: {IN_DIM}")

# ─── Dataset ──────────────────────────────────────────────────────────────────
dataset_full = AnchorPositiveDataset(
    qtree_vectors, gt_lookup,
    query_start=QUERY_START_ID,
    max_pos=30)
loader_full = DataLoader(
    dataset_full, batch_size=1024, shuffle=True,
    num_workers=0, pin_memory=False, drop_last=True)
print(f"Total pairs: {len(dataset_full)}")
print(f"Steps per epoch: {len(loader_full)}")

# ─── Training ─────────────────────────────────────────────────────────────────
EPOCHS    = 100   # more epochs than 10k since more data
optimizer = torch.optim.AdamW(model_full.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_loss = float('inf')
for epoch in range(EPOCHS):
    model_full_par.train()
    total_loss = 0.0
    total_steps = 0

    pbar = tqdm(loader_full, desc=f"Epoch {epoch+1:3d}/{EPOCHS}", leave=False)
    for anchors, positives, _, _ in pbar:
        anchors   = anchors.to(device)
        positives = positives.to(device)
        B         = anchors.shape[0]

        combined  = torch.cat([anchors, positives], dim=0)
        out       = model_full_par(combined)
        out_norm  = F.normalize(out, dim=1)
        a_emb     = out_norm[:B]
        p_emb     = out_norm[B:]

        loss, _ = vectorized_hard_triplet_loss(a_emb, p_emb, margin=0.3)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_full.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item()
        total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / total_steps
    scheduler.step()

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_full.state_dict(), '/tmp/best_compressor_full_v2.pt')

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nDone. Best loss: {best_loss:.4f}")

Model parameters: 79,358,976
Input dim: 18220
Total pairs: 1285479
Total pairs: 1285479
Steps per epoch: 1255


Epoch  10/100 | Loss: 0.1483 | Best: 0.1483 | LR: 0.000976


Epoch  20/100 | Loss: 0.1465 | Best: 0.1463 | LR: 0.000905


Epoch  30/100 | Loss: 0.1453 | Best: 0.1452 | LR: 0.000794


Epoch  40/100 | Loss: 0.1456 | Best: 0.1451 | LR: 0.000655


...

Epoch  60/100 | Loss: 0.1492 | Best: 0.1451 | LR: 0.000345


Epoch  70/100 | Loss: 0.1484 | Best: 0.1451 | LR: 0.000206


Epoch  80/100 | Loss: 0.1478 | Best: 0.1451 | LR: 0.000095


Epoch  90/100 | Loss: 0.1476 | Best: 0.1451 | LR: 0.000024


Epoch 100/100 | Loss: 0.1475 | Best: 0.1451 | LR: 0.000000

Done. Best loss: 0.1451


In [116]:
import torch
import torch.nn.functional as F
import numpy as np
import nmslib
import time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# ─── Load retrained model ─────────────────────────────────────────────────────
model_full = QuadtreeCompressorV1(in_dim=IN_DIM, out_dim=512).to(device)
model_full.load_state_dict(
    torch.load('/tmp/best_compressor_full_v2.pt', weights_only=True))
model_full.eval()
print("Full dataset model v2 loaded.")

# ─── Check embedding quality first ───────────────────────────────────────────
gt_sims, rand_sims = [], []
for i in range(200):
    qid    = QUERY_START_ID + i
    pos_id = gt_lookup.get(qid, [None])[0]
    if pos_id is None: continue
    rand_id = np.random.randint(0, QUERY_START_ID)

    vq = torch.tensor(qtree_vectors[qid],     dtype=torch.float32).unsqueeze(0).to(device)
    vp = torch.tensor(qtree_vectors[pos_id],  dtype=torch.float32).unsqueeze(0).to(device)
    vr = torch.tensor(qtree_vectors[rand_id], dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        eq = F.normalize(model_full.net(vq), dim=1)
        ep = F.normalize(model_full.net(vp), dim=1)
        er = F.normalize(model_full.net(vr), dim=1)

    gt_sims.append(F.cosine_similarity(eq, ep).item())
    rand_sims.append(F.cosine_similarity(eq, er).item())

print(f"GT pair cosine sim:  {np.mean(gt_sims):.4f} ± {np.std(gt_sims):.4f}")
print(f"Random cosine sim:   {np.mean(rand_sims):.4f} ± {np.std(rand_sims):.4f}")
print(f"Separation gap:      {np.mean(gt_sims) - np.mean(rand_sims):.4f}")
print(f"--> Old full model gap: 0.0125 | 10k model gap: ~0.08")

# ─── Generate embeddings ──────────────────────────────────────────────────────
print("\nGenerating cosine embeddings for full dataset...")
all_embs = []
with torch.no_grad():
    for start in tqdm(range(0, len(qtree_vectors), 512), desc="Embedding"):
        batch = torch.tensor(qtree_vectors[start:start+512],
                             dtype=torch.float32).to(device)
        raw   = model_full.net(batch)
        emb   = F.normalize(raw, dim=1)
        all_embs.append(emb.cpu().numpy())

embs_cos_full   = np.vstack(all_embs)
corpus_cos_full = embs_cos_full[:QUERY_START_ID]
query_cos_full  = embs_cos_full[QUERY_START_ID:]
corpus_qt_full  = qtree_vectors[:QUERY_START_ID]
query_qt_full   = qtree_vectors[QUERY_START_ID:]
print(f"Embeddings: {embs_cos_full.shape} | VecSize: {corpus_cos_full.nbytes/1024**2:.1f} MB")

# ─── Utilities ────────────────────────────────────────────────────────────────
def eval_recall(gt_lookup, nbrs, query_start_id, K):
    total = 0.0; count = 0
    for i, (ids, _) in enumerate(nbrs):
        qid = query_start_id + i
        gt  = set(gt_lookup.get(qid, [])[:K])
        if not gt: continue
        total += len(gt & set(ids[:K])) / len(gt)
        count += 1
    return total / count if count > 0 else 0.0

def rerank_single(args):
    ids, q_vec, corpus_qt = args
    sims = [(float(np.minimum(q_vec, corpus_qt[c]).sum() /
                   np.maximum(q_vec, corpus_qt[c]).sum()), c)
            for c in ids]
    sims.sort(reverse=True)
    return ([c for _, c in sims], [])

def rerank_parallel(query_qt, nbrs_raw, corpus_qt, n_workers=64):
    args = [(ids, query_qt[i], corpus_qt) for i, (ids, _) in enumerate(nbrs_raw)]
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        return list(ex.map(rerank_single, args))

# ─── Build cosine HNSW index ──────────────────────────────────────────────────
print("\nBuilding cosine HNSW index...")
import psutil, os
mem_before = psutil.Process(os.getpid()).memory_info().rss / 1024**2
idx_cos = nmslib.init(method='hnsw', space='cosinesimil')
for i in tqdm(range(len(corpus_cos_full)), desc="Adding", mininterval=2.0):
    idx_cos.addDataPoint(i, corpus_cos_full[i])
t0 = time.time()
idx_cos.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=True)
build_cos = time.time() - t0
mem_cos   = psutil.Process(os.getpid()).memory_info().rss / 1024**2 - mem_before
idx_cos.setQueryTimeParams({'efSearch': 200})
print(f"Built in {build_cos:.1f}s | Index memory: {mem_cos:.1f} MB")

# ─── Also build baseline WJ index ─────────────────────────────────────────────
print("\nBuilding baseline WJ HNSW index...")
mem_before = psutil.Process(os.getpid()).memory_info().rss / 1024**2
idx_bl = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in tqdm(range(len(corpus_qt_full)), desc="Adding", mininterval=2.0):
    idx_bl.addDataPoint(i, corpus_qt_full[i])
t0 = time.time()
idx_bl.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=True)
build_bl = time.time() - t0
mem_bl   = psutil.Process(os.getpid()).memory_info().rss / 1024**2 - mem_before
idx_bl.setQueryTimeParams({'efSearch': 200})
print(f"Built in {build_bl:.1f}s | Index memory: {mem_bl:.1f} MB")

# ─── Evaluate baseline ────────────────────────────────────────────────────────
print("\nEvaluating baseline...")
t0      = time.time()
nbrs_bl = idx_bl.knnQueryBatch(query_qt_full, k=500, num_threads=32)
qps_bl  = len(query_qt_full) / (time.time() - t0)
del idx_bl

# ─── Evaluate all our methods ─────────────────────────────────────────────────
print("Evaluating MLP+Cosine (no rerank)...")
t0          = time.time()
nbrs_cos500 = idx_cos.knnQueryBatch(query_cos_full, k=500, num_threads=32)
qps_cos     = len(query_cos_full) / (time.time() - t0)

print("Evaluating MLP+Cosine K=100 + WJ rerank...")
t0       = time.time()
nbrs_100 = idx_cos.knnQueryBatch(query_cos_full, k=100, num_threads=32)
t_h100   = time.time() - t0
t0       = time.time()
nbrs_rr100 = rerank_parallel(query_qt_full, nbrs_100, corpus_qt_full)
t_rr100  = time.time() - t0
qps_rr100 = len(query_cos_full) / (t_h100 + t_rr100)

print("Evaluating MLP+Cosine K=200 + WJ rerank...")
t0       = time.time()
nbrs_200 = idx_cos.knnQueryBatch(query_cos_full, k=200, num_threads=32)
t_h200   = time.time() - t0
t0       = time.time()
nbrs_rr200 = rerank_parallel(query_qt_full, nbrs_200, corpus_qt_full)
t_rr200  = time.time() - t0
qps_rr200 = len(query_cos_full) / (t_h200 + t_rr200)

# ─── Print full table ─────────────────────────────────────────────────────────
vec_bl  = corpus_qt_full.nbytes  / 1024**2
vec_cos = corpus_cos_full.nbytes / 1024**2

print(f"\n{'='*110}")
print(f"FINAL COMPREHENSIVE COMPARISON — Full Dataset (233k polygons, 32 threads)")
print(f"{'='*110}")
print(f"{'Method':<38} {'R@10':>7} {'R@50':>7} {'R@100':>7} {'R@500':>7} "
      f"{'QPS':>8} {'Build':>8} {'Vec(MB)':>9} {'Idx(MB)':>9}")
print("-"*110)

rows = [
    ("Baseline (Quadtree+WJ)",   nbrs_bl,     qps_bl,    build_bl, vec_bl,  mem_bl,  [10,50,100,500]),
    ("MLP+Cosine (no rerank)",   nbrs_cos500, qps_cos,   build_cos,vec_cos, mem_cos, [10,50,100,500]),
    ("MLP+Cosine K=100+WJrerank",nbrs_rr100,  qps_rr100, build_cos,vec_cos, mem_cos, [10,50,100]),
    ("MLP+Cosine K=200+WJrerank",nbrs_rr200,  qps_rr200, build_cos,vec_cos, mem_cos, [10,50,100]),
]

for name, nbrs, qps, build, vec, idx_mem, valid_k in rows:
    recs = {k: eval_recall(gt_lookup, nbrs, QUERY_START_ID, k) for k in valid_k}
    r10  = f"{recs.get(10,''):>7.4f}"  if 10  in valid_k else f"{'—':>7}"
    r50  = f"{recs.get(50,''):>7.4f}"  if 50  in valid_k else f"{'—':>7}"
    r100 = f"{recs.get(100,''):>7.4f}" if 100 in valid_k else f"{'—':>7}"
    r500 = f"{recs.get(500,''):>7.4f}" if 500 in valid_k else f"{'—':>7}"
    print(f"{name:<38} {r10} {r50} {r100} {r500} "
          f"{qps:>8.1f} {build:>7.1f}s {vec:>8.1f} {idx_mem:>8.1f}")

print(f"\nKey improvements (vs Baseline):")
print(f"  Vector size:  {vec_bl:.1f} MB → {vec_cos:.1f} MB  ({vec_bl/vec_cos:.1f}x smaller)")
print(f"  Index memory: {mem_bl:.1f} MB → {mem_cos:.1f} MB  ({mem_bl/mem_cos:.1f}x smaller)")
print(f"  Build time:   {build_bl:.1f}s → {build_cos:.1f}s  ({build_bl/build_cos:.1f}x faster)")
print(f"  QPS (no rerank): {qps_cos/qps_bl:.1f}x speedup")
print(f"  QPS (K=100+rerank): {qps_rr100/qps_bl:.1f}x vs baseline")

Full dataset model v2 loaded.
GT pair cosine sim:  1.0000 ± 0.0000
Random cosine sim:   1.0000 ± 0.0000
Separation gap:      0.0000
--> Old full model gap: 0.0125 | 10k model gap: ~0.08

Generating cosine embeddings for full dataset...


Embedding: 100%|██████████| 457/457 [00:08<00:00, 54.21it/s]


Embeddings: (233773, 512) | VecSize: 365.3 MB

Building cosine HNSW index...


Adding: 100%|██████████| 187019/187019 [00:00<00:00, 557364.36it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***********************************************************



Built in 42.1s | Index memory: 4.1 MB

Building baseline WJ HNSW index...


Adding: 100%|██████████| 187019/187019 [00:03<00:00, 49509.67it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
*******************************************************


2026-04-26 08:41:43 hnsw.cc:407 (CreateIndex) [INFO] No appropriate custom distance function for custom space


Built in 430.4s | Index memory: 3.0 MB

Evaluating baseline...


KeyboardInterrupt: 

In [117]:
import torch
import torch.nn.functional as F
import numpy as np

model_full.eval()

# ─── Layer by layer inspection ────────────────────────────────────────────────
print("=== Layer-by-layer activation on full dataset sample ===")
sample = torch.tensor(qtree_vectors[:128], dtype=torch.float32).to(device)

x = sample
for i, layer in enumerate(model_full.net):
    x = layer(x)
    print(f"Layer {i} ({layer.__class__.__name__:>12}): "
          f"mean={x.mean().item():>10.6f}, std={x.std().item():.6f}")

emb = F.normalize(x, dim=1)
print(f"\nAfter L2 normalize:")
print(f"  std:  {emb.std().item():.8f}")
print(f"  mean: {emb.mean().item():.8f}")

# Pairwise cosine sim
sim = torch.mm(emb[:32], emb[32:].T)
print(f"  Pairwise cosine sim mean: {sim.mean().item():.8f}")
print(f"  Pairwise cosine sim std:  {sim.std().item():.8f}")

# ─── Input statistics ─────────────────────────────────────────────────────────
print(f"\n=== Input statistics ===")
print(f"qtree_vectors dtype:    {qtree_vectors.dtype}")
print(f"qtree_vectors range:    [{qtree_vectors.min():.2e}, {qtree_vectors.max():.2e}]")
print(f"qtree_vectors mean:     {qtree_vectors.mean():.2e}")
print(f"qtree_vectors sparsity: {(qtree_vectors == 0).mean():.4f}")
print(f"qtree_vectors shape:    {qtree_vectors.shape}")

# ─── Check first layer output specifically ────────────────────────────────────
print(f"\n=== First linear layer analysis ===")
layer0_out = model_full.net[0](sample)
print(f"Layer 0 output std:  {layer0_out.std().item():.8f}")
print(f"Layer 0 output mean: {layer0_out.mean().item():.8f}")
print(f"Layer 0 output min:  {layer0_out.min().item():.8f}")
print(f"Layer 0 output max:  {layer0_out.max().item():.8f}")

# ─── Check if different samples give different outputs ────────────────────────
print(f"\n=== Sample diversity check ===")
s1 = torch.tensor(qtree_vectors[0:64],   dtype=torch.float32).to(device)
s2 = torch.tensor(qtree_vectors[64:128], dtype=torch.float32).to(device)
with torch.no_grad():
    e1 = F.normalize(model_full.net(s1), dim=1)
    e2 = F.normalize(model_full.net(s2), dim=1)

pairwise = torch.mm(e1, e2.T)
print(f"Cross-batch cosine sim mean: {pairwise.mean().item():.8f}")
print(f"Cross-batch cosine sim std:  {pairwise.std().item():.8f}")
print(f"Diagonal (same-position):    {pairwise.diag().mean().item():.8f}")
print(f"--> If all values ~1.0, complete collapse confirmed")
print(f"--> If std > 0.01, embeddings have meaningful diversity")

=== Layer-by-layer activation on full dataset sample ===
Layer 0 (      Linear): mean= -0.000000, std=0.000000
Layer 1 ( BatchNorm1d): mean=  0.079495, std=0.415823
Layer 2 (        ReLU): mean=  0.100240, std=0.409826
Layer 3 (      Linear): mean= 11.906528, std=39.457157
Layer 4 ( BatchNorm1d): mean=  4.334697, std=27.270853
Layer 5 (        ReLU): mean=  6.727527, std=25.386312
Layer 6 (      Linear): mean=  7.419313, std=155.688980
Layer 7 ( BatchNorm1d): mean=  2.503306, std=162.118164

After L2 normalize:
  std:  0.04418924
  mean: 0.00068233
  Pairwise cosine sim mean: 0.99999774
  Pairwise cosine sim std:  0.00000552

=== Input statistics ===
qtree_vectors dtype:    float32
qtree_vectors range:    [0.00e+00, 7.91e-02]
qtree_vectors mean:     1.26e-09
qtree_vectors sparsity: 0.7302
qtree_vectors shape:    (233773, 18220)

=== First linear layer analysis ===
Layer 0 output std:  0.00000027
Layer 0 output mean: -0.00000001
Layer 0 output min:  -0.00002311
Layer 0 output max:  0.00

In [118]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

# ─── Fixed MLP with log1p input scaling ───────────────────────────────────────
class QuadtreeCompressorV1Fixed(nn.Module):
    """MLP compressor with log1p input scaling for numerical stability."""
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False),
            nn.BatchNorm1d(4096),
            nn.ReLU(),
            nn.Linear(4096, 1024, bias=False),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
        )

    def forward(self, x, for_index=False):
        x   = torch.log1p(x * 1e6)   # ← critical fix
        out = self.net(x)
        if for_index:
            out = F.relu(out)
            out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return out

# ─── Verify fix immediately ───────────────────────────────────────────────────
model_fixed = QuadtreeCompressorV1Fixed(in_dim=IN_DIM, out_dim=512).to(device)

sample = torch.tensor(qtree_vectors[:128], dtype=torch.float32).to(device)
with torch.no_grad():
    # Check log1p output
    x_log = torch.log1p(sample * 1e6)
    print(f"After log1p: mean={x_log.mean().item():.4f}, std={x_log.std().item():.4f}")

    # Check layer 0
    layer0_out = model_fixed.net[0](x_log)
    print(f"Layer 0 std after fix: {layer0_out.std().item():.4f}  (was 0.0000003)")

    # Check embedding diversity
    emb = F.normalize(model_fixed.net(x_log), dim=1)
    sim = torch.mm(emb[:32], emb[32:].T)
    print(f"Pairwise cosine sim std: {sim.std().item():.4f}  (was 0.000006)")
    print(f"--> Want std > 0.01 for meaningful embeddings")

# ─── Train fixed model ────────────────────────────────────────────────────────
if torch.cuda.device_count() > 1:
    model_fixed_par = nn.DataParallel(model_fixed)
else:
    model_fixed_par = model_fixed

print(f"\nModel parameters: {sum(p.numel() for p in model_fixed.parameters()):,}")

dataset_full = AnchorPositiveDataset(
    qtree_vectors, gt_lookup,
    query_start=QUERY_START_ID,
    max_pos=30)
loader_full = DataLoader(
    dataset_full, batch_size=1024, shuffle=True,
    num_workers=0, pin_memory=False, drop_last=True)
print(f"Total pairs: {len(dataset_full)}")
print(f"Steps per epoch: {len(loader_full)}")

EPOCHS    = 50
optimizer = torch.optim.AdamW(model_fixed.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_loss = float('inf')
for epoch in range(EPOCHS):
    model_fixed_par.train()
    total_loss = 0.0; total_steps = 0

    pbar = tqdm(loader_full, desc=f"Epoch {epoch+1:3d}/{EPOCHS}", leave=False)
    for anchors, positives, _, _ in pbar:
        anchors   = anchors.to(device)
        positives = positives.to(device)
        B         = anchors.shape[0]

        combined = torch.cat([anchors, positives], dim=0)
        out      = model_fixed_par(combined)
        out_norm = F.normalize(out, dim=1)
        a_emb    = out_norm[:B]
        p_emb    = out_norm[B:]

        loss, _ = vectorized_hard_triplet_loss(a_emb, p_emb, margin=0.3)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_fixed.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item(); total_steps += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / total_steps
    scheduler.step()

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_fixed.state_dict(), '/tmp/best_compressor_full_fixed.pt')

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | "
              f"Best: {best_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print(f"\nDone. Best loss: {best_loss:.4f}")

After log1p: mean=0.0004, std=0.0039
Layer 0 std after fix: 0.0023  (was 0.0000003)
Pairwise cosine sim std: 0.5408  (was 0.000006)
--> Want std > 0.01 for meaningful embeddings

Model parameters: 79,358,976
Total pairs: 1285479
Total pairs: 1285479
Steps per epoch: 1255


Epoch   5/50 | Loss: 0.1503 | Best: 0.1503 | LR: 0.000976


Epoch  10/50 | Loss: 0.1449 | Best: 0.1449 | LR: 0.000905


Epoch  15/50 | Loss: 0.1436 | Best: 0.1436 | LR: 0.000794


Epoch  20/50 | Loss: 0.1429 | Best: 0.1428 | LR: 0.000655


...

Epoch  30/50 | Loss: 0.1426 | Best: 0.1424 | LR: 0.000345


Epoch  35/50 | Loss: 0.1424 | Best: 0.1423 | LR: 0.000206


Epoch  40/50 | Loss: 0.1424 | Best: 0.1423 | LR: 0.000095


Epoch  45/50 | Loss: 0.1427 | Best: 0.1423 | LR: 0.000024


Epoch  50/50 | Loss: 0.1427 | Best: 0.1423 | LR: 0.000000

Done. Best loss: 0.1423


In [119]:
import torch
import torch.nn.functional as F
import numpy as np
import nmslib
import time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import psutil, os

# ─── Load fixed model ─────────────────────────────────────────────────────────
model_fixed = QuadtreeCompressorV1Fixed(in_dim=IN_DIM, out_dim=512).to(device)
model_fixed.load_state_dict(
    torch.load('/tmp/best_compressor_full_fixed.pt', weights_only=True))
model_fixed.eval()
print("Fixed model loaded.")

# ─── Embedding quality check ──────────────────────────────────────────────────
print("\n=== Embedding quality check ===")
gt_sims, rand_sims = [], []
for i in range(200):
    qid    = QUERY_START_ID + i
    pos_id = gt_lookup.get(qid, [None])[0]
    if pos_id is None: continue
    rand_id = np.random.randint(0, QUERY_START_ID)

    vq = torch.tensor(qtree_vectors[qid],     dtype=torch.float32).unsqueeze(0).to(device)
    vp = torch.tensor(qtree_vectors[pos_id],  dtype=torch.float32).unsqueeze(0).to(device)
    vr = torch.tensor(qtree_vectors[rand_id], dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        eq = F.normalize(model_fixed(vq), dim=1)
        ep = F.normalize(model_fixed(vp), dim=1)
        er = F.normalize(model_fixed(vr), dim=1)

    gt_sims.append(F.cosine_similarity(eq, ep).item())
    rand_sims.append(F.cosine_similarity(eq, er).item())

gap = np.mean(gt_sims) - np.mean(rand_sims)
print(f"GT pair cosine sim:  {np.mean(gt_sims):.4f} ± {np.std(gt_sims):.4f}")
print(f"Random cosine sim:   {np.mean(rand_sims):.4f} ± {np.std(rand_sims):.4f}")
print(f"Separation gap:      {gap:.4f}  (10k model was ~0.08)")

if gap < 0.02:
    print("WARNING: gap still too low — embeddings may not work well")
else:
    print("OK: gap is meaningful — proceeding to evaluation")

# ─── Generate embeddings ──────────────────────────────────────────────────────
print("\nGenerating cosine embeddings...")
all_embs = []
with torch.no_grad():
    for start in tqdm(range(0, len(qtree_vectors), 512), desc="Embedding"):
        batch = torch.tensor(qtree_vectors[start:start+512],
                             dtype=torch.float32).to(device)
        emb   = F.normalize(model_fixed(batch), dim=1)
        all_embs.append(emb.cpu().numpy())

embs_cos_full   = np.vstack(all_embs)
corpus_cos_full = embs_cos_full[:QUERY_START_ID]
query_cos_full  = embs_cos_full[QUERY_START_ID:]
corpus_qt_full  = qtree_vectors[:QUERY_START_ID]
query_qt_full   = qtree_vectors[QUERY_START_ID:]

print(f"Embeddings: {embs_cos_full.shape}")
print(f"Vector size: {corpus_cos_full.nbytes/1024**2:.1f} MB "
      f"(baseline: {corpus_qt_full.nbytes/1024**2:.1f} MB, "
      f"{corpus_qt_full.nbytes/corpus_cos_full.nbytes:.1f}x larger)")

# ─── Utilities ────────────────────────────────────────────────────────────────
def eval_recall(gt_lookup, nbrs, query_start_id, K):
    total = 0.0; count = 0
    for i, (ids, _) in enumerate(nbrs):
        qid = query_start_id + i
        gt  = set(gt_lookup.get(qid, [])[:K])
        if not gt: continue
        total += len(gt & set(ids[:K])) / len(gt)
        count += 1
    return total / count if count > 0 else 0.0

def rerank_single(args):
    ids, q_vec, corpus_qt = args
    sims = [(float(np.minimum(q_vec, corpus_qt[c]).sum() /
                   np.maximum(q_vec, corpus_qt[c]).sum()), c)
            for c in ids]
    sims.sort(reverse=True)
    return ([c for _, c in sims], [])

def rerank_parallel(query_qt, nbrs_raw, corpus_qt, n_workers=64):
    args = [(ids, query_qt[i], corpus_qt)
            for i, (ids, _) in enumerate(nbrs_raw)]
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        return list(ex.map(rerank_single, args))

def get_mem():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

# ─── Build baseline WJ index ──────────────────────────────────────────────────
print("\n=== Building Baseline HNSW (WJ) ===")
m0 = get_mem()
idx_bl = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in tqdm(range(len(corpus_qt_full)), desc="Adding", mininterval=2.0):
    idx_bl.addDataPoint(i, corpus_qt_full[i])
t0 = time.time()
idx_bl.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=True)
build_bl = time.time() - t0
mem_bl   = get_mem() - m0
idx_bl.setQueryTimeParams({'efSearch': 200})
t0      = time.time()
nbrs_bl = idx_bl.knnQueryBatch(query_qt_full, k=500, num_threads=32)
qps_bl  = len(query_qt_full) / (time.time() - t0)
print(f"Build={build_bl:.1f}s | QPS={qps_bl:.1f} | IdxMem={mem_bl:.1f}MB")
del idx_bl

# ─── Build cosine index ───────────────────────────────────────────────────────
print("\n=== Building Cosine HNSW ===")
m0 = get_mem()
idx_cos = nmslib.init(method='hnsw', space='cosinesimil')
for i in tqdm(range(len(corpus_cos_full)), desc="Adding", mininterval=2.0):
    idx_cos.addDataPoint(i, corpus_cos_full[i])
t0 = time.time()
idx_cos.createIndex({'M': 20, 'efConstruction': 200, 'post': 1}, print_progress=True)
build_cos = time.time() - t0
mem_cos   = get_mem() - m0
idx_cos.setQueryTimeParams({'efSearch': 200})
print(f"Build={build_cos:.1f}s | IdxMem={mem_cos:.1f}MB")

# ─── Evaluate all methods ─────────────────────────────────────────────────────
print("\nEvaluating all methods...")

# No rerank K=500
t0          = time.time()
nbrs_cos500 = idx_cos.knnQueryBatch(query_cos_full, k=500, num_threads=32)
qps_cos     = len(query_cos_full) / (time.time() - t0)

# K=100 + rerank
t0 = time.time(); nbrs_r = idx_cos.knnQueryBatch(query_cos_full, k=100, num_threads=32); th=time.time()-t0
t0 = time.time(); nbrs_rr100 = rerank_parallel(query_qt_full, nbrs_r, corpus_qt_full); tr=time.time()-t0
qps_rr100 = len(query_cos_full) / (th + tr)

# K=200 + rerank
t0 = time.time(); nbrs_r = idx_cos.knnQueryBatch(query_cos_full, k=200, num_threads=32); th=time.time()-t0
t0 = time.time(); nbrs_rr200 = rerank_parallel(query_qt_full, nbrs_r, corpus_qt_full); tr=time.time()-t0
qps_rr200 = len(query_cos_full) / (th + tr)

# ─── Print final table ────────────────────────────────────────────────────────
vec_bl  = corpus_qt_full.nbytes  / 1024**2
vec_cos = corpus_cos_full.nbytes / 1024**2

print(f"\n{'='*112}")
print(f"FINAL COMPREHENSIVE COMPARISON — Full Dataset (233,773 polygons, 32 threads)")
print(f"{'='*112}")
print(f"{'Method':<38} {'R@10':>7} {'R@50':>7} {'R@100':>7} {'R@500':>7} "
      f"{'QPS':>8} {'Build':>8} {'Vec(MB)':>9} {'Idx(MB)':>9}")
print("-"*112)

rows = [
    ("Baseline (Quadtree+WJ)",     nbrs_bl,     qps_bl,    build_bl,  vec_bl,  mem_bl,  [10,50,100,500]),
    ("MLP+Cosine (no rerank)",     nbrs_cos500, qps_cos,   build_cos, vec_cos, mem_cos, [10,50,100,500]),
    ("MLP+Cosine K=100+WJrerank",  nbrs_rr100,  qps_rr100, build_cos, vec_cos, mem_cos, [10,50,100]),
    ("MLP+Cosine K=200+WJrerank",  nbrs_rr200,  qps_rr200, build_cos, vec_cos, mem_cos, [10,50,100]),
]

for name, nbrs, qps, build, vec, idx_m, valid_k in rows:
    recs = {k: eval_recall(gt_lookup, nbrs, QUERY_START_ID, k) for k in valid_k}
    r10  = f"{recs[10]:.4f}"   if 10  in valid_k else "  —   "
    r50  = f"{recs[50]:.4f}"   if 50  in valid_k else "  —   "
    r100 = f"{recs[100]:.4f}"  if 100 in valid_k else "  —   "
    r500 = f"{recs[500]:.4f}"  if 500 in valid_k else "  —   "
    print(f"{name:<38} {r10:>7} {r50:>7} {r100:>7} {r500:>7} "
          f"{qps:>8.1f} {build:>7.1f}s {vec:>8.1f} {idx_m:>8.1f}")

print(f"\n{'='*60}")
print(f"Key improvements vs Baseline:")
print(f"  Vector size:    {vec_bl:.1f} MB → {vec_cos:.1f} MB  ({vec_bl/vec_cos:.1f}x smaller)")
print(f"  Index memory:   {mem_bl:.1f} MB → {mem_cos:.1f} MB  ({mem_bl/mem_cos:.1f}x smaller)")
print(f"  Build time:     {build_bl:.1f}s → {build_cos:.1f}s  ({build_bl/build_cos:.1f}x faster)")
print(f"  QPS (no rerank): {qps_cos/qps_bl:.1f}x speedup")
print(f"  QPS K=100+rerank: {qps_rr100/qps_bl:.2f}x vs baseline")
print(f"  QPS K=200+rerank: {qps_rr200/qps_bl:.2f}x vs baseline")

Fixed model loaded.

=== Embedding quality check ===
GT pair cosine sim:  0.9338 ± 0.0569
Random cosine sim:   0.1222 ± 0.1723
Separation gap:      0.8117  (10k model was ~0.08)
OK: gap is meaningful — proceeding to evaluation

Generating cosine embeddings...


Embedding: 100%|██████████| 457/457 [00:07<00:00, 58.40it/s]


Embeddings: (233773, 512)
Vector size: 365.3 MB (baseline: 12998.5 MB, 35.6x larger)

=== Building Baseline HNSW (WJ) ===


Adding: 100%|██████████| 187019/187019 [00:06<00:00, 27395.35it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Build=421.6s | QPS=686.8 | IdxMem=0.5MB

=== Building Cosine HNSW ===


Adding: 100%|██████████| 187019/187019 [00:00<00:00, 540756.63it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Build=41.2s | IdxMem=0.0MB

Evaluating all methods...

FINAL COMPREHENSIVE COMPARISON — Full Dataset (233,773 polygons, 32 threads)
Method                                    R@10    R@50   R@100   R@500      QPS    Build   Vec(MB)   Idx(MB)
----------------------------------------------------------------------------------------------------------------
Baseline (Quadtree+WJ)                  0.9924  0.9952  0.9962  0.9862    686.8   421.6s  12998.5      0.5
MLP+Cosine (no rerank)                  0.6581  0.7264  0.7398  0.7643   4753.5    41.2s    365.3      0.0
MLP+Cosine K=100+WJrerank               0.9874  0.9151  0.7398    —       231.3    41.2s    365.3      0.0
MLP+Cosine K=200+WJrerank               0.9916  0.9804  0.9258    —       121.6    41.2s    365.3      0.0

Key improvements vs Baseline:
  Vector size:    12998.5 MB → 365.3 MB  (35.6x smaller)


ZeroDivisionError: float division by zero

In [121]:
import numpy as np
import time

def rerank_wj_batched(query_qt, nbrs_raw, corpus_qt, batch_size=1000):
    """
    Fully batched WJ reranking.
    For each query, compute WJ against all K candidates simultaneously.
    Process queries in batches to avoid memory issues.
    """
    all_ids    = np.array([ids for ids, _ in nbrs_raw])   # (Q, K)
    Q, K       = all_ids.shape
    reranked   = []

    for start in range(0, Q, batch_size):
        end      = min(start + batch_size, Q)
        batch_q  = query_qt[start:end]                    # (B, D)
        batch_ids= all_ids[start:end]                     # (B, K)

        # Gather all candidate vectors: (B, K, D)
        cand_vecs = corpus_qt[batch_ids.ravel()].reshape(end-start, K, -1)

        # Expand query: (B, 1, D) → broadcast over K
        q_exp = batch_q[:, np.newaxis, :]                 # (B, 1, D)

        # Vectorized min/max over D dimension
        mins  = np.minimum(q_exp, cand_vecs).sum(axis=2)  # (B, K)
        maxs  = np.maximum(q_exp, cand_vecs).sum(axis=2)  # (B, K)
        wj    = mins / np.maximum(maxs, 1e-10)            # (B, K)

        # Sort each row descending
        order = np.argsort(-wj, axis=1)                   # (B, K)

        for b in range(end - start):
            sorted_ids = batch_ids[b][order[b]]
            reranked.append((list(sorted_ids), []))

    return reranked

# ─── Benchmark batched reranking ──────────────────────────────────────────────
print("=== Batched vectorized WJ reranking ===")
print(f"Queries: {len(query_qt_full)} | Corpus dim: {corpus_qt_full.shape[1]}")

for K_cand in [100, 200]:
    idx_cos.setQueryTimeParams({'efSearch': 200})
    t0       = time.time()
    nbrs_raw = idx_cos.knnQueryBatch(query_cos_full, k=K_cand, num_threads=32)
    t_hnsw   = time.time() - t0

    t0      = time.time()
    nbrs_rr = rerank_wj_batched(query_qt_full, nbrs_raw, corpus_qt_full,
                                  batch_size=500)
    t_rr    = time.time() - t0

    qps_rr  = len(query_cos_full) / (t_hnsw + t_rr)

    valid_k = [k for k in [10, 50, 100] if k <= K_cand]
    recs    = {k: eval_recall(gt_lookup, nbrs_rr, QUERY_START_ID, k)
               for k in valid_k}

    print(f"\nK={K_cand} | HNSW={t_hnsw:.2f}s | Rerank={t_rr:.2f}s | QPS={qps_rr:.1f}")
    for k in valid_k:
        print(f"  R@{k:<4} = {recs[k]:.4f}")

print(f"\nBaseline QPS: {qps_bl:.1f}")
print(f"Target: beat baseline QPS ({qps_bl:.0f}) with K=100 rerank")

=== Batched vectorized WJ reranking ===
Queries: 46754 | Corpus dim: 18220

K=100 | HNSW=6.49s | Rerank=145.04s | QPS=308.6
  R@10   = 0.9874
  R@50   = 0.9151
  R@100  = 0.7398

K=200 | HNSW=7.20s | Rerank=338.07s | QPS=135.4
  R@10   = 0.9916
  R@50   = 0.9804
  R@100  = 0.9258

Baseline QPS: 686.8
Target: beat baseline QPS (687) with K=100 rerank


In [ ]:
import numpy as np
import time

# ─── Precise timing breakdown ─────────────────────────────────────────────────
print("=== Precise timing breakdown ===\n")

# How long does one WJ computation take?
q_vec   = query_qt_full[0]
c_vecs  = corpus_qt_full[:1000]

t0 = time.time()
for _ in range(100):
    mins = np.minimum(q_vec, c_vecs).sum(axis=1)
    maxs = np.maximum(q_vec, c_vecs).sum(axis=1)
    wj   = mins / maxs
t_per_1000 = (time.time() - t0) / 100
print(f"WJ computation for 1 query × 1000 candidates: {t_per_1000*1000:.2f}ms")
print(f"WJ computation for 46754 queries × 100 cands: "
      f"{t_per_1000 * 46754 / 10:.1f}s estimated")

# ─── The real QPS question: how many WJ comparisons does baseline do? ─────────
# HNSW with M=20, efSearch=200 visits ~200-400 nodes per query
# Each node visit = 1 WJ computation on 18220-dim vector
# Our rerank = K WJ computations per query on same vectors
# So K=100 rerank ≈ same work as HNSW doing 100 node visits

print(f"\n=== Theoretical analysis ===")
print(f"Baseline HNSW efSearch=200: ~200-400 WJ comparisons per query")
print(f"Our K=100 rerank:            100 WJ comparisons per query")
print(f"Our K=200 rerank:            200 WJ comparisons per query")
print(f"\nSo K=100 rerank should be FASTER than baseline, not slower.")
print(f"The bottleneck is Python overhead, not WJ computation itself.")

# ─── Try pure numpy without Python loop ───────────────────────────────────────
print(f"\n=== Testing fully vectorized batch reranking ===")

# Pre-gather all candidate vectors into one big array
nbrs_raw_100 = idx_cos.knnQueryBatch(query_cos_full, k=100, num_threads=32)
all_ids = np.array([ids for ids, _ in nbrs_raw_100], dtype=np.int32)  # (Q, 100)
Q, K    = all_ids.shape

t0 = time.time()
# Gather all candidates at once: (Q*K, D) then reshape
flat_ids  = all_ids.ravel()                          # (Q*K,)
cand_flat = corpus_qt_full[flat_ids]                 # (Q*K, D)
cand_3d   = cand_flat.reshape(Q, K, -1)              # (Q, K, D)
q_3d      = query_qt_full[:, np.newaxis, :]          # (Q, 1, D)
mins      = np.minimum(q_3d, cand_3d).sum(axis=2)   # (Q, K)
maxs      = np.maximum(q_3d, cand_3d).sum(axis=2)   # (Q, K)
wj        = mins / np.maximum(maxs, 1e-10)           # (Q, K)
order     = np.argsort(-wj, axis=1)                  # (Q, K)
t_rerank  = time.time() - t0

print(f"Full vectorized rerank time (K=100): {t_rerank:.2f}s")
print(f"Memory used: {cand_3d.nbytes/1024**2:.0f} MB for candidate array")

# Reconstruct results
nbrs_rr_fast = []
for i in range(Q):
    sorted_ids = all_ids[i][order[i]]
    nbrs_rr_fast.append((list(sorted_ids), []))

# Get QPS
t_hnsw   = 6.5  # from previous run
qps_fast = Q / (t_hnsw + t_rerank)
print(f"Total QPS (HNSW + rerank): {qps_fast:.1f}")

# Verify recall
recs = {k: eval_recall(gt_lookup, nbrs_rr_fast, QUERY_START_ID, k)
        for k in [10, 50]}
print(f"R@10={recs[10]:.4f} R@50={recs[50]:.4f}")
print(f"Baseline QPS: {qps_bl:.1f}")

=== Precise timing breakdown ===

WJ computation for 1 query × 1000 candidates: 22.75ms
WJ computation for 46754 queries × 100 cands: 106.3s estimated

=== Theoretical analysis ===
Baseline HNSW efSearch=200: ~200-400 WJ comparisons per query
Our K=100 rerank:            100 WJ comparisons per query
Our K=200 rerank:            200 WJ comparisons per query

So K=100 rerank should be FASTER than baseline, not slower.
The bottleneck is Python overhead, not WJ computation itself.

=== Testing fully vectorized batch reranking ===


In [1]:
import torch
import os

print("=== Post-restart check ===")
files = [
    '/tmp/best_compressor_full_fixed.pt',
    '/tmp/best_compressor_v1_clean.pt',
    '/tmp/best_minhash.pt',
    '/tmp/best_chisquare2.pt',
]
for f in files:
    exists = os.path.exists(f)
    size   = os.path.getsize(f)/1024**2 if exists else 0
    print(f"  {'OK' if exists else 'MISSING':>7} | {size:6.1f} MB | {f}")

print(f"\nCUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Ready to proceed.")

=== Post-restart check ===
       OK |  302.8 MB | /tmp/best_compressor_full_fixed.pt
       OK |  307.1 MB | /tmp/best_compressor_v1_clean.pt
       OK |   78.3 MB | /tmp/best_minhash.pt
       OK |  164.6 MB | /tmp/best_chisquare2.pt

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Ready to proceed.


In [8]:
import os
import numpy as np
from tqdm import tqdm

gt_dir   = '/raid/ruban/groundtruth/pk-query-187019'
gt_lookup = {}

for fname in tqdm(sorted(os.listdir(gt_dir)), desc="Loading GT"):
    fpath = os.path.join(gt_dir, fname)
    with open(fpath, 'r') as f:
        content = f.read().strip()

    # Format: "qid, n1, n2, n3, ..." one per line or all on one line
    # Try line by line first
    lines = content.split('\n')
    for line in lines:
        line = line.strip()
        if not line:
            continue
        ids = [int(x.strip()) for x in line.split(',') if x.strip().isdigit()]
        if ids:
            qid = ids[0]
            neighbors = ids[1:]
            if neighbors:
                gt_lookup[qid] = neighbors

print(f"GT queries loaded: {len(gt_lookup)}")
if gt_lookup:
    sample_key = next(iter(gt_lookup))
    print(f"Sample key: {sample_key}")
    print(f"Sample neighbors[:5]: {gt_lookup[sample_key][:5]}")
    print(f"Avg neighbors: {np.mean([len(v) for v in gt_lookup.values()]):.1f}")

Loading GT: 100%|██████████| 120/120 [00:58<00:00,  2.04it/s]

GT queries loaded: 44666
Sample key: 187019
Sample neighbors[:5]: [184495, 105197, 51737, 107491, 130402]
Avg neighbors: 4843.9


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nmslib
import time
import os
import glob
import pickle
from tqdm import tqdm

device = torch.device('cuda:0')

# ─── Model definition ─────────────────────────────────────────────────────────
class QuadtreeCompressorV1(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False),   nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x, for_index=False):
        out = self.net(x)
        if for_index:
            out = F.relu(out)
            out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return out

# ─── Check 10k encoding format ────────────────────────────────────────────────
enc_dir_10k = '/raid/ruban/encodings/pk-real10k0.002'
all_files   = sorted(os.listdir(enc_dir_10k))
print(f"Files in 10k enc dir: {len(all_files)}")
print(f"Sample filenames: {all_files[:3]}")

# Read first file to understand format
sample_file = os.path.join(enc_dir_10k, all_files[0])
print(f"\nFirst file: {all_files[0]}")
print(f"Size: {os.path.getsize(sample_file)} bytes")
with open(sample_file, 'r') as f:
    first_line = f.readline()
print(f"First line (first 200 chars): {first_line[:200]}")
print(f"Looks like: {first_line[:50].split()[:5]}")

Files in 10k enc dir: 80
Sample filenames: ['real_0.txt', 'real_1000.txt', 'real_1125.txt']

First file: real_0.txt
Size: 22311482 bytes
First line (first 200 chars): 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 
Looks like: ['0.0', '0.0', '0.0', '0.0', '0.0']


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nmslib
import time
import os
import glob
import pickle
from tqdm import tqdm

device = torch.device('cuda:0')
print(f"GPU: {torch.cuda.get_device_name(0)}")

# ─── Model definition ─────────────────────────────────────────────────────────
class QuadtreeCompressorV1(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False),   nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x, for_index=False):
        out = self.net(x)
        if for_index:
            out = F.relu(out)
            out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return out

# ─── Recall utility ───────────────────────────────────────────────────────────
def compute_recall_at_k(gt_lookup, nbrs, query_start_id, K):
    total = 0.0; count = 0
    for i, (ids, _) in enumerate(nbrs):
        qid = query_start_id + i
        gt  = set(gt_lookup.get(qid, [])[:K])
        if not gt: continue
        total += len(gt & set(ids[:K])) / len(gt)
        count += 1
    return total / count if count > 0 else 0.0, total, count

# ─── Load + cache 10k quadtree ────────────────────────────────────────────────
cache_10k = '/tmp/qt_10k.npy'
if os.path.exists(cache_10k):
    qt_10k = np.load(cache_10k)
    print(f"10k quadtree loaded from cache: {qt_10k.shape}")
else:
    print("Loading 10k quadtree from txt files...")
    enc_dir   = '/raid/ruban/encodings/pk-real10k0.002'
    all_files = sorted(os.listdir(enc_dir))
    arrays    = []
    for fname in tqdm(all_files, desc="Loading 10k"):
        fpath = os.path.join(enc_dir, fname)
        arr   = np.loadtxt(fpath, dtype=np.float32)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        arrays.append(arr)
    qt_10k = np.vstack(arrays)
    np.save(cache_10k, qt_10k)
    print(f"Saved to cache: {qt_10k.shape}")

IN_DIM_10K      = qt_10k.shape[1]
QUERY_START_10K = 8000
print(f"IN_DIM_10K={IN_DIM_10K} | QUERY_START_10K={QUERY_START_10K}")

# ─── Load + cache 10k GT ─────────────────────────────────────────────────────
cache_gt_10k = '/tmp/gt_lookup_10k.pkl'
if os.path.exists(cache_gt_10k):
    with open(cache_gt_10k, 'rb') as f:
        gt_lookup_10k = pickle.load(f)
    print(f"10k GT loaded from cache: {len(gt_lookup_10k)} queries")
else:
    print("Loading 10k GT...")
    gt_lookup_10k = {}
    gt_dir_10k    = '/raid/ruban/groundtruth/pk-query-10k'
    for fname in tqdm(sorted(os.listdir(gt_dir_10k)), desc="GT 10k"):
        fpath = os.path.join(gt_dir_10k, fname)
        with open(fpath, 'r') as f:
            content = f.read().strip()
        for line in content.split('\n'):
            line = line.strip()
            if not line: continue
            ids = [int(x.strip()) for x in line.split(',')
                   if x.strip().lstrip('-').isdigit()]
            if len(ids) >= 2:
                gt_lookup_10k[ids[0]] = ids[1:]
    with open(cache_gt_10k, 'wb') as f:
        pickle.dump(gt_lookup_10k, f)
    print(f"10k GT cached: {len(gt_lookup_10k)} queries")

# ─── Load full quadtree from cache ────────────────────────────────────────────
print("Loading full quadtree from cache...")
qtree_vectors   = np.load('/tmp/qtree_vectors_full.npy')
IN_DIM          = qtree_vectors.shape[1]
QUERY_START_ID  = 187019
corpus_qt_full  = qtree_vectors[:QUERY_START_ID]
query_qt_full   = qtree_vectors[QUERY_START_ID:]
print(f"Full quadtree: {qtree_vectors.shape}")

# ─── Load + cache full GT ─────────────────────────────────────────────────────
cache_gt_full = '/tmp/gt_lookup_full.pkl'
if os.path.exists(cache_gt_full):
    with open(cache_gt_full, 'rb') as f:
        gt_lookup = pickle.load(f)
    print(f"Full GT loaded from cache: {len(gt_lookup)} queries")
else:
    print("Loading full GT from similarityMap files...")
    gt_lookup = {}
    gt_dir    = '/raid/ruban/groundtruth/pk-query-187019'
    for fname in tqdm(sorted(os.listdir(gt_dir)), desc="Full GT"):
        fpath = os.path.join(gt_dir, fname)
        with open(fpath, 'r') as f:
            content = f.read().strip()
        for line in content.split('\n'):
            line = line.strip()
            if not line: continue
            ids = [int(x.strip()) for x in line.split(',')
                   if x.strip().lstrip('-').isdigit()]
            if len(ids) >= 2:
                gt_lookup[ids[0]] = ids[1:]
    with open(cache_gt_full, 'wb') as f:
        pickle.dump(gt_lookup, f)
    print(f"Full GT cached: {len(gt_lookup)} queries")

print(f"\n=== All data ready ===")
print(f"10k:  qt={qt_10k.shape} | GT={len(gt_lookup_10k)} queries")
print(f"Full: qt={qtree_vectors.shape} | GT={len(gt_lookup)} queries")

GPU: NVIDIA A100-SXM4-80GB
Loading 10k quadtree from txt files...


Loading 10k: 100%|██████████| 80/80 [00:42<00:00,  1.89it/s]


Saved to cache: (10000, 18499)
IN_DIM_10K=18499 | QUERY_START_10K=8000
Loading 10k GT...


GT 10k: 100%|██████████| 60/60 [00:00<00:00, 468.09it/s]

10k GT cached: 1818 queries
Loading full quadtree from cache...


Full quadtree: (233773, 18220)
Loading full GT from similarityMap files...


Full GT: 100%|██████████| 120/120 [01:08<00:00,  1.76it/s]


Full GT cached: 44666 queries

=== All data ready ===
10k:  qt=(10000, 18499) | GT=1818 queries
Full: qt=(233773, 18220) | GT=44666 queries


In [13]:
import numpy as np
import nmslib
import time
import torch
import torch.nn.functional as F

# Use best MLP WJ embeddings (already L1-normalized, non-negative)
# Reload best MLP
mlp = QuadtreeCompressorV1(in_dim=IN_DIM_10K, out_dim=512).to(device)
mlp.load_state_dict(torch.load('/tmp/best_compressor_v1_clean.pt', weights_only=True))
mlp.eval()

# Generate WJ embeddings (ReLU + L1 norm)
all_embs = []
qt_tensor = torch.tensor(qt_10k, dtype=torch.float32)
with torch.no_grad():
    for start in range(0, len(qt_tensor), 512):
        batch = qt_tensor[start:start+512].to(device)
        emb   = mlp(batch, for_index=True)  # ReLU + L1 norm
        all_embs.append(emb.cpu().numpy())

embs_wj   = np.vstack(all_embs)
print(f"WJ embeddings: {embs_wj.shape}")
print(f"Non-negative: {(embs_wj>=0).all()}")
print(f"Row sums: {embs_wj.sum(axis=1)[:3]}")  # should be ~1.0

# Apply sqrt transform
embs_sqrt = np.sqrt(embs_wj)
# Now L2-normalize for cosine index
norms     = np.linalg.norm(embs_sqrt, axis=1, keepdims=True)
embs_sqrt_l2 = embs_sqrt / np.maximum(norms, 1e-10)

corpus_wj   = embs_wj[:QUERY_START_10K]
query_wj    = embs_wj[QUERY_START_10K:]
corpus_sqrt = embs_sqrt_l2[:QUERY_START_10K]
query_sqrt  = embs_sqrt_l2[QUERY_START_10K:]

# ─── Compare 3 approaches ─────────────────────────────────────────────────────
print(f"\n{'Method':<35} {'R@10':>8} {'R@50':>8} {'QPS':>10}")
print("-" * 65)

# 1. WJ index (original)
idx_wj = nmslib.init(method='hnsw', space='WeightedJaccard')
for i in range(len(corpus_wj)): idx_wj.addDataPoint(i, corpus_wj[i])
idx_wj.createIndex({'M':20,'efConstruction':200,'post':1}, print_progress=False)
idx_wj.setQueryTimeParams({'efSearch':200})
t0 = time.time(); nbrs = idx_wj.knnQueryBatch(query_wj, k=50, num_threads=32)
qps = len(query_wj)/(time.time()-t0)
r10,_,_ = compute_recall_at_k(gt_lookup_10k, nbrs, QUERY_START_10K, K=10)
r50,_,_ = compute_recall_at_k(gt_lookup_10k, nbrs, QUERY_START_10K, K=50)
print(f"{'MLP (WJ index)':<35} {r10:>8.4f} {r50:>8.4f} {qps:>10.1f}")

# 2. Cosine on raw L2-normalized embeddings
embs_l2  = F.normalize(torch.tensor(embs_wj), dim=1).numpy()
corpus_l2 = embs_l2[:QUERY_START_10K]
query_l2  = embs_l2[QUERY_START_10K:]
idx_cos = nmslib.init(method='hnsw', space='cosinesimil')
for i in range(len(corpus_l2)): idx_cos.addDataPoint(i, corpus_l2[i])
idx_cos.createIndex({'M':20,'efConstruction':200,'post':1}, print_progress=False)
idx_cos.setQueryTimeParams({'efSearch':200})
t0 = time.time(); nbrs = idx_cos.knnQueryBatch(query_l2, k=50, num_threads=32)
qps = len(query_l2)/(time.time()-t0)
r10,_,_ = compute_recall_at_k(gt_lookup_10k, nbrs, QUERY_START_10K, K=10)
r50,_,_ = compute_recall_at_k(gt_lookup_10k, nbrs, QUERY_START_10K, K=50)
print(f"{'MLP (cosine, L2 norm)':<35} {r10:>8.4f} {r50:>8.4f} {qps:>10.1f}")

# 3. Cosine on sqrt-transformed embeddings (Bhattacharyya)
idx_sqrt = nmslib.init(method='hnsw', space='cosinesimil')
for i in range(len(corpus_sqrt)): idx_sqrt.addDataPoint(i, corpus_sqrt[i])
idx_sqrt.createIndex({'M':20,'efConstruction':200,'post':1}, print_progress=False)
idx_sqrt.setQueryTimeParams({'efSearch':200})
t0 = time.time(); nbrs = idx_sqrt.knnQueryBatch(query_sqrt, k=50, num_threads=32)
qps = len(query_sqrt)/(time.time()-t0)
r10,_,_ = compute_recall_at_k(gt_lookup_10k, nbrs, QUERY_START_10K, K=10)
r50,_,_ = compute_recall_at_k(gt_lookup_10k, nbrs, QUERY_START_10K, K=50)
print(f"{'MLP (cosine, sqrt/Bhattacharyya)':<35} {r10:>8.4f} {r50:>8.4f} {qps:>10.1f}")

print(f"\nBaseline: R@10=0.9965 R@50=0.9986 QPS=293")

WJ embeddings: (10000, 512)
Non-negative: True
Row sums: [0.99999994 1.         1.        ]

Method                                  R@10     R@50        QPS
-----------------------------------------------------------------
MLP (WJ index)                        0.0082   0.0139    36384.4
MLP (cosine, L2 norm)                 0.0083   0.0138    12390.4
MLP (cosine, sqrt/Bhattacharyya)      0.0078   0.0137    56620.1

Baseline: R@10=0.9965 R@50=0.9986 QPS=293
